# Notebook 16 — Yield Prediction Baselines

## 1. Objective

Develop and evaluate baseline regression models to predict historical agricultural yield for a district–crop–season–year observation. Establish a simple benchmark first, then test whether machine-learning models improve prediction accuracy on unseen later years.

This notebook focuses on **yield prediction only**. Crop recommendation and agricultural risk intelligence are separate modelling tasks.

## 2. Prediction contract

- **Prediction unit:** District × Crop × Season × Year
- **Target/output label (`y`):** `Yield`
- **Target type:** Continuous numerical variable
- **ML task:** Regression
- **Primary cohort:** Positive, finite Yield with positive Area and Production (107,267 records before the temporal split).
- **Sensitivity cohort:** All positive, finite, non-aggregate Yield records (166,813 records before the temporal split).
- **Training period:** 2013–14 through 2022–23
- **Holdout period:** 2023–24 through 2024–25
- **Evaluation principle:** Use a chronological holdout; do not rely on a random-only split.

The primary cohort is a modelling choice, not a guarantee that its measurements are correct. The sensitivity cohort will help determine how strongly cohort selection affects the results.

## 3. Feature configurations

### Configuration A — Core features

| Feature | Type | Purpose |
|---|---|---|
| `Crop` | Categorical | Represents the crop being grown |
| `Season` | Categorical | Represents the agricultural season |
| `Crop_Category` | Categorical | Represents the crop grouping |
| `FE_YearStart` | Numeric | Represents the starting year of the agricultural year |

These are the initial features available for the baseline experiments.

### Configuration B — Core + exploratory weather

Includes all four core features plus:

- `tmin_mean_c`
- `tmin_min_c`
- `tmax_mean_c`
- `tmax_max_c`
- `precipitation_total_mm`
- `precipitation_mean_mm_day`
- `precipitation_max_mm_day`
- `tmin_valid_days`
- `tmax_valid_days`
- `precipitation_valid_days`

**Weather limitation:** Weather is matched using a starting-calendar-year proxy. Full crop-season alignment is unverified, and weather coverage is geographically incomplete. Weather-enhanced results must therefore be treated as exploratory.

## 4. Models to evaluate

| Order | Model | Feature configuration | Role |
|---:|---|---|---|
| 1 | Global median baseline | No predictors; predicts the training-target median | Minimum benchmark |
| 2 | Crop–Season median baseline | Crop + Season | Tests whether group-level historical differences provide useful signal |
| 3 | Linear Regression | Core | Simple linear reference model |
| 4 | Ridge Regression | Core | Linear model with regularization |
| 5 | Decision Tree Regressor | Core | Captures nonlinear splits and interactions |
| 6 | Random Forest Regressor | Core | Ensemble of decision trees; nonlinear benchmark |
| 7 | HistGradientBoosting Regressor | Core | Boosted-tree benchmark for tabular data |
| 8 | Random Forest Regressor | Core + exploratory weather | Tests whether weather adds predictive value |
| 9 | HistGradientBoosting Regressor | Core + exploratory weather | Tests a second nonlinear model with weather features |

The initial benchmark set is intentionally compact. Additional algorithms will be considered only if they answer a specific modelling question or offer a meaningful comparison.

All learned preprocessing, including categorical encoding and imputation if needed, must be fitted using training data only.

## 5. Evaluation metrics

Report the following on the holdout set:

- **MAE:** Mean Absolute Error; average absolute prediction error in the target's reported units.
- **RMSE:** Root Mean Squared Error; penalizes larger errors more heavily.
- **R²:** Measures performance relative to predicting the holdout mean; it can be negative when predictions perform poorly.

Also inspect residuals, extreme errors, and performance by year and crop where sample sizes permit.

Do not choose a model based on R² alone. MAE, RMSE, target quality, stability, and practical limitations must also inform the decision.

## 6. Fair comparison and leakage controls

1. Use the same chronological train/holdout boundary for every model.
2. Fit all learned transformations on training data only.
3. Do not use `Yield`, `Area`, or `Production` as input features in this experiment.
4. Do not use future observations to construct training features.
5. Compare core and weather-enhanced models on the **same weather-complete holdout rows**; use a matching weather-complete training subset for the weather model.
6. Fit group-median baselines using training observations only. If a Crop–Season combination is unseen in training, fall back to the global training median.
7. Keep primary-cohort and sensitivity-cohort results separate.
8. Do not silently remove extreme targets or modify source measurements.
9. Do not describe predictive associations as causal effects.

## 7. Interpretation and intended outcome

The goal is to establish a credible benchmark and quantify whether added model complexity and exploratory weather information improve performance on later years.

This notebook does not yet establish reliable predictions for future crop seasons, unseen districts, or intervention planning. Such use would require additional validation, verified measurement units, improved seasonal weather alignment, and appropriate deployment-time inputs.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "weather_regional"

required_files = {
    "Integrated dataset": (
        DATA_DIR / "apy_weather_integrated_exploratory.csv"
    ),
    "Historical APY long dataset": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
    ),
}

print("Project root exists:", PROJECT_ROOT.exists())
print("Weather-regional directory exists:", DATA_DIR.exists())

print("\nRequired files:")
for label, path in required_files.items():
    print(f"\n{label}")
    print("  Path:", path)
    print("  Exists:", path.exists())

if DATA_DIR.exists():
    print("\nRelevant saved files in weather_regional:")
    for path in sorted(DATA_DIR.glob("*")):
        if path.is_file() and (
            "integrated" in path.name.lower()
            or "consistency" in path.name.lower()
            or "weather" in path.name.lower()
        ):
            print(f"  {path.name}")

Project root exists: True
Weather-regional directory exists: True

Required files:

Integrated dataset
  Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_weather_integrated_exploratory.csv
  Exists: True

Historical APY long dataset
  Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv
  Exists: True

Relevant saved files in weather_regional:
  apy_districts_without_weather_coverage.csv
  apy_weather_integrated_exploratory.csv
  apy_weather_integration_summary.csv
  apy_weather_proxy_district_year_coverage.csv
  crop_weather_integration_validation_summary.csv
  district_weather_annual_2013_2024.csv
  district_weather_annual_cleaned_flagged_2013_2024.csv
  district_weather_annual_preprocessed_2013_2024.csv


In [3]:
# Cell 2 — Reconstruct modelling cohorts from the saved integrated dataset.
# Source CSV remains unchanged.

import numpy as np
import pandas as pd
from pathlib import Path

integrated_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_weather_integrated_exploratory.csv"
)

integrated_df = pd.read_csv(integrated_path, low_memory=False)

print("Loaded integrated dataset:", integrated_df.shape)

# Validate the required columns before proceeding.
required_columns = [
    "State", "District", "Crop", "Season", "Year",
    "Crop_Category", "Area", "Production", "Yield",
    "tmin_mean_c", "tmin_min_c", "tmax_mean_c", "tmax_max_c",
    "precipitation_total_mm", "precipitation_mean_mm_day",
    "precipitation_max_mm_day", "tmin_valid_days",
    "tmax_valid_days", "precipitation_valid_days",
]

missing_columns = [
    col for col in required_columns
    if col not in integrated_df.columns
]

if missing_columns:
    raise ValueError(
        f"Integrated dataset is missing required columns: {missing_columns}"
    )

# Preserve the original loaded table and work on a separate copy.
model_df = integrated_df.copy()

# Normalize target and measurement columns to numeric.
for col in [
    "Yield", "Area", "Production",
    "tmin_mean_c", "tmin_min_c", "tmax_mean_c", "tmax_max_c",
    "precipitation_total_mm", "precipitation_mean_mm_day",
    "precipitation_max_mm_day", "tmin_valid_days",
    "tmax_valid_days", "precipitation_valid_days",
]:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# Convert agricultural year, e.g. "2013-14", to starting year 2013.
year_text = model_df["Year"].astype("string").str.strip()
year_start = year_text.str.extract(r"^(\d{4})", expand=False)
model_df["FE_YearStart"] = pd.to_numeric(year_start, errors="coerce")

if model_df["FE_YearStart"].isna().any():
    bad_years = model_df.loc[
        model_df["FE_YearStart"].isna(), "Year"
    ].drop_duplicates().head(10).tolist()
    raise ValueError(
        f"Could not parse some Year values. Examples: {bad_years}"
    )

# Exclude aggregate Total rows and retain positive finite targets.
is_aggregate = (
    model_df["Crop"].astype("string").str.strip().str.casefold()
    .eq("total")
    | model_df["Season"].astype("string").str.strip().str.casefold()
    .eq("total")
)

valid_positive_yield = (
    np.isfinite(model_df["Yield"])
    & model_df["Yield"].gt(0)
    & ~is_aggregate
)

sensitivity_df = model_df.loc[valid_positive_yield].copy()

# Primary cohort: positive yield with positive Area and Production.
positive_measurements = (
    np.isfinite(sensitivity_df["Area"])
    & np.isfinite(sensitivity_df["Production"])
    & sensitivity_df["Area"].gt(0)
    & sensitivity_df["Production"].gt(0)
)

primary_df = sensitivity_df.loc[positive_measurements].copy()

# Feature definitions.
core_categorical_features = ["Crop", "Season", "Crop_Category"]
core_numeric_features = ["FE_YearStart"]

weather_features = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days",
]

# Recreate the same chronological split.
primary_train = primary_df.loc[
    primary_df["FE_YearStart"] <= 2022
].copy()
primary_holdout = primary_df.loc[
    primary_df["FE_YearStart"] >= 2023
].copy()

sensitivity_train = sensitivity_df.loc[
    sensitivity_df["FE_YearStart"] <= 2022
].copy()
sensitivity_holdout = sensitivity_df.loc[
    sensitivity_df["FE_YearStart"] >= 2023
].copy()

# Validate split boundaries and cohort relationships.
assert len(primary_df) <= len(sensitivity_df)
assert primary_train["FE_YearStart"].max() < primary_holdout["FE_YearStart"].min()
assert sensitivity_train["FE_YearStart"].max() < sensitivity_holdout["FE_YearStart"].min()
assert primary_train["Yield"].gt(0).all()
assert primary_holdout["Yield"].gt(0).all()

print("\nReconstructed cohorts:")
print(f"Primary:      {len(primary_df):,}")
print(f"  Train:      {len(primary_train):,}")
print(f"  Holdout:    {len(primary_holdout):,}")
print(f"Sensitivity:  {len(sensitivity_df):,}")
print(f"  Train:      {len(sensitivity_train):,}")
print(f"  Holdout:    {len(sensitivity_holdout):,}")

print("\nPrimary cohort years:")
print(
    f"Train: {primary_train['Year'].min()} to "
    f"{primary_train['Year'].max()}"
)
print(
    f"Holdout: {primary_holdout['Year'].min()} to "
    f"{primary_holdout['Year'].max()}"
)

print("\nFeature definitions recreated.")
print("Core:", core_categorical_features + core_numeric_features)
print("Weather:", weather_features)
print("\nNo source CSV was modified.")

Loaded integrated dataset: (376272, 39)

Reconstructed cohorts:
Primary:      107,267
  Train:      88,551
  Holdout:    18,716
Sensitivity:  166,813
  Train:      136,622
  Holdout:    30,191

Primary cohort years:
Train: 2013-14 to 2022-23
Holdout: 2023-24 to 2024-25

Feature definitions recreated.
Core: ['Crop', 'Season', 'Crop_Category', 'FE_YearStart']
Weather: ['tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days']

No source CSV was modified.


In [4]:
# Cell 3 — Define common regression evaluation utilities.
# No model is trained in this cell.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression(y_true, y_pred):
    """Return consistent regression metrics for a set of predictions."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if len(y_true) == 0:
        raise ValueError("Evaluation data is empty.")

    if len(y_true) != len(y_pred):
        raise ValueError("True and predicted target lengths do not match.")

    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Evaluation contains non-finite target or prediction values.")

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def append_result(results, model_name, cohort, feature_set, y_true, y_pred):
    """Append one model evaluation to a results list."""
    metrics = evaluate_regression(y_true, y_pred)

    results.append({
        "Model": model_name,
        "Cohort": cohort,
        "Feature_Set": feature_set,
        "N_Evaluated": len(y_true),
        **metrics,
    })


# Store all future evaluation results here.
baseline_results = []

print("Evaluation utilities defined.")
print("Metrics: MAE, RMSE, R²")
print("Lower MAE/RMSE is better; higher R² is better.")
print("No model trained.")

Evaluation utilities defined.
Metrics: MAE, RMSE, R²
Lower MAE/RMSE is better; higher R² is better.
No model trained.


In [5]:
# Cell 4 — Global median baseline for primary and sensitivity cohorts.
# The median is calculated from training data only.

# Primary cohort
primary_global_median = primary_train["Yield"].median()
primary_global_predictions = np.full(
    len(primary_holdout), primary_global_median
)

append_result(
    baseline_results,
    "Global Median",
    "primary",
    "none",
    primary_holdout["Yield"],
    primary_global_predictions,
)

# Sensitivity cohort
sensitivity_global_median = sensitivity_train["Yield"].median()
sensitivity_global_predictions = np.full(
    len(sensitivity_holdout), sensitivity_global_median
)

append_result(
    baseline_results,
    "Global Median",
    "sensitivity",
    "none",
    sensitivity_holdout["Yield"],
    sensitivity_global_predictions,
)

# Display benchmark results
baseline_df = pd.DataFrame(baseline_results)

print(f"Primary training median yield: {primary_global_median:.4f}")
print(f"Sensitivity training median yield: {sensitivity_global_median:.4f}")
display(baseline_df.round(4))

Primary training median yield: 1870.0000
Sensitivity training median yield: 1531.0000


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106
1,Global Median,sensitivity,none,30191,1150.0961,2920.0047,-0.0351


In [6]:
# Cell 5 — Crop–Season median baseline.
# Group medians are fitted on training data only.

def add_crop_season_median_baseline(
    train_df, holdout_df, cohort_name, results
):
    global_median = train_df["Yield"].median()

    group_medians = (
        train_df.groupby(["Crop", "Season"], dropna=False)["Yield"]
        .median()
        .rename("Predicted_Yield")
        .reset_index()
    )

    # Preserve holdout row order while joining training-derived medians.
    holdout_eval = holdout_df[
        ["Crop", "Season", "Yield"]
    ].copy()
    holdout_eval["_row_order"] = np.arange(len(holdout_eval))

    holdout_eval = holdout_eval.merge(
        group_medians,
        on=["Crop", "Season"],
        how="left",
        sort=False,
        validate="many_to_one",
    )

    unseen_groups = holdout_eval["Predicted_Yield"].isna().sum()

    holdout_eval["Predicted_Yield"] = (
        holdout_eval["Predicted_Yield"].fillna(global_median)
    )

    holdout_eval = holdout_eval.sort_values("_row_order")

    append_result(
        results,
        "Crop-Season Median",
        cohort_name,
        "Crop + Season",
        holdout_eval["Yield"].to_numpy(),
        holdout_eval["Predicted_Yield"].to_numpy(),
    )

    print(f"{cohort_name.capitalize()} cohort:")
    print(f"  Training Crop-Season groups: {len(group_medians)}")
    print(f"  Holdout rows using global fallback: {unseen_groups:,}")


add_crop_season_median_baseline(
    primary_train, primary_holdout, "primary", baseline_results
)

add_crop_season_median_baseline(
    sensitivity_train, sensitivity_holdout, "sensitivity", baseline_results
)

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df.round(4))

Primary cohort:
  Training Crop-Season groups: 47
  Holdout rows using global fallback: 0
Sensitivity cohort:
  Training Crop-Season groups: 50
  Holdout rows using global fallback: 0


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106
1,Global Median,sensitivity,none,30191,1150.0961,2920.0047,-0.0351
2,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
3,Crop-Season Median,sensitivity,Crop + Season,30191,775.8573,2741.1952,0.0878


In [7]:
# Cell 6 — Define preprocessing for core features.
# No model is trained in this cell.

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

print("scikit-learn version:", sklearn.__version__)

core_categorical_features = ["Crop", "Season", "Crop_Category"]
core_numeric_features = ["FE_YearStart"]

# Compatibility with older and newer scikit-learn versions.
try:
    core_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    core_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

core_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", core_encoder, core_categorical_features),
        ("numeric", "passthrough", core_numeric_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Validate the feature columns before training.
required_core_columns = core_categorical_features + core_numeric_features

missing_train = [
    col for col in required_core_columns
    if col not in primary_train.columns
]
missing_holdout = [
    col for col in required_core_columns
    if col not in primary_holdout.columns
]

if missing_train or missing_holdout:
    raise ValueError(
        f"Missing training columns: {missing_train}; "
        f"missing holdout columns: {missing_holdout}"
    )

if primary_train[required_core_columns].isna().any().any():
    raise ValueError("Missing values found in primary training core features.")

if primary_holdout[required_core_columns].isna().any().any():
    raise ValueError("Missing values found in primary holdout core features.")

print("Core categorical features:", core_categorical_features)
print("Core numeric features:", core_numeric_features)
print("Training rows:", len(primary_train))
print("Holdout rows:", len(primary_holdout))
print("Preprocessor defined; no model trained.")

scikit-learn version: 1.8.0
Core categorical features: ['Crop', 'Season', 'Crop_Category']
Core numeric features: ['FE_YearStart']
Training rows: 88551
Holdout rows: 18716
Preprocessor defined; no model trained.


In [8]:
# Cell 7 — Linear Regression using core features.
# Preprocessing is fitted on training data only.

from sklearn.linear_model import LinearRegression

X_train_core = primary_train[required_core_columns]
y_train = primary_train["Yield"].astype(float)

X_holdout_core = primary_holdout[required_core_columns]
y_holdout = primary_holdout["Yield"].astype(float)

linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_regression_pipeline.fit(X_train_core, y_train)

linear_predictions = linear_regression_pipeline.predict(X_holdout_core)

append_result(
    baseline_results,
    "Linear Regression",
    "primary",
    "core",
    y_holdout,
    linear_predictions,
)

# Compare all evaluated models on the primary cohort.
primary_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'primary'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("Linear Regression evaluation:")
print(f"  Holdout rows: {len(y_holdout):,}")
print(f"  Predictions finite: {np.isfinite(linear_predictions).all()}")
print(f"  Minimum prediction: {linear_predictions.min():.4f}")
print(f"  Maximum prediction: {linear_predictions.max():.4f}")

display(primary_comparison.round(4))

Linear Regression evaluation:
  Holdout rows: 18,716
  Predictions finite: True
  Minimum prediction: 646.7837
  Maximum prediction: 3798.1098


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Linear Regression,primary,core,18716,813.4888,1137.7899,0.3719
1,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
2,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106


In [9]:
# Cell 8 — Ridge Regression using core features.
# Preprocessing and scaling are fitted on training data only.

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            core_categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            core_numeric_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", ridge_preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)

ridge_pipeline.fit(X_train_core, y_train)
ridge_predictions = ridge_pipeline.predict(X_holdout_core)

append_result(
    baseline_results,
    "Ridge Regression",
    "primary",
    "core",
    y_holdout,
    ridge_predictions,
)

primary_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'primary'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("Ridge Regression evaluation:")
print(f"  Predictions finite: {np.isfinite(ridge_predictions).all()}")
print(f"  Minimum prediction: {ridge_predictions.min():.4f}")
print(f"  Maximum prediction: {ridge_predictions.max():.4f}")

display(primary_comparison.round(4))

Ridge Regression evaluation:
  Predictions finite: True
  Minimum prediction: 647.1510
  Maximum prediction: 3797.8731


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Linear Regression,primary,core,18716,813.4888,1137.7899,0.3719
1,Ridge Regression,primary,core,18716,813.5162,1137.8026,0.3719
2,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
3,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106


In [10]:
# Cell 9 — Decision Tree Regressor using core features.
# The preprocessing is fitted on training data only.

from sklearn.tree import DecisionTreeRegressor

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=12,
                min_samples_leaf=50,
                random_state=42,
            ),
        ),
    ]
)

decision_tree_pipeline.fit(X_train_core, y_train)
tree_predictions = decision_tree_pipeline.predict(X_holdout_core)

append_result(
    baseline_results,
    "Decision Tree",
    "primary",
    "core",
    y_holdout,
    tree_predictions,
)

primary_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'primary'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("Decision Tree evaluation:")
print(f"  Predictions finite: {np.isfinite(tree_predictions).all()}")
print(f"  Minimum prediction: {tree_predictions.min():.4f}")
print(f"  Maximum prediction: {tree_predictions.max():.4f}")
print(f"  Actual maximum holdout yield: {y_holdout.max():.4f}")

display(primary_comparison.round(4))

Decision Tree evaluation:
  Predictions finite: True
  Minimum prediction: 578.5214
  Maximum prediction: 4512.5207
  Actual maximum holdout yield: 12237.0000


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Decision Tree,primary,core,18716,780.4694,1122.9861,0.3881
1,Linear Regression,primary,core,18716,813.4888,1137.7899,0.3719
2,Ridge Regression,primary,core,18716,813.5162,1137.8026,0.3719
3,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
4,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106


In [11]:
# Cell 10 — Random Forest Regressor using core features.

from sklearn.ensemble import RandomForestRegressor

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=16,
                min_samples_leaf=20,
                max_features=1.0,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

random_forest_pipeline.fit(X_train_core, y_train)
rf_predictions = random_forest_pipeline.predict(X_holdout_core)

append_result(
    baseline_results,
    "Random Forest",
    "primary",
    "core",
    y_holdout,
    rf_predictions,
)

primary_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'primary'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("Random Forest evaluation:")
print(f"  Predictions finite: {np.isfinite(rf_predictions).all()}")
print(f"  Minimum prediction: {rf_predictions.min():.4f}")
print(f"  Maximum prediction: {rf_predictions.max():.4f}")
print(f"  Actual maximum holdout yield: {y_holdout.max():.4f}")

display(primary_comparison.round(4))

Random Forest evaluation:
  Predictions finite: True
  Minimum prediction: 579.7378
  Maximum prediction: 4508.1976
  Actual maximum holdout yield: 12237.0000


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,Random Forest,primary,core,18716,782.1447,1114.3597,0.3975
1,Decision Tree,primary,core,18716,780.4694,1122.9861,0.3881
2,Linear Regression,primary,core,18716,813.4888,1137.7899,0.3719
3,Ridge Regression,primary,core,18716,813.5162,1137.8026,0.3719
4,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
5,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106


In [12]:
# Cell 11 — HistGradientBoosting Regressor using core features.
# Dense one-hot encoding is required for this estimator.

from sklearn.ensemble import HistGradientBoostingRegressor

hist_gradient_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

hist_gradient_pipeline.fit(X_train_core, y_train)
hist_gradient_predictions = hist_gradient_pipeline.predict(X_holdout_core)

append_result(
    baseline_results,
    "HistGradientBoosting",
    "primary",
    "core",
    y_holdout,
    hist_gradient_predictions,
)

primary_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'primary'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("HistGradientBoosting evaluation:")
print(
    "  Predictions finite:",
    np.isfinite(hist_gradient_predictions).all()
)
print(f"  Minimum prediction: {hist_gradient_predictions.min():.4f}")
print(f"  Maximum prediction: {hist_gradient_predictions.max():.4f}")
print(f"  Actual maximum holdout yield: {y_holdout.max():.4f}")

display(primary_comparison.round(4))

HistGradientBoosting evaluation:
  Predictions finite: True
  Minimum prediction: 618.8712
  Maximum prediction: 4566.8246
  Actual maximum holdout yield: 12237.0000


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,HistGradientBoosting,primary,core,18716,780.2750,1111.9985,0.4001
1,Random Forest,primary,core,18716,782.1447,1114.3597,0.3975
2,Decision Tree,primary,core,18716,780.4694,1122.9861,0.3881
3,Linear Regression,primary,core,18716,813.4888,1137.7899,0.3719
4,Ridge Regression,primary,core,18716,813.5162,1137.8026,0.3719
5,Crop-Season Median,primary,Crop + Season,18716,795.3182,1183.8378,0.3200
6,Global Median,primary,none,18716,1116.6854,1512.9737,-0.1106


In [13]:
# Cell 12 — Evaluate core-feature models on the sensitivity cohort.
# All models are fitted on sensitivity training data only.

X_sensitivity_train = sensitivity_train[required_core_columns]
y_sensitivity_train = sensitivity_train["Yield"].astype(float)

X_sensitivity_holdout = sensitivity_holdout[required_core_columns]
y_sensitivity_holdout = sensitivity_holdout["Yield"].astype(float)

sensitivity_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=50,
        random_state=42,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=20,
        max_features=1.0,
        n_jobs=-1,
        random_state=42,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    ),
}

sensitivity_model_pipelines = {}

for model_name, estimator in sensitivity_models.items():
    # Use scaled year for Ridge; the other models use the core preprocessor.
    if model_name == "Ridge Regression":
        preprocessor = ridge_preprocessor
    else:
        preprocessor = core_preprocessor

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    pipeline.fit(X_sensitivity_train, y_sensitivity_train)
    predictions = pipeline.predict(X_sensitivity_holdout)

    append_result(
        baseline_results,
        model_name,
        "sensitivity",
        "core",
        y_sensitivity_holdout,
        predictions,
    )

    sensitivity_model_pipelines[model_name] = pipeline

sensitivity_comparison = (
    pd.DataFrame(baseline_results)
    .query("Cohort == 'sensitivity'")
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("Sensitivity cohort evaluation complete.")
print(f"Holdout rows per model: {len(y_sensitivity_holdout):,}")
display(sensitivity_comparison.round(4))

Sensitivity cohort evaluation complete.
Holdout rows per model: 30,191


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,HistGradientBoosting,sensitivity,core,30191,763.2701,2704.8005,0.1118
1,Random Forest,sensitivity,core,30191,764.7603,2705.0083,0.1117
2,Decision Tree,sensitivity,core,30191,764.8650,2705.6722,0.1113
3,Linear Regression,sensitivity,core,30191,797.6466,2711.3253,0.1075
4,Ridge Regression,sensitivity,core,30191,797.6634,2711.3283,0.1075
5,Crop-Season Median,sensitivity,Crop + Season,30191,775.8573,2741.1952,0.0878
6,Global Median,sensitivity,none,30191,1150.0961,2920.0047,-0.0351


In [14]:
# Cell 13 — Audit weather completeness for the primary cohort.
# No model is trained in this cell.

weather_columns = weather_features

def weather_completeness_report(df, split_name):
    complete_mask = df[weather_columns].notna().all(axis=1)

    print(f"{split_name}:")
    print(f"  Total rows: {len(df):,}")
    print(f"  Complete weather rows: {complete_mask.sum():,}")
    print(f"  Incomplete weather rows: {(~complete_mask).sum():,}")
    print(f"  Complete weather percentage: {complete_mask.mean() * 100:.2f}%")

    return complete_mask

primary_train_weather_mask = weather_completeness_report(
    primary_train, "Primary training"
)

primary_holdout_weather_mask = weather_completeness_report(
    primary_holdout, "Primary holdout"
)

print("\nMissing values per weather feature in primary training:")
display(primary_train[weather_columns].isna().sum().to_frame("Missing_Train"))

print("Missing values per weather feature in primary holdout:")
display(primary_holdout[weather_columns].isna().sum().to_frame("Missing_Holdout"))

Primary training:
  Total rows: 88,551
  Complete weather rows: 76,317
  Incomplete weather rows: 12,234
  Complete weather percentage: 86.18%
Primary holdout:
  Total rows: 18,716
  Complete weather rows: 16,230
  Incomplete weather rows: 2,486
  Complete weather percentage: 86.72%

Missing values per weather feature in primary training:


,Missing_Train
tmin_mean_c,12234
tmin_min_c,12234
tmax_mean_c,12234
tmax_max_c,12234
precipitation_total_mm,12234
precipitation_mean_mm_day,12234
precipitation_max_mm_day,12234
tmin_valid_days,12234
tmax_valid_days,12234
precipitation_valid_days,12234


Missing values per weather feature in primary holdout:


,Missing_Holdout
tmin_mean_c,2486
tmin_min_c,2486
tmax_mean_c,2486
tmax_max_c,2486
precipitation_total_mm,2486
precipitation_mean_mm_day,2486
precipitation_max_mm_day,2486
tmin_valid_days,2486
tmax_valid_days,2486
precipitation_valid_days,2486


In [15]:
# Cell 14 — Create matched complete-weather datasets.
# No model is trained in this cell.

# Restrict training and holdout to rows with all weather features available.
primary_train_weather = primary_train.loc[
    primary_train_weather_mask
].copy()

primary_holdout_weather = primary_holdout.loc[
    primary_holdout_weather_mask
].copy()

# Core-feature comparison must use exactly the same rows as the weather model.
X_train_core_weather = primary_train_weather[
    required_core_columns
]

X_holdout_core_weather = primary_holdout_weather[
    required_core_columns
]

# Core + weather feature matrices.
weather_model_features = required_core_columns + weather_columns

X_train_core_plus_weather = primary_train_weather[
    weather_model_features
]

X_holdout_core_plus_weather = primary_holdout_weather[
    weather_model_features
]

y_train_weather = primary_train_weather["Yield"].astype(float)
y_holdout_weather = primary_holdout_weather["Yield"].astype(float)

# Validate the matched datasets.
assert X_train_core_weather.index.equals(
    X_train_core_plus_weather.index
)
assert X_holdout_core_weather.index.equals(
    X_holdout_core_plus_weather.index
)
assert X_train_core_plus_weather[weather_columns].notna().all().all()
assert X_holdout_core_plus_weather[weather_columns].notna().all().all()

print("Matched weather experiment datasets created.")
print(f"Training rows: {len(X_train_core_weather):,}")
print(f"Holdout rows: {len(X_holdout_core_weather):,}")
print(f"Core feature count: {len(required_core_columns)}")
print(f"Core + weather feature count: {len(weather_model_features)}")
print("Weather features contain no missing values in these subsets.")
print("Core and weather models will use identical rows.")

Matched weather experiment datasets created.
Training rows: 76,317
Holdout rows: 16,230
Core feature count: 4
Core + weather feature count: 14
Weather features contain no missing values in these subsets.
Core and weather models will use identical rows.


In [16]:
# Cell 15 — Retrain HistGradientBoosting on the matched weather subset.
# This is the fair benchmark for the upcoming weather experiment.

matched_core_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

matched_core_pipeline.fit(
    X_train_core_weather,
    y_train_weather,
)

matched_core_predictions = matched_core_pipeline.predict(
    X_holdout_core_weather
)

append_result(
    baseline_results,
    "HistGradientBoosting Matched",
    "primary_weather_complete",
    "core",
    y_holdout_weather,
    matched_core_predictions,
)

matched_core_metrics = evaluate_regression(
    y_holdout_weather,
    matched_core_predictions,
)

print("Matched core-model benchmark:")
print(f"  Training rows: {len(X_train_core_weather):,}")
print(f"  Holdout rows: {len(X_holdout_core_weather):,}")
print(f"  MAE:  {matched_core_metrics['MAE']:.4f}")
print(f"  RMSE: {matched_core_metrics['RMSE']:.4f}")
print(f"  R²:   {matched_core_metrics['R2']:.4f}")
print(
    "  Predictions finite:",
    np.isfinite(matched_core_predictions).all(),
)

Matched core-model benchmark:
  Training rows: 76,317
  Holdout rows: 16,230
  MAE:  782.2365
  RMSE: 1106.4506
  R²:   0.3977
  Predictions finite: True


In [17]:

# Cell 16 — Test whether weather features improve the matched benchmark.

weather_hgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor.__class__(
            transformers=[
                (
                    "categorical",
                    core_encoder,
                    core_categorical_features,
                ),
                (
                    "numeric",
                    "passthrough",
                    core_numeric_features + weather_features,
                ),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

weather_hgb_pipeline.fit(
    X_train_core_plus_weather,
    y_train_weather,
)

weather_hgb_predictions = weather_hgb_pipeline.predict(
    X_holdout_core_plus_weather
)

append_result(
    baseline_results,
    "HistGradientBoosting + Weather",
    "primary_weather_complete",
    "core+weather",
    y_holdout_weather,
    weather_hgb_predictions,
)

weather_hgb_metrics = evaluate_regression(
    y_holdout_weather,
    weather_hgb_predictions,
)

mae_change = weather_hgb_metrics["MAE"] - matched_core_metrics["MAE"]
rmse_change = weather_hgb_metrics["RMSE"] - matched_core_metrics["RMSE"]
r2_change = weather_hgb_metrics["R2"] - matched_core_metrics["R2"]

mae_pct_change = 100 * mae_change / matched_core_metrics["MAE"]
rmse_pct_change = 100 * rmse_change / matched_core_metrics["RMSE"]

print("HistGradientBoosting + weather:")
print(f"  Training rows: {len(X_train_core_plus_weather):,}")
print(f"  Holdout rows: {len(X_holdout_core_plus_weather):,}")
print(f"  MAE:  {weather_hgb_metrics['MAE']:.4f}")
print(f"  RMSE: {weather_hgb_metrics['RMSE']:.4f}")
print(f"  R²:   {weather_hgb_metrics['R2']:.4f}")
print(f"  Predictions finite: {np.isfinite(weather_hgb_predictions).all()}")

print("\nChange versus matched core-only model:")
print(f"  MAE change:  {mae_change:+.4f} ({mae_pct_change:+.2f}%)")
print(f"  RMSE change: {rmse_change:+.4f} ({rmse_pct_change:+.2f}%)")
print(f"  R² change:   {r2_change:+.4f}")
print(
    "  Same evaluation rows:",
    X_holdout_core_plus_weather.index.equals(X_holdout_core_weather.index),
)


HistGradientBoosting + weather:
  Training rows: 76,317
  Holdout rows: 16,230
  MAE:  722.4263
  RMSE: 1092.8120
  R²:   0.4124
  Predictions finite: True

Change versus matched core-only model:
  MAE change:  -59.8103 (-7.65%)
  RMSE change: -13.6386 (-1.23%)
  R² change:   +0.0148
  Same evaluation rows: True


In [18]:

# Cell 17 — Review the current model leaderboard.

leaderboard = pd.DataFrame(baseline_results)

leaderboard = leaderboard.sort_values(
    by=["Cohort", "RMSE"],
    ascending=[True, True],
).reset_index(drop=True)

display(
    leaderboard.round(
        {
            "MAE": 3,
            "RMSE": 3,
            "R2": 4,
        }
    )
)

print("\nBest model by RMSE within each cohort:")
for cohort, group in leaderboard.groupby("Cohort", sort=False):
    best = group.loc[group["RMSE"].idxmin()]
    print(
        f"{cohort}: {best['Model']} | "
        f"RMSE={best['RMSE']:.3f}, "
        f"MAE={best['MAE']:.3f}, "
        f"R²={best['R2']:.4f}, "
        f"N={int(best['N_Evaluated']):,}"
    )


,Model,Cohort,Feature_Set,N_Evaluated,MAE,RMSE,R2
0,HistGradientBoosting,primary,core,18716,780.275,1111.999,0.4001
1,Random Forest,primary,core,18716,782.145,1114.360,0.3975
2,Decision Tree,primary,core,18716,780.469,1122.986,0.3881
3,Linear Regression,primary,core,18716,813.489,1137.790,0.3719
4,Ridge Regression,primary,core,18716,813.516,1137.803,0.3719
5,Crop-Season Median,primary,Crop + Season,18716,795.318,1183.838,0.3200
6,Global Median,primary,none,18716,1116.685,1512.974,-0.1106
7,HistGradientBoosting + Weather,primary_weather_complete,core+weather,16230,722.426,1092.812,0.4124
8,HistGradientBoosting Matched,primary_weather_complete,core,16230,782.237,1106.451,0.3977
9,HistGradientBoosting,sensitivity,core,30191,763.270,2704.800,0.1118



Best model by RMSE within each cohort:
primary: HistGradientBoosting | RMSE=1111.999, MAE=780.275, R²=0.4001, N=18,716
primary_weather_complete: HistGradientBoosting + Weather | RMSE=1092.812, MAE=722.426, R²=0.4124, N=16,230
sensitivity: HistGradientBoosting | RMSE=2704.800, MAE=763.270, R²=0.1118, N=30,191


In [19]:

# Cell 18 — Year-wise evaluation of the weather-enhanced model.

yearwise_df = primary_holdout_weather[
    ["FE_YearStart", "Yield"]
].copy()

yearwise_df["Actual_Yield"] = y_holdout_weather.to_numpy()
yearwise_df["Predicted_Yield"] = weather_hgb_predictions
yearwise_df["Absolute_Error"] = np.abs(
    yearwise_df["Actual_Yield"] - yearwise_df["Predicted_Yield"]
)
yearwise_df["Squared_Error"] = (
    yearwise_df["Actual_Yield"] - yearwise_df["Predicted_Yield"]
) ** 2

yearwise_results = []

for year, group in yearwise_df.groupby("FE_YearStart", sort=True):
    metrics = evaluate_regression(
        group["Actual_Yield"],
        group["Predicted_Yield"],
    )
    yearwise_results.append(
        {
            "Holdout_Year": int(year),
            "N": len(group),
            **metrics,
        }
    )

yearwise_results_df = pd.DataFrame(yearwise_results)

display(
    yearwise_results_df.round(
        {"MAE": 3, "RMSE": 3, "R2": 4}
    )
)

print("\nPrediction error by year:")
for _, row in yearwise_results_df.iterrows():
    print(
        f"{int(row['Holdout_Year'])}-{int(row['Holdout_Year']) + 1}: "
        f"N={int(row['N']):,}, "
        f"MAE={row['MAE']:.2f}, "
        f"RMSE={row['RMSE']:.2f}, "
        f"R²={row['R2']:.4f}"
    )



,Holdout_Year,N,MAE,RMSE,R2
0,2023,8116,717.933,1097.377,0.4167
1,2024,8114,726.920,1088.226,0.4069



Prediction error by year:
2023-2024: N=8,116, MAE=717.93, RMSE=1097.38, R²=0.4167
2024-2025: N=8,114, MAE=726.92, RMSE=1088.23, R²=0.4069


In [20]:

# Cell 19 — Crop-level error analysis on the weather-complete holdout.

crop_error_df = primary_holdout_weather[
    ["Crop", "Yield"]
].copy()

crop_error_df["Actual_Yield"] = y_holdout_weather.to_numpy()
crop_error_df["Predicted_Yield"] = weather_hgb_predictions
crop_error_df["Absolute_Error"] = np.abs(
    crop_error_df["Actual_Yield"] - crop_error_df["Predicted_Yield"]
)
crop_error_df["Squared_Error"] = (
    crop_error_df["Actual_Yield"] - crop_error_df["Predicted_Yield"]
) ** 2

crop_error_results = (
    crop_error_df.groupby("Crop", dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        Actual_Mean=("Actual_Yield", "mean"),
        Predicted_Mean=("Predicted_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        MSE=("Squared_Error", "mean"),
    )
    .reset_index()
)

crop_error_results["RMSE"] = np.sqrt(crop_error_results["MSE"])
crop_error_results = crop_error_results.drop(columns="MSE")

# Require enough observations to avoid overinterpreting tiny groups.
crop_error_results = crop_error_results[
    crop_error_results["N"] >= 100
].sort_values("MAE", ascending=False).reset_index(drop=True)

print("Crops with the highest MAE (minimum 100 holdout rows):")
display(
    crop_error_results.head(15).round(
        {
            "Actual_Mean": 2,
            "Predicted_Mean": 2,
            "MAE": 2,
            "RMSE": 2,
        }
    )
)


Crops with the highest MAE (minimum 100 holdout rows):


,Crop,N,Actual_Mean,Predicted_Mean,MAE,RMSE
0,Maize,1347,3444.75,3507.60,1069.38,1493.40
1,Nutri/Coarse Cereals,1862,2827.98,2851.03,1010.21,1431.51
2,Total Food Grains,2935,2477.79,2626.41,838.49,1168.53
3,Cereals,2683,2919.82,2995.09,765.11,1151.39
4,Rice,1529,2792.97,3012.64,690.39,1116.94
5,Ragi,142,1324.55,1710.28,663.39,879.96
6,Bajra,365,1774.62,2012.55,636.16,943.79
7,Jowar,412,1434.96,1628.17,625.67,872.63
8,Shree Anna /Nutri Cereals,835,1499.50,1749.28,624.58,912.39
9,Wheat,756,3086.68,3099.26,609.45,784.11


In [21]:

# Cell 20 — Crop-level bias and relative error diagnostics.

crop_error_df["Signed_Error"] = (
    crop_error_df["Predicted_Yield"] - crop_error_df["Actual_Yield"]
)

crop_bias_results = (
    crop_error_df.groupby("Crop", dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        Actual_Mean=("Actual_Yield", "mean"),
        Predicted_Mean=("Predicted_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Overprediction_Rate=(
            "Signed_Error",
            lambda values: (values > 0).mean() * 100,
        ),
        Underprediction_Rate=(
            "Signed_Error",
            lambda values: (values < 0).mean() * 100,
        ),
    )
    .reset_index()
)

crop_bias_results = crop_bias_results[
    crop_bias_results["N"] >= 100
].copy()

crop_bias_results["MAE_as_pct_of_actual_mean"] = (
    100
    * crop_bias_results["MAE"]
    / crop_bias_results["Actual_Mean"].abs().replace(0, np.nan)
)

crop_bias_results["Mean_Bias_as_pct_of_actual_mean"] = (
    100
    * crop_bias_results["Mean_Signed_Error"]
    / crop_bias_results["Actual_Mean"].abs().replace(0, np.nan)
)

crop_bias_results = crop_bias_results.sort_values(
    "MAE_as_pct_of_actual_mean",
    ascending=False,
).reset_index(drop=True)

print("Crops with the largest MAE relative to their mean actual yield:")
display(
    crop_bias_results[
        [
            "Crop",
            "N",
            "Actual_Mean",
            "Predicted_Mean",
            "MAE",
            "MAE_as_pct_of_actual_mean",
            "Mean_Signed_Error",
            "Mean_Bias_as_pct_of_actual_mean",
            "Overprediction_Rate",
            "Underprediction_Rate",
        ]
    ].head(15).round(2)
)


Crops with the largest MAE relative to their mean actual yield:


,Crop,N,Actual_Mean,Predicted_Mean,MAE,MAE_as_pct_of_actual_mean,Mean_Signed_Error,Mean_Bias_as_pct_of_actual_mean,Overprediction_Rate,Underprediction_Rate
0,Other Pulses,546,1036.55,1360.19,567.07,54.71,323.64,31.22,74.18,25.82
1,Ragi,142,1324.55,1710.28,663.39,50.08,385.73,29.12,78.87,21.13
2,Jowar,412,1434.96,1628.17,625.67,43.60,193.21,13.46,64.32,35.68
3,Urad,649,739.86,888.24,317.10,42.86,148.39,20.06,66.72,33.28
4,Shree Anna /Nutri Cereals,835,1499.50,1749.28,624.58,41.65,249.78,16.66,67.07,32.93
5,Tur,418,977.80,1206.61,390.93,39.98,228.81,23.40,72.49,27.51
6,Moong,534,758.69,853.10,281.30,37.08,94.41,12.44,63.48,36.52
7,Bajra,365,1774.62,2012.55,636.16,35.85,237.93,13.41,60.82,39.18
8,Nutri/Coarse Cereals,1862,2827.98,2851.03,1010.21,35.72,23.05,0.81,58.22,41.78
9,Total Food Grains,2935,2477.79,2626.41,838.49,33.84,148.63,6.00,55.09,44.91


In [22]:

# Cell 21 — Compare weather HGB against a Crop–Season median baseline
# on exactly the same weather-complete holdout rows.

matched_baseline_df = primary_holdout_weather[
    ["Crop", "Season", "Yield"]
].copy()

matched_baseline_df["Actual_Yield"] = y_holdout_weather.to_numpy()
matched_baseline_df["Weather_HGB_Prediction"] = weather_hgb_predictions

# Use only training data to construct the baseline lookup.
matched_group_medians = (
    primary_train_weather
    .groupby(["Crop", "Season"])["Yield"]
    .median()
)

matched_global_median = float(primary_train_weather["Yield"].median())

matched_baseline_df["Crop_Season_Median_Prediction"] = [
    matched_group_medians.get((crop, season), matched_global_median)
    for crop, season in zip(
        matched_baseline_df["Crop"],
        matched_baseline_df["Season"],
    )
]

matched_baseline_df["HGB_Absolute_Error"] = np.abs(
    matched_baseline_df["Actual_Yield"]
    - matched_baseline_df["Weather_HGB_Prediction"]
)

matched_baseline_df["Median_Absolute_Error"] = np.abs(
    matched_baseline_df["Actual_Yield"]
    - matched_baseline_df["Crop_Season_Median_Prediction"]
)

matched_crop_comparison = (
    matched_baseline_df.groupby("Crop")
    .agg(
        N=("Actual_Yield", "size"),
        Actual_Mean=("Actual_Yield", "mean"),
        HGB_MAE=("HGB_Absolute_Error", "mean"),
        Crop_Season_Median_MAE=("Median_Absolute_Error", "mean"),
    )
    .reset_index()
)

matched_crop_comparison["MAE_Improvement"] = (
    matched_crop_comparison["Crop_Season_Median_MAE"]
    - matched_crop_comparison["HGB_MAE"]
)

matched_crop_comparison["MAE_Improvement_Pct"] = (
    100
    * matched_crop_comparison["MAE_Improvement"]
    / matched_crop_comparison["Crop_Season_Median_MAE"].replace(0, np.nan)
)

matched_crop_comparison = matched_crop_comparison[
    matched_crop_comparison["N"] >= 100
].sort_values("MAE_Improvement_Pct").reset_index(drop=True)

print("Crops where weather HGB adds the least value versus Crop–Season median:")
display(
    matched_crop_comparison.head(15).round(
        {
            "Actual_Mean": 2,
            "HGB_MAE": 2,
            "Crop_Season_Median_MAE": 2,
            "MAE_Improvement": 2,
            "MAE_Improvement_Pct": 2,
        }
    )
)

print("\nOverall matched comparison:")
print(
    "Weather HGB MAE:",
    round(
        mean_absolute_error(
            matched_baseline_df["Actual_Yield"],
            matched_baseline_df["Weather_HGB_Prediction"],
        ),
        3,
    ),
)
print(
    "Crop–Season median MAE:",
    round(
        mean_absolute_error(
            matched_baseline_df["Actual_Yield"],
            matched_baseline_df["Crop_Season_Median_Prediction"],
        ),
        3,
    ),
)


Crops where weather HGB adds the least value versus Crop–Season median:


,Crop,N,Actual_Mean,HGB_MAE,Crop_Season_Median_MAE,MAE_Improvement,MAE_Improvement_Pct
0,Other Pulses,546,1036.55,567.07,375.56,-191.50,-50.99
1,Tur,418,977.80,390.93,269.72,-121.20,-44.94
2,Urad,649,739.86,317.10,228.68,-88.42,-38.66
3,Ragi,142,1324.55,663.39,503.97,-159.42,-31.63
4,Small Millets,146,1029.08,340.90,280.14,-60.75,-21.69
5,Gram,529,1278.43,372.22,307.92,-64.30,-20.88
6,Lentil,314,962.47,244.82,207.58,-37.24,-17.94
7,Moong,534,758.69,281.30,239.92,-41.39,-17.25
8,Jowar,412,1434.96,625.67,585.62,-40.04,-6.84
9,Shree Anna /Nutri Cereals,835,1499.50,624.58,613.40,-11.19,-1.82



Overall matched comparison:
Weather HGB MAE: 722.426
Crop–Season median MAE: 791.051


In [23]:

# Cell 22 — Summarize crop-level wins without changing the model.

valid_crop_comparison = matched_crop_comparison.dropna(
    subset=["HGB_MAE", "Crop_Season_Median_MAE"]
).copy()

valid_crop_comparison["Winning_Method"] = np.where(
    valid_crop_comparison["HGB_MAE"] < valid_crop_comparison["Crop_Season_Median_MAE"],
    "Weather HGB",
    np.where(
        valid_crop_comparison["HGB_MAE"] > valid_crop_comparison["Crop_Season_Median_MAE"],
        "Crop–Season Median",
        "Tie",
    ),
)

crop_win_counts = (
    valid_crop_comparison["Winning_Method"]
    .value_counts()
    .rename_axis("Method")
    .reset_index(name="Crop_Groups_Won")
)

print("Crop-level wins among groups with at least 100 holdout rows:")
display(crop_win_counts)

print(
    "\nGroups evaluated:",
    len(valid_crop_comparison),
)
print(
    "Groups where weather HGB wins:",
    int((valid_crop_comparison["Winning_Method"] == "Weather HGB").sum()),
)
print(
    "Groups where Crop–Season median wins:",
    int((valid_crop_comparison["Winning_Method"] == "Crop–Season Median").sum()),
)

overall_hgb_mae = mean_absolute_error(
    matched_baseline_df["Actual_Yield"],
    matched_baseline_df["Weather_HGB_Prediction"],
)
overall_median_mae = mean_absolute_error(
    matched_baseline_df["Actual_Yield"],
    matched_baseline_df["Crop_Season_Median_Prediction"],
)

print("\nOverall matched holdout MAE:")
print(f"  Weather HGB:       {overall_hgb_mae:.3f}")
print(f"  Crop–Season median: {overall_median_mae:.3f}")
print(
    "  Relative MAE reduction:",
    f"{100 * (overall_median_mae - overall_hgb_mae) / overall_median_mae:.2f}%",
)


Crop-level wins among groups with at least 100 holdout rows:


,Method,Crop_Groups_Won
0,Crop–Season Median,10
1,Weather HGB,8



Groups evaluated: 18
Groups where weather HGB wins: 8
Groups where Crop–Season median wins: 10

Overall matched holdout MAE:
  Weather HGB:       722.426
  Crop–Season median: 791.051
  Relative MAE reduction: 8.68%


In [24]:

# Cell 23 — Error distribution for the weather-enhanced HGB model.

weather_error_analysis = matched_baseline_df[
    ["Crop", "Season", "Actual_Yield", "Weather_HGB_Prediction"]
].copy()

weather_error_analysis["Signed_Error"] = (
    weather_error_analysis["Weather_HGB_Prediction"]
    - weather_error_analysis["Actual_Yield"]
)

weather_error_analysis["Absolute_Error"] = (
    weather_error_analysis["Signed_Error"].abs()
)

error_quantiles = weather_error_analysis["Absolute_Error"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Absolute error quantiles:")
display(
    error_quantiles.rename_axis("Quantile")
    .rename("Absolute_Error")
    .to_frame()
    .round(2)
)

print("\nSigned-error summary (positive means overprediction):")
display(
    weather_error_analysis["Signed_Error"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    .to_frame("Signed_Error")
    .round(2)
)

print("\nShare of total absolute error contributed by the worst 5% of rows:")
sorted_abs_errors = weather_error_analysis["Absolute_Error"].sort_values(
    ascending=False
)
worst_5_count = max(1, int(np.ceil(0.05 * len(sorted_abs_errors))))

worst_5_share = (
    sorted_abs_errors.head(worst_5_count).sum()
    / sorted_abs_errors.sum()
)

print(f"  Rows in worst 5%: {worst_5_count:,}")
print(f"  Share of total absolute error: {100 * worst_5_share:.2f}%")


Absolute error quantiles:


,Absolute_Error
Quantile,
0.50,496.15
0.75,941.32
0.90,1545.87
0.95,2094.93
0.99,4126.13



Signed-error summary (positive means overprediction):


,Signed_Error
count,16230.00
mean,131.91
std,1084.86
min,-7508.43
50%,150.55
90%,1165.99
95%,1609.37
99%,3353.29
max,9572.42



Share of total absolute error contributed by the worst 5% of rows:
  Rows in worst 5%: 812
  Share of total absolute error: 23.22%


In [25]:

# Cell 24 — Inspect the most severe holdout prediction errors.

worst_predictions = weather_error_analysis.copy()

worst_predictions["FE_YearStart"] = (
    primary_holdout_weather.loc[
        worst_predictions.index, "FE_YearStart"
    ].to_numpy()
)

worst_predictions["District"] = (
    primary_holdout_weather.loc[
        worst_predictions.index, "District"
    ].to_numpy()
)

worst_predictions["Actual_Yield"] = (
    worst_predictions["Actual_Yield"].astype(float)
)

worst_predictions["Predicted_Yield"] = (
    worst_predictions["Weather_HGB_Prediction"].astype(float)
)

worst_predictions = worst_predictions.sort_values(
    "Absolute_Error",
    ascending=False,
).head(20)

display(
    worst_predictions[
        [
            "District",
            "Crop",
            "Season",
            "FE_YearStart",
            "Actual_Yield",
            "Predicted_Yield",
            "Signed_Error",
            "Absolute_Error",
        ]
    ].round(2).reset_index(drop=True)
)


,District,Crop,Season,FE_YearStart,Actual_Yield,Predicted_Yield,Signed_Error,Absolute_Error
0,Tirupati,Nutri/Coarse Cereals,Kharif,2023,3590.0,13162.42,9572.42,9572.42
1,Prakasam,Total Food Grains,Kharif,2024,1094.0,10049.06,8955.06,8955.06
2,Tirupati,Cereals,Kharif,2023,3745.0,12603.31,8858.31,8858.31
3,Tirupati,Rice,Kharif,2023,3754.0,12523.65,8769.65,8769.65
4,Tirupati,Total Food Grains,Kharif,2023,3740.0,12206.82,8466.82,8466.82
5,Prakasam,Maize,Kharif,2024,4127.0,12484.84,8357.84,8357.84
6,Prakasam,Nutri/Coarse Cereals,Kharif,2024,2675.0,11021.89,8346.89,8346.89
7,Prakasam,Maize,Kharif,2023,3460.0,11787.28,8327.28,8327.28
8,Dr. B.R. Ambedkar Konaseema,Cereals,Kharif,2023,3518.0,11769.24,8251.24,8251.24
9,Prakasam,Total Food Grains,Kharif,2023,1022.0,9258.50,8236.50,8236.50


In [26]:

# Cell 25 — Audit district/year error concentration and crop-category overlap.

district_year_errors = weather_error_analysis.copy()

district_year_errors["District"] = primary_holdout_weather.loc[
    district_year_errors.index, "District"
].to_numpy()

district_year_errors["FE_YearStart"] = primary_holdout_weather.loc[
    district_year_errors.index, "FE_YearStart"
].to_numpy()

district_year_errors["Signed_Error"] = (
    district_year_errors["Weather_HGB_Prediction"]
    - district_year_errors["Actual_Yield"]
)

district_year_summary = (
    district_year_errors.groupby(
        ["District", "FE_YearStart"], dropna=False
    )
    .agg(
        N=("Actual_Yield", "size"),
        Actual_Mean=("Actual_Yield", "mean"),
        Predicted_Mean=("Weather_HGB_Prediction", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
    )
    .reset_index()
)

district_year_summary = district_year_summary[
    district_year_summary["N"] >= 5
].sort_values("MAE", ascending=False)

print("District-year groups with the highest MAE (at least 5 records):")
display(district_year_summary.head(15).round(2))

print("\nCrop labels among the 20 worst individual errors:")
worst_crop_counts = (
    worst_predictions["Crop"]
    .value_counts(dropna=False)
    .rename_axis("Crop")
    .reset_index(name="Count_in_Worst_20")
)
display(worst_crop_counts)

print("\nAggregate-like labels among the 20 worst errors:")
aggregate_like_labels = {
    "Total Food Grains",
    "Cereals",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Small Millets",
}

worst_aggregate_count = worst_predictions["Crop"].isin(
    aggregate_like_labels
).sum()

print(
    f"{worst_aggregate_count} of {len(worst_predictions)} "
    "worst-error rows use aggregate-like or broad category labels."
)


District-year groups with the highest MAE (at least 5 records):


,District,FE_YearStart,N,Actual_Mean,Predicted_Mean,MAE,Mean_Signed_Error,Max_Absolute_Error
1065,South Goa,2023,6,2801.00,9127.12,6326.12,6326.12,8195.01
1133,Tirupati,2023,9,3684.33,9963.61,6279.28,6279.28,9572.42
809,North Goa,2023,7,2626.43,8621.72,5995.29,5995.29,7808.01
1066,South Goa,2024,6,2287.50,8027.57,5740.07,5740.07,7466.79
810,North Goa,2024,7,2359.00,7584.93,5225.93,5225.93,6807.79
874,Prakasam,2024,19,2347.53,7201.16,4853.63,4853.63,8955.06
873,Prakasam,2023,20,2115.75,6849.62,4733.87,4733.87,8327.28
911,Ramanathapuram,2023,7,1220.00,5145.62,3925.62,3925.62,6429.82
1213,West Godavari,2023,7,4127.57,7844.09,3719.20,3716.52,8097.24
1134,Tirupati,2024,9,3522.22,7100.37,3578.15,3578.15,5730.81



Crop labels among the 20 worst individual errors:


,Crop,Count_in_Worst_20
0,Cereals,6
1,Rice,5
2,Total Food Grains,5
3,Nutri/Coarse Cereals,2
4,Maize,2



Aggregate-like labels among the 20 worst errors:
13 of 20 worst-error rows use aggregate-like or broad category labels.


In [28]:

# Cell 26 — Quantify how aggregate-like crop labels affect holdout errors.

aggregate_like_labels = {
    "Total Food Grains",
    "Cereals",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Small Millets",
}

aggregate_mask = primary_holdout_weather["Crop"].isin(
    aggregate_like_labels
).to_numpy()

aggregate_audit_df = pd.DataFrame(
    {
        "Category_Type": np.where(
            aggregate_mask,
            "Aggregate-like / broad label",
            "Other crop label",
        ),
        "Actual_Yield": y_holdout_weather.to_numpy(),
        "Predicted_Yield": weather_hgb_predictions,
    },
    index=primary_holdout_weather.index,
)

aggregate_audit_df["Absolute_Error"] = np.abs(
    aggregate_audit_df["Predicted_Yield"]
    - aggregate_audit_df["Actual_Yield"]
)

aggregate_audit_df["Signed_Error"] = (
    aggregate_audit_df["Predicted_Yield"]
    - aggregate_audit_df["Actual_Yield"]
)

aggregate_summary = (
    aggregate_audit_df.groupby("Category_Type")
    .agg(
        N=("Actual_Yield", "size"),
        Actual_Mean=("Actual_Yield", "mean"),
        Predicted_Mean=("Predicted_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Signed_Error=("Signed_Error", "mean"),
    )
    .reset_index()
)

aggregate_summary["Share_of_Holdout"] = (
    aggregate_summary["N"] / len(aggregate_audit_df) * 100
)

aggregate_summary["Share_of_Total_Absolute_Error"] = (
    aggregate_audit_df.groupby("Category_Type")["Absolute_Error"].sum().values
    / aggregate_audit_df["Absolute_Error"].sum()
    * 100
)

display(
    aggregate_summary.round(
        {
            "Actual_Mean": 2,
            "Predicted_Mean": 2,
            "MAE": 2,
            "Mean_Signed_Error": 2,
            "Share_of_Holdout": 2,
            "Share_of_Total_Absolute_Error": 2,
        }
    )
)

print("\nOverall absolute-error total:", round(
    aggregate_audit_df["Absolute_Error"].sum(), 2
))
print(
    "Aggregate-like rows:",
    int(aggregate_mask.sum()),
    f"({100 * aggregate_mask.mean():.2f}% of matched holdout)",
)


,Category_Type,N,Actual_Mean,Predicted_Mean,MAE,Mean_Signed_Error,Share_of_Holdout,Share_of_Total_Absolute_Error
0,Aggregate-like / broad label,8461,2573.48,2680.32,823.31,106.84,52.13,59.41
1,Other crop label,7769,2073.64,2232.84,612.55,159.21,47.87,40.59



Overall absolute-error total: 11724978.05
Aggregate-like rows: 8461 (52.13% of matched holdout)


In [29]:

# Cell 27 — Inspect crop-label frequencies across train and holdout.

crop_frequency_audit = pd.concat(
    [
        primary_train.assign(Split="Train"),
        primary_holdout_weather.assign(Split="Weather-complete holdout"),
    ],
    ignore_index=True,
)

crop_frequency_summary = (
    crop_frequency_audit.groupby(["Split", "Crop"], dropna=False)
    .agg(
        N=("Yield", "size"),
        Mean_Yield=("Yield", "mean"),
        Median_Yield=("Yield", "median"),
        Min_Yield=("Yield", "min"),
        Max_Yield=("Yield", "max"),
    )
    .reset_index()
)

print("Most frequent crop labels in training:")
display(
    crop_frequency_summary[
        crop_frequency_summary["Split"] == "Train"
    ]
    .sort_values("N", ascending=False)
    .head(25)
    .round(2)
)

print("\nMost frequent crop labels in the weather-complete holdout:")
display(
    crop_frequency_summary[
        crop_frequency_summary["Split"] == "Weather-complete holdout"
    ]
    .sort_values("N", ascending=False)
    .head(25)
    .round(2)
)

print("\nCrop labels present in holdout but absent from training:")
train_crop_labels = set(primary_train["Crop"].dropna().unique())
holdout_crop_labels = set(primary_holdout_weather["Crop"].dropna().unique())

print(sorted(holdout_crop_labels - train_crop_labels))


Most frequent crop labels in training:


,Split,Crop,N,Mean_Yield,Median_Yield,Min_Yield,Max_Yield
14,Train,Total Food Grains,15513,2299.41,2156.0,17.0,44815.0
2,Train,Cereals,14711,2624.76,2497.0,19.0,44815.0
8,Train,Nutri/Coarse Cereals,9502,2498.93,2097.0,19.0,12525.0
11,Train,Rice,9302,2554.48,2509.0,97.0,44815.0
6,Train,Maize,6777,3147.16,2518.0,42.0,13752.0
12,Train,Shree Anna /Nutri Cereals,4624,1381.64,1199.0,31.0,8139.0
17,Train,Wheat,4138,2845.03,2811.0,161.0,6010.0
16,Train,Urad,3452,658.60,635.0,33.0,3000.0
9,Train,Other Pulses,2989,1153.72,928.0,23.0,24581.0
3,Train,Gram,2840,1152.01,1121.5,103.0,3156.0



Most frequent crop labels in the weather-complete holdout:


,Split,Crop,N,Mean_Yield,Median_Yield,Min_Yield,Max_Yield
32,Weather-complete holdout,Total Food Grains,2935,2477.79,2352.0,18.0,8147.0
20,Weather-complete holdout,Cereals,2683,2919.82,2835.0,51.0,9510.0
26,Weather-complete holdout,Nutri/Coarse Cereals,1862,2827.98,2415.0,276.0,12237.0
29,Weather-complete holdout,Rice,1529,2792.97,2796.0,465.0,5684.0
24,Weather-complete holdout,Maize,1347,3444.75,2923.0,543.0,12237.0
30,Weather-complete holdout,Shree Anna /Nutri Cereals,835,1499.50,1271.0,114.0,8057.0
35,Weather-complete holdout,Wheat,756,3086.68,3157.5,534.0,5544.0
34,Weather-complete holdout,Urad,649,739.86,662.0,122.0,1917.0
27,Weather-complete holdout,Other Pulses,546,1036.55,926.0,138.0,7962.0
25,Weather-complete holdout,Moong,534,758.69,731.0,108.0,2199.0



Crop labels present in holdout but absent from training:
[]


In [30]:

# Cell 28 — Inspect extreme yields and their underlying records.

audit_columns = [
    col for col in [
        "State", "District", "Crop", "Season", "Year",
        "FE_YearStart", "Yield", "Area", "Production",
    ]
    if col in integrated_df.columns
]

extreme_yield_rows = integrated_df.loc[
    pd.to_numeric(integrated_df["Yield"], errors="coerce") >= 40000,
    audit_columns,
].copy()

extreme_yield_rows["Yield"] = pd.to_numeric(
    extreme_yield_rows["Yield"], errors="coerce"
)

print("Number of integrated records with Yield >= 40,000:",
      len(extreme_yield_rows))

print("\nExtreme-yield records, sorted by yield:")
display(
    extreme_yield_rows.sort_values(
        ["Yield", "District", "Crop"],
        ascending=[False, True, True],
    ).head(40)
)

print("\nCount of extreme records by crop:")
display(
    extreme_yield_rows.groupby("Crop", dropna=False)
    .agg(
        N=("Yield", "size"),
        Min_Yield=("Yield", "min"),
        Max_Yield=("Yield", "max"),
        Distinct_Districts=("District", "nunique"),
        Distinct_Years=("Year", "nunique"),
    )
    .sort_values("N", ascending=False)
    .round(2)
)

print("\nExact maximum yield in each modeling split:")
for split_name, split_df in [
    ("Primary train", primary_train),
    ("Weather-complete holdout", primary_holdout_weather),
]:
    split_yield = pd.to_numeric(split_df["Yield"], errors="coerce")
    max_yield = split_yield.max()

    print(f"\n{split_name}: maximum Yield = {max_yield}")
    display(
        split_df.loc[
            split_yield.eq(max_yield),
            audit_columns,
        ].head(20)
    )


Number of integrated records with Yield >= 40,000: 13

Extreme-yield records, sorted by yield:


,State,District,Crop,Season,Year,Yield,Area,Production
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,385540.0,0.00,0.00
313625,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,385540.0,0.00,0.00
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,161579.0,0.00,0.00
313621,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,2023-24,161579.0,0.00,0.00
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,105588.0,0.00,0.00
313610,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,105588.0,0.00,0.00
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,103571.0,0.00,0.01
265537,Madhya Pradesh,Narmadapuram,Other Pulses,Total,2021-22,103571.0,0.00,0.01
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,98889.0,0.00,0.01
264974,Madhya Pradesh,Harda,Other Pulses,Total,2021-22,98889.0,0.00,0.01



Count of extreme records by crop:


,N,Min_Yield,Max_Yield,Distinct_Districts,Distinct_Years
Crop,,,,,
Other Pulses,4,98889.0,103571.0,2,1
Moong,2,105588.0,105588.0,1,1
Total Food Grains,2,44815.0,161579.0,2,2
Urad,2,385540.0,385540.0,1,1
Cereals,1,44815.0,44815.0,1,1
Rice,1,44815.0,44815.0,1,1
Total Pulses,1,161579.0,161579.0,1,1



Exact maximum yield in each modeling split:

Primary train: maximum Yield = 44815.0


,State,District,Crop,Season,Year,Yield,Area,Production
288718,Goa,North Goa,Cereals,Kharif,2022-23,44815.0,0.01,0.47
288727,Goa,North Goa,Rice,Kharif,2022-23,44815.0,0.01,0.47
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,44815.0,0.01,0.47



Weather-complete holdout: maximum Yield = 12237.0


,State,District,Crop,Season,Year,Yield,Area,Production
375529,West Bengal,Dakshin Dinajpur,Maize,Summer,2024-25,12237.0,0.02,0.29
375535,West Bengal,Dakshin Dinajpur,Nutri/Coarse Cereals,Summer,2024-25,12237.0,0.02,0.29


In [31]:

# Cell 29 — Audit extreme positive-area records and cross-crop duplicates.

audit_cols = [
    "State", "District", "Crop", "Season", "Year",
    "Yield", "Area", "Production",
]
audit_cols = [c for c in audit_cols if c in primary_df.columns]

primary_audit = primary_df.copy()

for col in ["Yield", "Area", "Production"]:
    primary_audit[col] = pd.to_numeric(
        primary_audit[col], errors="coerce"
    )

print("Primary cohort rows:", len(primary_audit))
print(
    "Rows with Yield >= 10,000:",
    int((primary_audit["Yield"] >= 10000).sum()),
)
print(
    "Rows with Yield >= 10,000 and Area <= 0.05:",
    int(
        (
            (primary_audit["Yield"] >= 10000)
            & (primary_audit["Area"] <= 0.05)
        ).sum()
    ),
)
print(
    "Rows with Yield >= 10,000 and Production <= 1:",
    int(
        (
            (primary_audit["Yield"] >= 10000)
            & (primary_audit["Production"] <= 1)
        ).sum()
    ),
)

extreme_positive_records = primary_audit.loc[
    primary_audit["Yield"] >= 10000,
    audit_cols,
].sort_values("Yield", ascending=False)

print("\nHighest-yield records that passed the primary cohort filters:")
display(extreme_positive_records.head(30))

# Find records identical on location, season, year, and numeric
# yield/area/production, but carrying different crop labels.
duplicate_key_cols = [
    c for c in [
        "State", "District", "Season", "Year",
        "Yield", "Area", "Production",
    ]
    if c in primary_audit.columns
]

cross_crop_groups = (
    primary_audit.groupby(duplicate_key_cols, dropna=False)
    .agg(
        N_Rows=("Crop", "size"),
        N_Crop_Labels=("Crop", "nunique"),
        Crop_Labels=(
            "Crop",
            lambda s: " | ".join(sorted(s.dropna().astype(str).unique())),
        ),
    )
    .reset_index()
)

cross_crop_groups = cross_crop_groups[
    (cross_crop_groups["N_Rows"] > 1)
    & (cross_crop_groups["N_Crop_Labels"] > 1)
].sort_values(
    ["Yield", "N_Crop_Labels"],
    ascending=False,
)

print(
    "\nIdentical location/year/season/yield/area/production groups "
    "with multiple crop labels:",
    len(cross_crop_groups),
)
display(cross_crop_groups.head(25))


Primary cohort rows: 107267
Rows with Yield >= 10,000: 103
Rows with Yield >= 10,000 and Area <= 0.05: 47
Rows with Yield >= 10,000 and Production <= 1: 55

Highest-yield records that passed the primary cohort filters:


,State,District,Crop,Season,Year,Yield,Area,Production
288727,Goa,North Goa,Rice,Kharif,2022-23,44815.0,0.01,0.47
288718,Goa,North Goa,Cereals,Kharif,2022-23,44815.0,0.01,0.47
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,44815.0,0.01,0.47
288736,Goa,South Goa,Cereals,Kharif,2022-23,39221.0,0.01,0.47
288750,Goa,South Goa,Total Food Grains,Kharif,2022-23,39221.0,0.01,0.47
288745,Goa,South Goa,Rice,Kharif,2022-23,39221.0,0.01,0.47
196743,Haryana,Panipat,Other Pulses,Kharif,2019-20,24581.0,0.01,0.16
196457,Haryana,Jind,Other Pulses,Kharif,2019-20,23524.0,0.01,0.12
165387,Haryana,Panipat,Other Pulses,Kharif,2018-19,20267.0,0.01,0.12
196880,Haryana,Sonipat,Other Pulses,Kharif,2019-20,20143.0,0.02,0.35



Identical location/year/season/yield/area/production groups with multiple crop labels: 10526


,State,District,Season,Year,Yield,Area,Production,N_Rows,N_Crop_Labels,Crop_Labels
17531,Goa,North Goa,Kharif,2022-23,44815.0,0.01,0.47,3,3,Cereals | Rice | Total Food Grains
17561,Goa,South Goa,Kharif,2022-23,39221.0,0.01,0.47,3,3,Cereals | Rice | Total Food Grains
91515,West Bengal,Dakshin Dinajpur,Summer,2022-23,12525.0,0.05,0.58,2,2,Maize | Nutri/Coarse Cereals
91523,West Bengal,Dakshin Dinajpur,Summer,2024-25,12237.0,0.02,0.29,2,2,Maize | Nutri/Coarse Cereals
91519,West Bengal,Dakshin Dinajpur,Summer,2023-24,11620.0,0.03,0.33,2,2,Maize | Nutri/Coarse Cereals
92456,West Bengal,Malda,Rabi,2024-25,11295.0,0.30,3.34,2,2,Maize | Nutri/Coarse Cereals
14162,Bihar,Supaul,Rabi,2019-20,11139.0,0.05,0.51,2,2,Maize | Nutri/Coarse Cereals
72457,Telangana,Mulugu,Rabi,2019-20,10958.0,0.03,0.29,2,2,Maize | Nutri/Coarse Cereals
94090,West Bengal,Uttar Dinajpur,Rabi,2021-22,10550.0,0.40,4.20,2,2,Maize | Nutri/Coarse Cereals
94082,West Bengal,Uttar Dinajpur,Rabi,2020-21,10500.0,0.40,4.20,2,2,Maize | Nutri/Coarse Cereals


In [32]:

# Cell 30 — Diagnose the relationship between recorded Area and extreme Yield.

area_yield_audit = primary_df[["Area", "Production", "Yield"]].copy()

for col in ["Area", "Production", "Yield"]:
    area_yield_audit[col] = pd.to_numeric(
        area_yield_audit[col], errors="coerce"
    )

area_yield_audit = area_yield_audit.loc[
    np.isfinite(area_yield_audit["Area"])
    & np.isfinite(area_yield_audit["Production"])
    & np.isfinite(area_yield_audit["Yield"])
    & (area_yield_audit["Area"] > 0)
    & (area_yield_audit["Production"] > 0)
].copy()

area_yield_audit["Area_Band"] = pd.cut(
    area_yield_audit["Area"],
    bins=[0, 0.01, 0.05, 0.1, 0.5, 1, np.inf],
    labels=[
        "(0, 0.01]",
        "(0.01, 0.05]",
        "(0.05, 0.1]",
        "(0.1, 0.5]",
        "(0.5, 1]",
        "> 1",
    ],
    include_lowest=True,
)

area_band_summary = (
    area_yield_audit.groupby("Area_Band", observed=False)
    .agg(
        N=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        P95_Yield=("Yield", lambda s: s.quantile(0.95)),
        Max_Yield=("Yield", "max"),
        N_Yield_Above_10000=("Yield", lambda s: (s >= 10000).sum()),
    )
    .reset_index()
)

area_band_summary["Pct_Yield_Above_10000"] = (
    100
    * area_band_summary["N_Yield_Above_10000"]
    / area_band_summary["N"].replace(0, np.nan)
)

print("Yield distribution by recorded Area band:")
display(area_band_summary.round(3))

print("\nArea distribution in the primary cohort:")
display(
    area_yield_audit["Area"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("Area")
    .round(4)
)

print("\nYield >= 10,000: area and production summary:")
display(
    area_yield_audit.loc[
        area_yield_audit["Yield"] >= 10000,
        ["Area", "Production", "Yield"],
    ]
    .describe()
    .round(4)
)


Yield distribution by recorded Area band:


,Area_Band,N,Median_Yield,P95_Yield,Max_Yield,N_Yield_Above_10000,Pct_Yield_Above_10000
0,"(0, 0.01]",16796,1557.0,4299.0,44815.0,25,0.149
1,"(0.01, 0.05]",23654,1445.0,4166.1,20143.0,22,0.093
2,"(0.05, 0.1]",11648,1685.0,4579.3,11978.0,8,0.069
3,"(0.1, 0.5]",26351,1994.0,4711.0,12526.0,46,0.175
4,"(0.5, 1]",12186,2414.0,4545.0,10802.0,2,0.016
5,> 1,16632,2600.5,4750.0,8200.0,0,0.000



Area distribution in the primary cohort:


,Area
count,107267.0000
mean,0.4507
std,0.7490
min,0.0100
1%,0.0100
5%,0.0100
10%,0.0100
25%,0.0300
50%,0.1100
75%,0.5700



Yield >= 10,000: area and production summary:


,Area,Production,Yield
count,103.0000,103.0000,103.0000
mean,0.1681,1.8357,13270.5922
std,0.1685,1.8124,7623.3924
min,0.0100,0.0500,10000.0000
25%,0.0200,0.3250,10263.5000
50%,0.0800,0.8000,10763.0000
75%,0.3100,3.5000,11928.5000
max,0.5700,6.2000,44815.0000


In [33]:

# Cell 31 — Verify the yield arithmetic and inspect cross-crop duplicates.

yield_check = primary_df[
    [
        "State", "District", "Crop", "Season", "Year",
        "Yield", "Area", "Production",
    ]
].copy()

for col in ["Yield", "Area", "Production"]:
    yield_check[col] = pd.to_numeric(yield_check[col], errors="coerce")

yield_check = yield_check.loc[
    np.isfinite(yield_check["Yield"])
    & np.isfinite(yield_check["Area"])
    & np.isfinite(yield_check["Production"])
    & (yield_check["Area"] > 0)
].copy()

yield_check["Production_Divided_By_Area"] = (
    yield_check["Production"] / yield_check["Area"]
)

yield_check["Yield_Ratio"] = (
    yield_check["Yield"] / yield_check["Production_Divided_By_Area"].replace(0, np.nan)
)

print("Yield compared with Production / Area:")
display(
    yield_check[
        ["Yield", "Production_Divided_By_Area", "Yield_Ratio"]
    ]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .round(4)
)

print("\nLargest yields and their arithmetic ratios:")
display(
    yield_check.nlargest(20, "Yield")[
        [
            "State", "District", "Crop", "Season", "Year",
            "Yield", "Area", "Production",
            "Production_Divided_By_Area", "Yield_Ratio",
        ]
    ].round(4)
)

# Check whether identical area/production values recur across crop labels
# for the same district, season, and year.
duplicate_keys = [
    "State", "District", "Season", "Year", "Area", "Production"
]

duplicate_measurements = (
    yield_check.groupby(duplicate_keys, dropna=False)
    .agg(
        N_Rows=("Crop", "size"),
        N_Crops=("Crop", "nunique"),
        Crop_Labels=(
            "Crop",
            lambda s: " | ".join(sorted(s.dropna().astype(str).unique())),
        ),
        Yield_Values=(
            "Yield",
            lambda s: " | ".join(
                sorted(s.dropna().astype(str).unique())
            ),
        ),
    )
    .reset_index()
)

duplicate_measurements = duplicate_measurements.loc[
    (duplicate_measurements["N_Rows"] > 1)
    & (duplicate_measurements["N_Crops"] > 1)
].sort_values(["N_Crops", "N_Rows"], ascending=False)

print("\nCross-crop groups with identical location/time/area/production:")
print("Groups:", len(duplicate_measurements))
display(duplicate_measurements.head(25))


Yield compared with Production / Area:


,Yield,Production_Divided_By_Area,Yield_Ratio
count,107267.0000,107267.0000,107267.0000
mean,2163.6252,2.1503,1020.4123
std,1422.8980,1.4389,186.7410
min,15.0000,0.0172,342.0000
1%,265.0000,0.2560,600.0000
5%,500.0000,0.5000,788.5000
25%,1111.0000,1.0000,984.4000
50%,1912.0000,2.0000,1000.0000
75%,2912.0000,2.9598,1018.1818
95%,4561.0000,4.6000,1325.0000



Largest yields and their arithmetic ratios:


,State,District,Crop,Season,Year,Yield,Area,Production,Production_Divided_By_Area,Yield_Ratio
288718,Goa,North Goa,Cereals,Kharif,2022-23,44815.0,0.01,0.47,47.0000,953.5106
288727,Goa,North Goa,Rice,Kharif,2022-23,44815.0,0.01,0.47,47.0000,953.5106
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,44815.0,0.01,0.47,47.0000,953.5106
288736,Goa,South Goa,Cereals,Kharif,2022-23,39221.0,0.01,0.47,47.0000,834.4894
288745,Goa,South Goa,Rice,Kharif,2022-23,39221.0,0.01,0.47,47.0000,834.4894
288750,Goa,South Goa,Total Food Grains,Kharif,2022-23,39221.0,0.01,0.47,47.0000,834.4894
196743,Haryana,Panipat,Other Pulses,Kharif,2019-20,24581.0,0.01,0.16,16.0000,1536.3125
196457,Haryana,Jind,Other Pulses,Kharif,2019-20,23524.0,0.01,0.12,12.0000,1960.3333
165387,Haryana,Panipat,Other Pulses,Kharif,2018-19,20267.0,0.01,0.12,12.0000,1688.9167
196880,Haryana,Sonipat,Other Pulses,Kharif,2019-20,20143.0,0.02,0.35,17.5000,1151.0286



Cross-crop groups with identical location/time/area/production:
Groups: 17623


,State,District,Season,Year,Area,Production,N_Rows,N_Crops,Crop_Labels,Yield_Values
1527,Andhra Pradesh,Nandyal,Summer,2024-25,0.01,0.02,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,2215.0
2991,Andhra Pradesh,Y.S.R. Kadapa,Summer,2024-25,0.02,0.05,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,2215.0
6779,Assam,South Salmara Mancachar,Kharif,2021-22,0.01,0.01,5,5,Cereals | Nutri/Coarse Cereals | Shree Anna /N...,1268.0 | 1282.0 | 1486.0 | 902.0
15710,Gujarat,Amreli,Kharif,2020-21,0.02,0.03,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,1278.0 | 1368.0
15725,Gujarat,Amreli,Kharif,2024-25,0.01,0.02,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,1292.0 | 1900.0 | 1971.0
15776,Gujarat,Amreli,Summer,2016-17,0.01,0.02,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,1716.0 | 2368.0 | 2497.0
15779,Gujarat,Amreli,Summer,2018-19,0.01,0.02,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,2444.0 | 2836.0 | 2912.0
16582,Gujarat,Bhavnagar,Summer,2013-14,0.05,0.14,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,2707.0 | 2793.0 | 2794.0
16642,Gujarat,Botad,Kharif,2020-21,0.02,0.01,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,504.0 | 507.0 | 553.0
16683,Gujarat,Botad,Summer,2020-21,0.01,0.02,5,5,Bajra | Cereals | Nutri/Coarse Cereals | Shree...,1742.0 | 2779.0


In [34]:

# Cell 32 — Test whether Yield follows Production / Area × 1,000.

scaling_check = primary_df[
    ["State", "District", "Crop", "Season", "Year",
     "Yield", "Area", "Production"]
].copy()

for col in ["Yield", "Area", "Production"]:
    scaling_check[col] = pd.to_numeric(
        scaling_check[col], errors="coerce"
    )

scaling_check = scaling_check.loc[
    np.isfinite(scaling_check["Yield"])
    & np.isfinite(scaling_check["Area"])
    & np.isfinite(scaling_check["Production"])
    & (scaling_check["Area"] > 0)
].copy()

scaling_check["Implied_Yield_x1000"] = (
    scaling_check["Production"] / scaling_check["Area"] * 1000
)

scaling_check["Absolute_Difference"] = (
    scaling_check["Yield"] - scaling_check["Implied_Yield_x1000"]
).abs()

scaling_check["Relative_Difference_Pct"] = (
    scaling_check["Absolute_Difference"]
    / scaling_check["Yield"].abs()
    * 100
)

print("Agreement with Production / Area × 1,000:")
display(
    scaling_check[
        ["Yield", "Implied_Yield_x1000",
         "Absolute_Difference", "Relative_Difference_Pct"]
    ]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(3)
)

for tolerance_pct in [1, 5, 10]:
    within = (
        scaling_check["Relative_Difference_Pct"] <= tolerance_pct
    ).mean() * 100
    print(f"Within {tolerance_pct}% relative difference: {within:.2f}%")

print("\nLargest relative discrepancies:")
display(
    scaling_check.sort_values(
        "Relative_Difference_Pct", ascending=False
    )[
        [
            "State", "District", "Crop", "Season", "Year",
            "Yield", "Area", "Production",
            "Implied_Yield_x1000", "Relative_Difference_Pct",
        ]
    ].head(20).round(3)
)


Agreement with Production / Area × 1,000:


,Yield,Implied_Yield_x1000,Absolute_Difference,Relative_Difference_Pct
count,107267.000,107267.000,107267.000,107267.000
mean,2163.625,2150.305,137.721,8.126
std,1422.898,1438.866,291.746,14.827
min,15.000,17.241,0.000,0.000
50%,1912.000,2000.000,29.000,1.680
90%,3898.000,3966.885,385.000,25.429
95%,4561.000,4600.000,660.700,39.024
99%,6974.000,7000.000,1417.000,66.667
max,44815.000,47000.000,11524.000,192.398


Within 1% relative difference: 42.40%
Within 5% relative difference: 66.22%
Within 10% relative difference: 76.66%

Largest relative discrepancies:


,State,District,Crop,Season,Year,Yield,Area,Production,Implied_Yield_x1000,Relative_Difference_Pct
239291,Odisha,Malkangiri,Other Pulses,Rabi,2020-21,342.0,0.01,0.01,1000.0,192.398
288222,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Urad,Kharif,2022-23,347.0,0.01,0.01,1000.0,188.184
37596,Chhattisgarh,Rajnandgaon,Lentil,Rabi,2014-15,352.0,0.01,0.01,1000.0,184.091
324890,Karnataka,Chikkamagaluru,Jowar,Kharif,2023-24,353.0,0.01,0.01,1000.0,183.286
170340,Madhya Pradesh,Bhind,Urad,Kharif,2018-19,353.0,0.01,0.01,1000.0,183.286
103178,Himachal Pradesh,Shimla,Urad,Kharif,2016-17,355.0,0.01,0.01,1000.0,181.690
112974,Odisha,Angul,Urad,Kharif,2016-17,357.0,0.01,0.01,1000.0,180.112
298039,Maharashtra,Akola,Bajra,Kharif,2022-23,359.0,0.01,0.01,1000.0,178.552
84417,Rajasthan,Dungarpur,Tur,Kharif,2015-16,361.0,0.01,0.01,1000.0,177.008
40226,Himachal Pradesh,Chamba,Urad,Kharif,2014-15,361.0,0.01,0.01,1000.0,177.008


In [35]:
# Cell 33 — Locate original APY files and inspect available metadata.

from pathlib import Path

search_roots = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "data" / "processed",
]

print("Potential original APY source files:\n")

source_files = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in {
                ".csv", ".xlsx", ".xls", ".json", ".txt", ".pdf"
            }:
                name = path.name.lower()
                if any(term in name for term in [
                    "apy", "agric", "crop", "yield", "production",
                    "metadata", "dictionary", "readme"
                ]):
                    source_files.append(path)

for path in sorted(set(source_files)):
    print(path.relative_to(PROJECT_ROOT))

print("\nRelevant files already identified:")
print("Integrated dataset:", integrated_path)
print(
    "Historical APY dataset:",
    PROJECT_ROOT / "data" / "processed"
    / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
)

Potential original APY source files:

data\processed\.ipynb_checkpoints\historical_food_grains_apy_2013-14_to_2024-25-checkpoint.csv
data\processed\historical_food_grains_apy_2013-14_to_2024-25.csv
data\processed\historical_food_grains_apy_cleaned_flagged.csv
data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv
data\processed\weather_regional\apy_districts_without_weather_coverage.csv
data\processed\weather_regional\apy_weather_integrated_exploratory.csv
data\processed\weather_regional\apy_weather_integration_summary.csv
data\processed\weather_regional\apy_weather_proxy_district_year_coverage.csv
data\processed\weather_regional\crop_weather_integration_validation_summary.csv
data\processed\weather_regional\yield_area_production_ratio_review.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0005_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0006_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0055_weather_2013_2024.csv
data\raw\weather_pilot_2

In [36]:
# Cell 34 — Inspect APY schemas and existing yield-ratio audit.

from pathlib import Path
import pandas as pd

data_dir = PROJECT_ROOT / "data" / "processed"

files_to_inspect = {
    "Historical APY": data_dir / "historical_food_grains_apy_2013-14_to_2024-25.csv",
    "Cleaned and flagged APY": data_dir / "historical_food_grains_apy_cleaned_flagged.csv",
    "Long-format APY": data_dir / "historical_food_grains_apy_long_2013-14_to_2024-25.csv",
    "Yield ratio review": (
        data_dir / "weather_regional" / "yield_area_production_ratio_review.csv"
    ),
}

for label, path in files_to_inspect.items():
    print(f"\n{'=' * 75}\n{label}\nFile: {path.name}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        sample = pd.read_csv(path, nrows=5, low_memory=False)
        print("Columns:", sample.columns.tolist())
        display(sample.head(3))
    except Exception as exc:
        print("Could not read as CSV:", repr(exc))


Historical APY
File: historical_food_grains_apy_2013-14_to_2024-25.csv
Columns: ['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Production-2013-14', 'Yield-2013-14', 'Area-2014-15', 'Production-2014-15', 'Yield-2014-15', 'Area-2015-16', 'Production-2015-16', 'Yield-2015-16', 'Area-2016-17', 'Production-2016-17', 'Yield-2016-17', 'Area-2017-18', 'Production-2017-18', 'Yield-2017-18', 'Area-2018-19', 'Production-2018-19', 'Yield-2018-19', 'Area-2019-20', 'Production-2019-20', 'Yield-2019-20', 'Area-2020-21', 'Production-2020-21', 'Yield-2020-21', 'Area-2021-22', 'Production-2021-22', 'Yield-2021-22', 'Area-2022-23', 'Production-2022-23', 'Yield-2022-23', 'Area-2023-24', 'Production-2023-24', 'Yield-2023-24', 'Area-2024-25', 'Production-2024-25', 'Yield-2024-25']


,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,NaN,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,1133.0,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0



Cleaned and flagged APY
File: historical_food_grains_apy_cleaned_flagged.csv
Columns: ['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Production-2013-14', 'Yield-2013-14', 'Area-2014-15', 'Production-2014-15', 'Yield-2014-15', 'Area-2015-16', 'Production-2015-16', 'Yield-2015-16', 'Area-2016-17', 'Production-2016-17', 'Yield-2016-17', 'Area-2017-18', 'Production-2017-18', 'Yield-2017-18', 'Area-2018-19', 'Production-2018-19', 'Yield-2018-19', 'Area-2019-20', 'Production-2019-20', 'Yield-2019-20', 'Area-2020-21', 'Production-2020-21', 'Yield-2020-21', 'Area-2021-22', 'Production-2021-22', 'Yield-2021-22', 'Area-2022-23', 'Production-2022-23', 'Yield-2022-23', 'Area-2023-24', 'Production-2023-24', 'Yield-2023-24', 'Area-2024-25', 'Production-2024-25', 'Yield-2024-25', 'Crop_Category', 'QC-2013-14-AreaZero-ProductionPositive', 'QC-2013-14-AreaPositive-ProductionZero', 'QC-2013-14-NegativeMeasurements', 'QC-2013-14-PositiveYieldWithoutPositiveAreaProduction', 'QC-2014-15-AreaZero

,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,QC-2022-23-NegativeMeasurements,QC-2022-23-PositiveYieldWithoutPositiveAreaProduction,QC-2023-24-AreaZero-ProductionPositive,QC-2023-24-AreaPositive-ProductionZero,QC-2023-24-NegativeMeasurements,QC-2023-24-PositiveYieldWithoutPositiveAreaProduction,QC-2024-25-AreaZero-ProductionPositive,QC-2024-25-AreaPositive-ProductionZero,QC-2024-25-NegativeMeasurements,QC-2024-25-PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,False,True,False,False,False,True,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,False,False,False,False,False,False,False,False,False,False
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,False,True,False,False,False,True,False,False,False,True



Long-format APY
File: historical_food_grains_apy_long_2013-14_to_2024-25.csv
Columns: ['State', 'District', 'Crop', 'Season', 'Crop_Category', 'Year', 'Area', 'Production', 'Yield', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_NegativeMeasurements', 'QC_PositiveYieldWithoutPositiveAreaProduction']


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True



Yield ratio review
File: yield_area_production_ratio_review.csv
Columns: ['Crop', 'Season', 'Year', 'Area', 'Production', 'Yield', 'production_per_area_raw', 'yield_to_raw_ratio', 'ratio_relative_deviation_from_1000', 'FE_RatioReviewFlag', 'FE_RatioReviewReason']


,Crop,Season,Year,Area,Production,Yield,production_per_area_raw,yield_to_raw_ratio,ratio_relative_deviation_from_1000,FE_RatioReviewFlag,FE_RatioReviewReason
0,Cereals,Kharif,2013-14,0.08,0.15,2010.0,1.875,1072.0,0.0720,False,Within 10% of provisional 1000 scale
1,Cereals,Total,2013-14,0.08,0.16,2013.0,2.000,1006.5,0.0065,False,Within 10% of provisional 1000 scale
2,Rice,Kharif,2013-14,0.08,0.15,2010.0,1.875,1072.0,0.0720,False,Within 10% of provisional 1000 scale


In [37]:
# Cell 35 — Summarize the existing yield-ratio review and its flags.

ratio_review_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_regional"
    / "yield_area_production_ratio_review.csv"
)

ratio_review = pd.read_csv(ratio_review_path, low_memory=False)

print("Audit rows:", len(ratio_review))
print("\nReview flag counts:")
display(
    ratio_review["FE_RatioReviewFlag"]
    .value_counts(dropna=False)
    .rename_axis("Flag")
    .to_frame("Count")
)

print("\nReview reasons:")
display(
    ratio_review["FE_RatioReviewReason"]
    .value_counts(dropna=False)
    .rename_axis("Reason")
    .to_frame("Count")
)

print("\nDeviation from provisional 1,000 scaling factor:")
display(
    ratio_review["ratio_relative_deviation_from_1000"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .to_frame("Relative_Deviation")
    .round(4)
)

print("\nLargest flagged discrepancies:")
display(
    ratio_review.loc[ratio_review["FE_RatioReviewFlag"]]
    .sort_values(
        "ratio_relative_deviation_from_1000",
        ascending=False
    )[
        [
            "Crop", "Season", "Year", "Area", "Production", "Yield",
            "production_per_area_raw", "yield_to_raw_ratio",
            "ratio_relative_deviation_from_1000",
            "FE_RatioReviewReason"
        ]
    ]
    .head(15)
    .round(4)
)

Audit rows: 186587

Review flag counts:


,Count
Flag,
False,144879
True,41708



Review reasons:


,Count
Reason,
Within 10% of provisional 1000 scale,144879
Ratio differs by more than 10% from provisional 1000 scale,41708



Deviation from provisional 1,000 scaling factor:


,Relative_Deviation
count,186587.0000
mean,0.0811
std,0.1596
min,0.0000
50%,0.0149
90%,0.2470
95%,0.3817
99%,0.7760
max,1.9350



Largest flagged discrepancies:


,Crop,Season,Year,Area,Production,Yield,production_per_area_raw,yield_to_raw_ratio,ratio_relative_deviation_from_1000,FE_RatioReviewReason
137928,Maize,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0,1.935,Ratio differs by more than 10% from provisiona...
137931,Nutri/Coarse Cereals,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0,1.935,Ratio differs by more than 10% from provisiona...
155753,Maize,Kharif,2023-24,0.01,0.01,2911.0,1.0,2911.0,1.911,Ratio differs by more than 10% from provisiona...
80970,Nutri/Coarse Cereals,Rabi,2018-19,0.01,0.01,2909.0,1.0,2909.0,1.909,Ratio differs by more than 10% from provisiona...
80962,Cereals,Rabi,2018-19,0.01,0.01,2905.0,1.0,2905.0,1.905,Ratio differs by more than 10% from provisiona...
120322,Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,Ratio differs by more than 10% from provisiona...
120331,Nutri/Coarse Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,Ratio differs by more than 10% from provisiona...
120325,Maize,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,Ratio differs by more than 10% from provisiona...
155515,Cereals,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0,1.883,Ratio differs by more than 10% from provisiona...
155518,Rice,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0,1.883,Ratio differs by more than 10% from provisiona...


In [38]:
# Cell 36 — Quantify provisional yield-ratio flags in the primary cohort.

ratio_cohort = primary_df[
    ["State", "District", "Crop", "Season", "Year",
     "Yield", "Area", "Production"]
].copy()

for col in ["Yield", "Area", "Production"]:
    ratio_cohort[col] = pd.to_numeric(
        ratio_cohort[col], errors="coerce"
    )

ratio_cohort = ratio_cohort.loc[
    np.isfinite(ratio_cohort["Yield"])
    & np.isfinite(ratio_cohort["Area"])
    & np.isfinite(ratio_cohort["Production"])
    & (ratio_cohort["Yield"] > 0)
    & (ratio_cohort["Area"] > 0)
    & (ratio_cohort["Production"] > 0)
].copy()

ratio_cohort["Implied_Yield_x1000"] = (
    ratio_cohort["Production"] / ratio_cohort["Area"] * 1000
)

ratio_cohort["Ratio_Deviation"] = (
    ratio_cohort["Yield"] / ratio_cohort["Implied_Yield_x1000"] - 1
).abs()

ratio_cohort["Provisional_Ratio_Flag"] = (
    ratio_cohort["Ratio_Deviation"] > 0.10
)

print("Primary cohort rows audited:", len(ratio_cohort))
print("\nProvisional flag counts:")
display(
    ratio_cohort["Provisional_Ratio_Flag"]
    .value_counts()
    .rename_axis("Flagged")
    .to_frame("Rows")
    .assign(
        Percentage=lambda x: (
            x["Rows"] / len(ratio_cohort) * 100
        ).round(2)
    )
)

print("\nComparison by provisional flag:")
display(
    ratio_cohort.groupby("Provisional_Ratio_Flag")
    .agg(
        Rows=("Yield", "size"),
        Mean_Yield=("Yield", "mean"),
        Median_Yield=("Yield", "median"),
        P95_Yield=("Yield", lambda x: x.quantile(0.95)),
        Max_Yield=("Yield", "max"),
        Mean_Area=("Area", "mean"),
        Median_Area=("Area", "median"),
        Mean_Ratio_Deviation=("Ratio_Deviation", "mean"),
    )
    .round(3)
)

print("\nFlag rates by yield threshold:")
for threshold in [5000, 10000]:
    subset = ratio_cohort.loc[ratio_cohort["Yield"] >= threshold]
    if len(subset):
        print(
            f"Yield >= {threshold}: {len(subset):,} rows; "
            f"{subset['Provisional_Ratio_Flag'].mean() * 100:.2f}% flagged"
        )

print("\nFlag rates by area band:")
ratio_cohort["Area_Band"] = pd.cut(
    ratio_cohort["Area"],
    bins=[0, 0.01, 0.05, 0.1, 0.5, 1, np.inf],
    labels=["(0, 0.01]", "(0.01, 0.05]", "(0.05, 0.1]",
            "(0.1, 0.5]", "(0.5, 1]", ">1"],
    include_lowest=True
)

display(
    ratio_cohort.groupby("Area_Band", observed=False)
    .agg(
        Rows=("Yield", "size"),
        Flagged_Rows=("Provisional_Ratio_Flag", "sum"),
        Median_Yield=("Yield", "median"),
        Max_Yield=("Yield", "max"),
    )
    .assign(
        Flag_Rate_Pct=lambda x: (
            x["Flagged_Rows"] / x["Rows"] * 100
        ).round(2)
    )
    .round(3)
)

Primary cohort rows audited: 107267

Provisional flag counts:


,Rows,Percentage
Flagged,,
False,82274,76.7
True,24993,23.3



Comparison by provisional flag:


,Rows,Mean_Yield,Median_Yield,P95_Yield,Max_Yield,Mean_Area,Median_Area,Mean_Ratio_Deviation
Provisional_Ratio_Flag,,,,,,,,
False,82274,2291.821,2076.5,4638.35,44815.0,0.581,0.24,0.019
True,24993,1741.619,1363.0,4119.00,39221.0,0.021,0.01,0.303



Flag rates by yield threshold:
Yield >= 5000: 3,536 rows; 19.68% flagged
Yield >= 10000: 103 rows; 27.18% flagged

Flag rates by area band:


,Rows,Flagged_Rows,Median_Yield,Max_Yield,Flag_Rate_Pct
Area_Band,,,,,
"(0, 0.01]",16796,12948,1557.0,44815.0,77.09
"(0.01, 0.05]",23654,11063,1445.0,20143.0,46.77
"(0.05, 0.1]",11648,837,1685.0,11978.0,7.19
"(0.1, 0.5]",26351,142,1994.0,12526.0,0.54
"(0.5, 1]",12186,3,2414.0,10802.0,0.02
>1,16632,0,2600.5,8200.0,0.00


In [39]:
# Cell 37 — Compare model errors for flagged vs unflagged holdout records.

# Preserve original row indices so flags align with the modeling data.
ratio_flag_by_index = ratio_cohort["Provisional_Ratio_Flag"]

error_flag_check = primary_holdout_weather[
    ["State", "District", "Crop", "Season", "Year", "Yield"]
].copy()

error_flag_check["Actual_Yield"] = y_holdout_weather.to_numpy()
error_flag_check["Predicted_Yield"] = np.asarray(
    weather_hgb_predictions, dtype=float
)

error_flag_check["Signed_Error"] = (
    error_flag_check["Predicted_Yield"]
    - error_flag_check["Actual_Yield"]
)

error_flag_check["Absolute_Error"] = (
    error_flag_check["Signed_Error"].abs()
)

error_flag_check["Provisional_Ratio_Flag"] = (
    error_flag_check.index.map(ratio_flag_by_index)
)

assert error_flag_check["Provisional_Ratio_Flag"].notna().all(), (
    "Some holdout rows could not be matched to the ratio audit."
)

error_flag_check["Provisional_Ratio_Flag"] = (
    error_flag_check["Provisional_Ratio_Flag"].astype(bool)
)

print("Weather-complete holdout rows:", len(error_flag_check))
print("\nFlag distribution:")
display(
    error_flag_check["Provisional_Ratio_Flag"]
    .value_counts()
    .rename_axis("Provisional_Ratio_Flag")
    .to_frame("Rows")
    .assign(
        Percentage=lambda x: (
            x["Rows"] / len(error_flag_check) * 100
        ).round(2)
    )
)

print("\nPrediction errors by provisional flag:")
display(
    error_flag_check.groupby("Provisional_Ratio_Flag")
    .agg(
        Rows=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        Median_Actual=("Actual_Yield", "median"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(x**2))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
    )
    .round(3)
)

print("\nShare of total absolute error:")
total_abs_error = error_flag_check["Absolute_Error"].sum()

display(
    error_flag_check.groupby("Provisional_Ratio_Flag")
    .agg(
        Rows=("Absolute_Error", "size"),
        Total_Absolute_Error=("Absolute_Error", "sum"),
    )
    .assign(
        Row_Share_Pct=lambda x: (
            x["Rows"] / len(error_flag_check) * 100
        ).round(2),
        Error_Share_Pct=lambda x: (
            x["Total_Absolute_Error"] / total_abs_error * 100
        ).round(2),
    )
    .round(3)
)

print("\nFlagged records among the 20 largest absolute errors:")
display(
    error_flag_check.nlargest(20, "Absolute_Error")[
        [
            "State", "District", "Crop", "Season", "Year",
            "Actual_Yield", "Predicted_Yield", "Absolute_Error",
            "Provisional_Ratio_Flag",
        ]
    ].round(2)
)

Weather-complete holdout rows: 16230

Flag distribution:


,Rows,Percentage
Provisional_Ratio_Flag,,
False,12449,76.7
True,3781,23.3



Prediction errors by provisional flag:


,Rows,Mean_Actual,Median_Actual,MAE,RMSE,Mean_Signed_Error,Max_Absolute_Error
Provisional_Ratio_Flag,,,,,,,
False,12449,2480.657,2307.0,729.675,1103.883,90.870,8955.062
True,3781,1852.052,1426.0,698.560,1055.541,267.021,9572.421



Share of total absolute error:


,Rows,Total_Absolute_Error,Row_Share_Pct,Error_Share_Pct
Provisional_Ratio_Flag,,,,
False,12449,9083723.783,76.7,77.47
True,3781,2641254.267,23.3,22.53



Flagged records among the 20 largest absolute errors:


,State,District,Crop,Season,Year,Actual_Yield,Predicted_Yield,Absolute_Error,Provisional_Ratio_Flag
314552,Andhra Pradesh,Tirupati,Nutri/Coarse Cereals,Kharif,2023-24,3590.0,13162.42,9572.42,True
345743,Andhra Pradesh,Prakasam,Total Food Grains,Kharif,2024-25,1094.0,10049.06,8955.06,False
314540,Andhra Pradesh,Tirupati,Cereals,Kharif,2023-24,3745.0,12603.31,8858.31,False
314561,Andhra Pradesh,Tirupati,Rice,Kharif,2023-24,3754.0,12523.65,8769.65,False
314569,Andhra Pradesh,Tirupati,Total Food Grains,Kharif,2023-24,3740.0,12206.82,8466.82,False
345717,Andhra Pradesh,Prakasam,Maize,Kharif,2024-25,4127.0,12484.84,8357.84,False
345723,Andhra Pradesh,Prakasam,Nutri/Coarse Cereals,Kharif,2024-25,2675.0,11021.89,8346.89,False
314361,Andhra Pradesh,Prakasam,Maize,Kharif,2023-24,3460.0,11787.28,8327.28,False
313895,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,Cereals,Kharif,2023-24,3518.0,11769.24,8251.24,False
314387,Andhra Pradesh,Prakasam,Total Food Grains,Kharif,2023-24,1022.0,9258.50,8236.50,False


In [40]:
# Cell 38 — Diagnose weather-feature distribution in the worst 1% errors.

error_weather_check = primary_holdout_weather[
    ["State", "District", "Crop", "Season", "Year", "Yield"]
    + weather_features
].copy()

error_weather_check["Actual_Yield"] = y_holdout_weather.to_numpy()
error_weather_check["Predicted_Yield"] = np.asarray(
    weather_hgb_predictions, dtype=float
)
error_weather_check["Absolute_Error"] = (
    error_weather_check["Predicted_Yield"]
    - error_weather_check["Actual_Yield"]
).abs()

worst_cutoff = error_weather_check["Absolute_Error"].quantile(0.99)
worst_weather_rows = error_weather_check.loc[
    error_weather_check["Absolute_Error"] >= worst_cutoff
].copy()

print(f"99th-percentile absolute-error cutoff: {worst_cutoff:.2f}")
print("Worst-error rows:", len(worst_weather_rows))

print("\nWeather feature distributions:")
weather_distribution_rows = []

for feature in weather_features:
    train_values = pd.to_numeric(
        primary_train_weather[feature], errors="coerce"
    ).dropna()

    holdout_values = pd.to_numeric(
        primary_holdout_weather[feature], errors="coerce"
    ).dropna()

    worst_values = pd.to_numeric(
        worst_weather_rows[feature], errors="coerce"
    ).dropna()

    weather_distribution_rows.append({
        "Feature": feature,
        "Train_Median": train_values.median(),
        "Train_P01": train_values.quantile(0.01),
        "Train_P99": train_values.quantile(0.99),
        "Holdout_Median": holdout_values.median(),
        "Worst_Error_Median": worst_values.median(),
        "Worst_Error_Min": worst_values.min(),
        "Worst_Error_Max": worst_values.max(),
        "Worst_Outside_Train_P01_P99_Pct": (
            (
                (worst_values < train_values.quantile(0.01))
                | (worst_values > train_values.quantile(0.99))
            ).mean() * 100
        ),
    })

display(pd.DataFrame(weather_distribution_rows).round(3))

print("\nWorst-error rows by state and year:")
display(
    worst_weather_rows.groupby(["State", "Year"])
    .agg(
        Rows=("Absolute_Error", "size"),
        Mean_Absolute_Error=("Absolute_Error", "mean"),
        Mean_Signed_Error=(
            "Predicted_Yield",
            lambda pred: (
                pred - worst_weather_rows.loc[pred.index, "Actual_Yield"]
            ).mean()
        ),
    )
    .sort_values("Mean_Absolute_Error", ascending=False)
    .round(2)
    .head(15)
)

99th-percentile absolute-error cutoff: 4126.13
Worst-error rows: 163

Weather feature distributions:


,Feature,Train_Median,Train_P01,Train_P99,Holdout_Median,Worst_Error_Median,Worst_Error_Min,Worst_Error_Max,Worst_Outside_Train_P01_P99_Pct
0,tmin_mean_c,19.740,1.521,25.067,20.048,24.964,17.830,27.557,43.558
1,tmin_min_c,5.150,-15.430,19.350,5.620,18.170,2.340,24.110,26.994
2,tmax_mean_c,32.151,11.968,35.249,32.175,31.834,28.819,33.929,0.000
3,tmax_max_c,44.470,21.290,48.770,44.820,41.750,34.110,47.400,0.000
4,precipitation_total_mm,1074.730,369.950,3485.610,1090.520,1350.210,765.620,4478.310,6.748
5,precipitation_mean_mm_day,2.942,1.011,9.524,2.981,3.689,2.098,12.236,6.748
6,precipitation_max_mm_day,63.710,24.550,192.420,67.320,105.210,31.830,240.940,9.202
7,tmin_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000
8,tmax_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000
9,precipitation_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000



Worst-error rows by state and year:


Rows  Mean_Absolute_Error  Mean_Signed_Error
State          Year                                                 
West Bengal    2024-25     2              7146.30           -7146.30
Andhra Pradesh 2023-24    26              6891.60            6144.48
Rajasthan      2024-25     1              6332.96           -6332.96
               2023-24     1              6332.59           -6332.59
Goa            2023-24    13              6147.98            6147.98
West Bengal    2023-24     5              6145.22           -6145.22
Goa            2024-25     9              6123.22            6123.22
Assam          2024-25     3              6058.11           -6058.11
Andhra Pradesh 2024-25    39              5834.09            4578.14
Tamil Nadu     2023-24    12              5153.06            1924.89
Bihar          2023-24    12              4833.18           -4833.18
Tamil Nadu     2024-25     8              4794.57            3544.69
Madhya Pradesh 2023-24     3              4720.57           -4720.57
Karnataka      2024-25     4              4690.37            4690.37
Bihar          2024-25    11              4683.26           -4683.26

In [41]:
# Cell 39 — Inspect weather distributions and worst individual predictions.

print("1. Weather distribution: training vs holdout vs worst-error rows")

weather_distribution_df = pd.DataFrame(weather_distribution_rows)

display(
    weather_distribution_df.round(3)
)

print("\n2. Fifteen worst individual predictions")

worst_record_columns = [
    "State", "District", "Crop", "Season", "Year",
    "Actual_Yield", "Predicted_Yield", "Absolute_Error"
] + weather_features

display(
    error_weather_check.nlargest(15, "Absolute_Error")[
        worst_record_columns
    ].round(3)
)

print("\n3. Worst-error weather values relative to training ranges")

weather_range_rows = []

for feature in weather_features:
    train_values = pd.to_numeric(
        primary_train_weather[feature], errors="coerce"
    ).dropna()

    worst_values = pd.to_numeric(
        worst_weather_rows[feature], errors="coerce"
    ).dropna()

    lower = train_values.quantile(0.01)
    upper = train_values.quantile(0.99)

    weather_range_rows.append({
        "Feature": feature,
        "Train_P01": lower,
        "Train_Median": train_values.median(),
        "Train_P99": upper,
        "Worst_Rows_Below_P01_Pct": (
            (worst_values < lower).mean() * 100
        ),
        "Worst_Rows_Above_P99_Pct": (
            (worst_values > upper).mean() * 100
        ),
    })

display(pd.DataFrame(weather_range_rows).round(3))

1. Weather distribution: training vs holdout vs worst-error rows


,Feature,Train_Median,Train_P01,Train_P99,Holdout_Median,Worst_Error_Median,Worst_Error_Min,Worst_Error_Max,Worst_Outside_Train_P01_P99_Pct
0,tmin_mean_c,19.740,1.521,25.067,20.048,24.964,17.830,27.557,43.558
1,tmin_min_c,5.150,-15.430,19.350,5.620,18.170,2.340,24.110,26.994
2,tmax_mean_c,32.151,11.968,35.249,32.175,31.834,28.819,33.929,0.000
3,tmax_max_c,44.470,21.290,48.770,44.820,41.750,34.110,47.400,0.000
4,precipitation_total_mm,1074.730,369.950,3485.610,1090.520,1350.210,765.620,4478.310,6.748
5,precipitation_mean_mm_day,2.942,1.011,9.524,2.981,3.689,2.098,12.236,6.748
6,precipitation_max_mm_day,63.710,24.550,192.420,67.320,105.210,31.830,240.940,9.202
7,tmin_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000
8,tmax_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000
9,precipitation_valid_days,365.000,365.000,366.000,365.000,366.000,365.000,366.000,0.000



2. Fifteen worst individual predictions


,State,District,Crop,Season,Year,Actual_Yield,Predicted_Yield,Absolute_Error,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days
314552,Andhra Pradesh,Tirupati,Nutri/Coarse Cereals,Kharif,2023-24,3590.0,13162.421,9572.421,25.174,19.07,32.671,41.77,1325.24,3.631,240.94,365.0,365.0,365.0
345743,Andhra Pradesh,Prakasam,Total Food Grains,Kharif,2024-25,1094.0,10049.062,8955.062,25.226,18.58,32.888,43.65,1077.51,2.944,93.82,366.0,366.0,366.0
314540,Andhra Pradesh,Tirupati,Cereals,Kharif,2023-24,3745.0,12603.309,8858.309,25.174,19.07,32.671,41.77,1325.24,3.631,240.94,365.0,365.0,365.0
314561,Andhra Pradesh,Tirupati,Rice,Kharif,2023-24,3754.0,12523.651,8769.651,25.174,19.07,32.671,41.77,1325.24,3.631,240.94,365.0,365.0,365.0
314569,Andhra Pradesh,Tirupati,Total Food Grains,Kharif,2023-24,3740.0,12206.824,8466.824,25.174,19.07,32.671,41.77,1325.24,3.631,240.94,365.0,365.0,365.0
345717,Andhra Pradesh,Prakasam,Maize,Kharif,2024-25,4127.0,12484.837,8357.837,25.226,18.58,32.888,43.65,1077.51,2.944,93.82,366.0,366.0,366.0
345723,Andhra Pradesh,Prakasam,Nutri/Coarse Cereals,Kharif,2024-25,2675.0,11021.892,8346.892,25.226,18.58,32.888,43.65,1077.51,2.944,93.82,366.0,366.0,366.0
314361,Andhra Pradesh,Prakasam,Maize,Kharif,2023-24,3460.0,11787.284,8327.284,24.963,18.17,33.289,41.75,922.27,2.527,105.91,365.0,365.0,365.0
313895,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,Cereals,Kharif,2023-24,3518.0,11769.240,8251.240,25.212,18.87,31.686,41.52,1169.41,3.204,105.21,365.0,365.0,365.0
314387,Andhra Pradesh,Prakasam,Total Food Grains,Kharif,2023-24,1022.0,9258.497,8236.497,24.963,18.17,33.289,41.75,922.27,2.527,105.91,365.0,365.0,365.0



3. Worst-error weather values relative to training ranges


,Feature,Train_P01,Train_Median,Train_P99,Worst_Rows_Below_P01_Pct,Worst_Rows_Above_P99_Pct
0,tmin_mean_c,1.521,19.740,25.067,0.0,43.558
1,tmin_min_c,-15.430,5.150,19.350,0.0,26.994
2,tmax_mean_c,11.968,32.151,35.249,0.0,0.000
3,tmax_max_c,21.290,44.470,48.770,0.0,0.000
4,precipitation_total_mm,369.950,1074.730,3485.610,0.0,6.748
5,precipitation_mean_mm_day,1.011,2.942,9.524,0.0,6.748
6,precipitation_max_mm_day,24.550,63.710,192.420,0.0,9.202
7,tmin_valid_days,365.000,365.000,366.000,0.0,0.000
8,tmax_valid_days,365.000,365.000,366.000,0.0,0.000
9,precipitation_valid_days,365.000,365.000,366.000,0.0,0.000


In [42]:
# Cell 40 — Trace unusually high holdout predictions to training distributions.

high_prediction_threshold = 8000

high_prediction_rows = error_weather_check.loc[
    error_weather_check["Predicted_Yield"] >= high_prediction_threshold
].copy()

print(f"Holdout predictions >= {high_prediction_threshold}: "
      f"{len(high_prediction_rows):,}")
print(f"Share of weather-complete holdout: "
      f"{len(high_prediction_rows) / len(error_weather_check) * 100:.2f}%")

print("\n1. High predictions by year and crop")
if len(high_prediction_rows):
    display(
        high_prediction_rows.groupby(["Year", "Crop"])
        .agg(
            Rows=("Yield", "size"),
            Mean_Actual=("Yield", "mean"),
            Mean_Predicted=("Predicted_Yield", "mean"),
            Max_Predicted=("Predicted_Yield", "max"),
            Mean_Absolute_Error=("Absolute_Error", "mean"),
        )
        .sort_values("Mean_Predicted", ascending=False)
        .round(2)
        .head(25)
    )

print("\n2. Training target distribution for the same crop-season combinations")

train_target_summary = (
    primary_train_weather.groupby(["Crop", "Season"])["Yield"]
    .agg(
        Train_Rows="size",
        Train_Median="median",
        Train_P95=lambda x: x.quantile(0.95),
        Train_P99=lambda x: x.quantile(0.99),
        Train_Max="max",
    )
    .reset_index()
)

if len(high_prediction_rows):
    high_prediction_combinations = (
        high_prediction_rows[["Crop", "Season"]]
        .drop_duplicates()
    )

    high_prediction_training_context = high_prediction_combinations.merge(
        train_target_summary,
        on=["Crop", "Season"],
        how="left",
        validate="one_to_one",
    )

    display(
        high_prediction_training_context
        .sort_values("Train_Max", ascending=False)
        .round(2)
    )

print("\n3. Highest-yield training records")
display(
    primary_train_weather.nlargest(20, "Yield")[
        [
            "State", "District", "Crop", "Season", "Year",
            "Yield", "Area", "Production"
        ]
    ].round(3)
)

Holdout predictions >= 8000: 70
Share of weather-complete holdout: 0.43%

1. High predictions by year and crop


Rows  Mean_Actual  Mean_Predicted  \
Year    Crop                                                           
2023-24 Nutri/Coarse Cereals          2      3100.50        11601.84   
        Cereals                       7      3324.00        11026.03   
        Rice                          7      3378.00        10899.71   
        Total Food Grains             7      3046.57        10603.49   
2024-25 Maize                         5      5453.20        10529.58   
        Nutri/Coarse Cereals          4      3827.25        10084.63   
2023-24 Maize                         3      5521.00         9888.84   
        Bajra                         2      2866.00         9465.97   
        Shree Anna /Nutri Cereals     2      2812.00         9395.58   
2024-25 Rice                         10      3261.80         9256.66   
        Cereals                      10      3234.10         9211.64   
        Total Food Grains             9      2966.00         9058.54   
        Bajra                         1      2718.00         9007.99   
        Shree Anna /Nutri Cereals     1      2103.00         8504.84   

                                   Max_Predicted  Mean_Absolute_Error  
Year    Crop                                                           
2023-24 Nutri/Coarse Cereals            13162.42              8501.34  
        Cereals                         12603.31              7702.03  
        Rice                            12523.65              7521.71  
        Total Food Grains               12206.82              7556.92  
2024-25 Maize                           12484.84              5076.38  
        Nutri/Coarse Cereals            11021.89              6257.38  
2023-24 Maize                           11787.28              4535.06  
        Bajra                           10723.52              6599.97  
        Shree Anna /Nutri Cereals       10563.76              6583.58  
2024-25 Rice                            10269.78              5994.86  
        Cereals                         10550.89              5977.54  
        Total Food Grains               10049.06              6092.54  
        Bajra                            9007.99              6289.99  
        Shree Anna /Nutri Cereals        8504.84              6401.84


2. Training target distribution for the same crop-season combinations


,Crop,Season,Train_Rows,Train_Median,Train_P95,Train_P99,Train_Max
0,Cereals,Kharif,5543,2144.0,4125.50,5024.32,44815.0
2,Rice,Kharif,4741,2314.0,4214.00,4934.00,44815.0
3,Total Food Grains,Kharif,5636,1935.5,3933.50,4798.25,44815.0
1,Maize,Rabi,1335,4009.0,9033.20,10375.14,12526.0
5,Maize,Kharif,3790,2183.0,6190.35,9069.07,11978.0
6,Nutri/Coarse Cereals,Kharif,4740,1781.5,4447.20,6721.90,11338.0
7,Shree Anna /Nutri Cereals,Kharif,2941,1161.0,2522.00,3289.60,4953.0
4,Bajra,Kharif,1472,1503.5,2826.45,3555.77,4782.0



3. Highest-yield training records


,State,District,Crop,Season,Year,Yield,Area,Production
288718,Goa,North Goa,Cereals,Kharif,2022-23,44815.0,0.01,0.47
288727,Goa,North Goa,Rice,Kharif,2022-23,44815.0,0.01,0.47
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,44815.0,0.01,0.47
288736,Goa,South Goa,Cereals,Kharif,2022-23,39221.0,0.01,0.47
288745,Goa,South Goa,Rice,Kharif,2022-23,39221.0,0.01,0.47
288750,Goa,South Goa,Total Food Grains,Kharif,2022-23,39221.0,0.01,0.47
196743,Haryana,Panipat,Other Pulses,Kharif,2019-20,24581.0,0.01,0.16
196457,Haryana,Jind,Other Pulses,Kharif,2019-20,23524.0,0.01,0.12
165387,Haryana,Panipat,Other Pulses,Kharif,2018-19,20267.0,0.01,0.12
196880,Haryana,Sonipat,Other Pulses,Kharif,2019-20,20143.0,0.02,0.35


In [43]:
# Cell 41 — Sensitivity experiment for extreme training targets.
# Diagnostic only: this is NOT a decision to remove these records permanently.

from sklearn.base import clone
from sklearn.pipeline import make_pipeline

extreme_target_threshold = 10000

# Keep only training rows below the experimental threshold.
non_extreme_mask = y_train_weather < extreme_target_threshold

X_train_non_extreme = X_train_core_plus_weather.loc[non_extreme_mask]
y_train_non_extreme = y_train_weather.loc[non_extreme_mask]

excluded_count = int((~non_extreme_mask).sum())

print("Original weather-complete training rows:", len(y_train_weather))
print("Rows excluded in this experiment:", excluded_count)
print("Experimental training rows:", len(y_train_non_extreme))
print(
    "Percentage of training rows excluded:",
    round(excluded_count / len(y_train_weather) * 100, 3),
)

# Rebuild the same feature preprocessing and model configuration.
sensitivity_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", clone(core_encoder), core_categorical_features),
        (
            "numeric",
            "passthrough",
            core_numeric_features + weather_features,
        ),
    ],
    remainder="drop",
)

extreme_sensitive_model = make_pipeline(
    sensitivity_preprocessor,
    HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    ),
)

extreme_sensitive_model.fit(
    X_train_non_extreme,
    y_train_non_extreme,
)

# Predict on the unchanged, identical holdout set.
predictions_without_extremes = extreme_sensitive_model.predict(
    X_holdout_core_plus_weather
)

original_metrics = evaluate_regression(
    y_holdout_weather,
    weather_hgb_predictions,
)

sensitivity_metrics = evaluate_regression(
    y_holdout_weather,
    predictions_without_extremes,
)

sensitivity_comparison = pd.DataFrame([
    {
        "Model": "Original weather HGB",
        **original_metrics,
    },
    {
        "Model": "HGB trained without Yield >= 10000",
        **sensitivity_metrics,
    },
])

display(sensitivity_comparison.round(4))

# Compare errors specifically on holdout records with extreme actual yields.
extreme_actual_mask = y_holdout_weather.to_numpy() >= extreme_target_threshold

print("\nHoldout rows with actual Yield >= 10000:", int(extreme_actual_mask.sum()))

if extreme_actual_mask.any():
    print("\nMetrics on extreme-actual-yield holdout rows:")
    display(pd.DataFrame([
        {
            "Model": "Original weather HGB",
            **evaluate_regression(
                y_holdout_weather.to_numpy()[extreme_actual_mask],
                np.asarray(weather_hgb_predictions)[extreme_actual_mask],
            ),
        },
        {
            "Model": "HGB trained without Yield >= 10000",
            **evaluate_regression(
                y_holdout_weather.to_numpy()[extreme_actual_mask],
                predictions_without_extremes[extreme_actual_mask],
            ),
        },
    ]).round(4))

print("\nLargest predictions from experimental model:")
display(
    error_weather_check[
        ["State", "District", "Crop", "Season", "Year", "Actual_Yield"]
    ].assign(
        Original_Prediction=np.asarray(weather_hgb_predictions),
        Prediction_Without_Extremes=predictions_without_extremes,
    )
    .sort_values("Original_Prediction", ascending=False)
    .head(15)
    .round(2)
)

Original weather-complete training rows: 76317
Rows excluded in this experiment: 85
Experimental training rows: 76232
Percentage of training rows excluded: 0.111


,Model,MAE,RMSE,R2
0,Original weather HGB,722.4263,1092.8120,0.4124
1,HGB trained without Yield >= 10000,656.1463,935.8975,0.5690



Holdout rows with actual Yield >= 10000: 12

Metrics on extreme-actual-yield holdout rows:


,Model,MAE,RMSE,R2
0,Original weather HGB,4910.4667,5180.7419,-41.6539
1,HGB trained without Yield >= 10000,4782.2266,5098.2084,-40.3057



Largest predictions from experimental model:


,State,District,Crop,Season,Year,Actual_Yield,Original_Prediction,Prediction_Without_Extremes
314552,Andhra Pradesh,Tirupati,Nutri/Coarse Cereals,Kharif,2023-24,3590.0,13162.42,4777.01
314540,Andhra Pradesh,Tirupati,Cereals,Kharif,2023-24,3745.0,12603.31,4024.97
314561,Andhra Pradesh,Tirupati,Rice,Kharif,2023-24,3754.0,12523.65,3946.91
345717,Andhra Pradesh,Prakasam,Maize,Kharif,2024-25,4127.0,12484.84,5831.98
314569,Andhra Pradesh,Tirupati,Total Food Grains,Kharif,2023-24,3740.0,12206.82,3527.27
345171,Andhra Pradesh,Bapatla,Maize,Kharif,2024-25,5253.0,11889.29,5934.24
314361,Andhra Pradesh,Prakasam,Maize,Kharif,2023-24,3460.0,11787.28,5895.16
314669,Andhra Pradesh,West Godavari,Cereals,Kharif,2023-24,3672.0,11769.24,3919.32
313895,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,Cereals,Kharif,2023-24,3518.0,11769.24,3919.32
313906,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,Rice,Kharif,2023-24,3518.0,11433.90,3929.00


In [44]:
# Cell 42 — Audit extreme-yield training records before any data-quality decision.

extreme_train_audit = primary_train_weather.loc[
    y_train_weather >= 10000
].copy()

print("Extreme training records:", len(extreme_train_audit))
print(
    "Share of weather-complete training data:",
    round(len(extreme_train_audit) / len(primary_train_weather) * 100, 4),
)

audit_columns = [
    col for col in [
        "State", "District", "Crop", "Season", "Year",
        "Area", "Production", "Yield"
    ]
    if col in extreme_train_audit.columns
]

print("\n1. All extreme training records")
display(
    extreme_train_audit[audit_columns]
    .sort_values("Yield", ascending=False)
    .round(4)
)

print("\n2. Extreme records by crop and season")
display(
    extreme_train_audit.groupby(["Crop", "Season"])
    .agg(
        Records=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        Maximum_Yield=("Yield", "max"),
        Median_Area=("Area", "median"),
        Median_Production=("Production", "median"),
    )
    .sort_values("Records", ascending=False)
    .round(3)
)

print("\n3. Extreme records by area band")
extreme_train_audit["Area_Band"] = pd.cut(
    extreme_train_audit["Area"],
    bins=[0, 0.01, 0.05, 0.1, 0.5, 1, float("inf")],
    labels=[
        "(0, 0.01]", "(0.01, 0.05]", "(0.05, 0.1]",
        "(0.1, 0.5]", "(0.5, 1]", ">1"
    ],
    include_lowest=True,
)

display(
    extreme_train_audit.groupby("Area_Band", observed=False)
    .agg(
        Records=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        Maximum_Yield=("Yield", "max"),
    )
    .round(3)
)

print("\n4. Potential repeated district-year-season measurements")
duplicate_check = (
    extreme_train_audit.groupby(
        ["State", "District", "Year", "Season"], dropna=False
    )
    .agg(
        Records=("Yield", "size"),
        Distinct_Crops=("Crop", "nunique"),
        Crops=("Crop", lambda x: ", ".join(sorted(x.astype(str).unique()))),
        Minimum_Yield=("Yield", "min"),
        Maximum_Yield=("Yield", "max"),
    )
    .query("Records > 1")
    .sort_values(["Records", "Maximum_Yield"], ascending=False)
)

display(duplicate_check.round(3))

Extreme training records: 85
Share of weather-complete training data: 0.1114

1. All extreme training records


,State,District,Crop,Season,Year,Area,Production,Yield
288718,Goa,North Goa,Cereals,Kharif,2022-23,0.01,0.47,44815.0
288727,Goa,North Goa,Rice,Kharif,2022-23,0.01,0.47,44815.0
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,44815.0
288736,Goa,South Goa,Cereals,Kharif,2022-23,0.01,0.47,39221.0
288750,Goa,South Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,39221.0
...,...,...,...,...,...,...,...,...
211690,Tamil Nadu,Perambalur,Maize,Kharif,2019-20,0.56,5.60,10058.0
86216,Tamil Nadu,Namakkal,Maize,Kharif,2015-16,0.04,0.40,10047.0
192298,Bihar,Kishanganj,Nutri/Coarse Cereals,Rabi,2019-20,0.01,0.13,10023.0
196527,Haryana,Karnal,Other Pulses,Kharif,2019-20,0.03,0.32,10000.0



2. Extreme records by crop and season


,,Records,Median_Yield,Maximum_Yield,Median_Area,Median_Production
Crop,Season,,,,,
Maize,Rabi,22,10541.5,12526.0,0.280,3.085
Nutri/Coarse Cereals,Rabi,17,10500.0,11969.0,0.290,3.170
Maize,Kharif,14,10652.0,11978.0,0.115,1.170
Other Pulses,Kharif,14,11934.0,24581.0,0.010,0.105
Cereals,Kharif,3,39221.0,44815.0,0.010,0.470
Rice,Kharif,3,39221.0,44815.0,0.010,0.470
Maize,Summer,3,10200.0,12525.0,0.050,0.580
Total Food Grains,Kharif,3,39221.0,44815.0,0.010,0.470
Nutri/Coarse Cereals,Summer,3,10200.0,12525.0,0.050,0.580



3. Extreme records by area band


,Records,Median_Yield,Maximum_Yield
Area_Band,,,
"(0, 0.01]",22,12979.0,44815.0
"(0.01, 0.05]",17,10716.0,20143.0
"(0.05, 0.1]",4,10497.0,11978.0
"(0.1, 0.5]",40,10587.0,12526.0
"(0.5, 1]",2,10430.0,10802.0
>1,0,NaN,NaN



4. Potential repeated district-year-season measurements


Records  Distinct_Crops  \
State          District         Year    Season                            
Tamil Nadu     Perambalur       2017-18 Kharif        4               4   
Goa            North Goa        2022-23 Kharif        3               3   
               South Goa        2022-23 Kharif        3               3   
Andhra Pradesh Guntur           2022-23 Rabi          3               3   
West Bengal    Dakshin Dinajpur 2022-23 Summer        2               2   
Bihar          Katihar          2016-17 Rabi          2               2   
Andhra Pradesh Eluru            2022-23 Rabi          2               2   
               Bapatla          2022-23 Rabi          2               2   
Tamil Nadu     Perambalur       2014-15 Kharif        2               2   
Bihar          Supaul           2019-20 Rabi          2               2   
Telangana      Mulugu           2019-20 Rabi          2               2   
Bihar          Katihar          2022-23 Rabi          2               2   
Andhra Pradesh East Godavari    2022-23 Rabi          2               2   
West Bengal    Uttar Dinajpur   2021-22 Rabi          2               2   
                                2020-21 Rabi          2               2   
Bihar          Katihar          2017-18 Rabi          2               2   
Telangana      Warangal         2016-17 Rabi          2               2   
Bihar          Katihar          2015-16 Rabi          2               2   
                                2019-20 Rabi          2               2   
               Kishanganj       2019-20 Rabi          2               2   
West Bengal    Uttar Dinajpur   2019-20 Rabi          2               2   
                                2018-19 Summer        2               2   
               Dakshin Dinajpur 2021-22 Summer        2               2   
Bihar          Supaul           2018-19 Rabi          2               2   

                                                                                            Crops  \
State          District         Year    Season                                                      
Tamil Nadu     Perambalur       2017-18 Kharif  Cereals, Maize, Nutri/Coarse Cereals, Total Fo...   
Goa            North Goa        2022-23 Kharif                   Cereals, Rice, Total Food Grains   
               South Goa        2022-23 Kharif                   Cereals, Rice, Total Food Grains   
Andhra Pradesh Guntur           2022-23 Rabi                 Cereals, Maize, Nutri/Coarse Cereals   
West Bengal    Dakshin Dinajpur 2022-23 Summer                        Maize, Nutri/Coarse Cereals   
Bihar          Katihar          2016-17 Rabi                          Maize, Nutri/Coarse Cereals   
Andhra Pradesh Eluru            2022-23 Rabi                          Maize, Nutri/Coarse Cereals   
               Bapatla          2022-23 Rabi                          Maize, Nutri/Coarse Cereals   
Tamil Nadu     Perambalur       2014-15 Kharif                        Maize, Nutri/Coarse Cereals   
Bihar          Supaul           2019-20 Rabi                          Maize, Nutri/Coarse Cereals   
Telangana      Mulugu           2019-20 Rabi                          Maize, Nutri/Coarse Cereals   
Bihar          Katihar          2022-23 Rabi                          Maize, Nutri/Coarse Cereals   
Andhra Pradesh East Godavari    2022-23 Rabi                          Maize, Nutri/Coarse Cereals   
West Bengal    Uttar Dinajpur   2021-22 Rabi                          Maize, Nutri/Coarse Cereals   
                                2020-21 Rabi                          Maize, Nutri/Coarse Cereals   
Bihar          Katihar          2017-18 Rabi                          Maize, Nutri/Coarse Cereals   
Telangana      Warangal         2016-17 Rabi                          Maize, Nutri/Coarse Cereals   
Bihar          Katihar          2015-16 Rabi                          Maize, Nutri/Coarse Cereals   
                                2019-20 Rabi                          Maize, 

In [45]:
# Cell 43 — Verify extreme training targets against the saved APY source.

source_long_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
)

source_long_df = pd.read_csv(source_long_path)

match_keys = ["State", "District", "Crop", "Season", "Year"]
measurement_columns = ["Area", "Production", "Yield"]

source_check = source_long_df[
    match_keys + measurement_columns
].copy()

extreme_source_check = extreme_train_audit[
    match_keys + measurement_columns
].copy()

# Normalize key fields to avoid false mismatches caused by whitespace.
for column in match_keys:
    source_check[column] = source_check[column].astype("string").str.strip()
    extreme_source_check[column] = (
        extreme_source_check[column].astype("string").str.strip()
    )

# Convert measurements to numeric for exact source comparisons.
for column in measurement_columns:
    source_check[column] = pd.to_numeric(
        source_check[column], errors="coerce"
    )
    extreme_source_check[column] = pd.to_numeric(
        extreme_source_check[column], errors="coerce"
    )

source_key_counts = source_check.groupby(
    match_keys, dropna=False
).size()

duplicate_source_keys = int((source_key_counts > 1).sum())

print("Historical source rows:", len(source_check))
print("Duplicate source key combinations:", duplicate_source_keys)

# Aggregate matching source rows so duplicates remain visible in the audit.
source_summary = (
    source_check.groupby(match_keys, dropna=False)
    .agg(
        Source_Match_Count=("Yield", "size"),
        Source_Area_Min=("Area", "min"),
        Source_Area_Max=("Area", "max"),
        Source_Production_Min=("Production", "min"),
        Source_Production_Max=("Production", "max"),
        Source_Yield_Min=("Yield", "min"),
        Source_Yield_Max=("Yield", "max"),
    )
    .reset_index()
)

source_comparison = extreme_source_check.merge(
    source_summary,
    on=match_keys,
    how="left",
    validate="many_to_one",
)

source_comparison["Yield_Matches_Source"] = np.isclose(
    source_comparison["Yield"],
    source_comparison["Source_Yield_Min"],
    equal_nan=True,
) & np.isclose(
    source_comparison["Yield"],
    source_comparison["Source_Yield_Max"],
    equal_nan=True,
)

source_comparison["Area_Matches_Source"] = np.isclose(
    source_comparison["Area"],
    source_comparison["Source_Area_Min"],
    equal_nan=True,
) & np.isclose(
    source_comparison["Area"],
    source_comparison["Source_Area_Max"],
    equal_nan=True,
)

source_comparison["Production_Matches_Source"] = np.isclose(
    source_comparison["Production"],
    source_comparison["Source_Production_Min"],
    equal_nan=True,
) & np.isclose(
    source_comparison["Production"],
    source_comparison["Source_Production_Max"],
    equal_nan=True,
)

print("\nSource match summary:")
print(
    source_comparison[
        [
            "Source_Match_Count",
            "Yield_Matches_Source",
            "Area_Matches_Source",
            "Production_Matches_Source",
        ]
    ].value_counts(dropna=False)
)

print("\nRecords missing from source or with measurement mismatches:")
mismatch_mask = (
    source_comparison["Source_Match_Count"].isna()
    | ~source_comparison["Yield_Matches_Source"]
    | ~source_comparison["Area_Matches_Source"]
    | ~source_comparison["Production_Matches_Source"]
)

display(
    source_comparison.loc[
        mismatch_mask,
        match_keys + measurement_columns + [
            "Source_Match_Count",
            "Source_Area_Min", "Source_Area_Max",
            "Source_Production_Min", "Source_Production_Max",
            "Source_Yield_Min", "Source_Yield_Max",
        ],
    ].round(4)
)

print("\nGoa extreme records in the historical source:")
display(
    source_comparison.loc[
        (source_comparison["State"] == "Goa"),
        match_keys + measurement_columns + [
            "Source_Match_Count",
            "Source_Area_Min", "Source_Production_Min",
            "Source_Yield_Min", "Source_Yield_Max",
        ],
    ].round(4)
)

Historical source rows: 376272
Duplicate source key combinations: 0

Source match summary:
Source_Match_Count  Yield_Matches_Source  Area_Matches_Source  Production_Matches_Source
1                   True                  True                 True                         85
Name: count, dtype: int64

Records missing from source or with measurement mismatches:


,State,District,Crop,Season,Year,Area,Production,Yield,Source_Match_Count,Source_Area_Min,Source_Area_Max,Source_Production_Min,Source_Production_Max,Source_Yield_Min,Source_Yield_Max



Goa extreme records in the historical source:


,State,District,Crop,Season,Year,Area,Production,Yield,Source_Match_Count,Source_Area_Min,Source_Production_Min,Source_Yield_Min,Source_Yield_Max
77,Goa,North Goa,Cereals,Kharif,2022-23,0.01,0.47,44815.0,1,0.01,0.47,44815.0,44815.0
78,Goa,North Goa,Rice,Kharif,2022-23,0.01,0.47,44815.0,1,0.01,0.47,44815.0,44815.0
79,Goa,North Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,44815.0,1,0.01,0.47,44815.0,44815.0
80,Goa,South Goa,Cereals,Kharif,2022-23,0.01,0.47,39221.0,1,0.01,0.47,39221.0,39221.0
81,Goa,South Goa,Rice,Kharif,2022-23,0.01,0.47,39221.0,1,0.01,0.47,39221.0,39221.0
82,Goa,South Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,39221.0,1,0.01,0.47,39221.0,39221.0


In [46]:
# Cell 44 — Audit identical measurements across different crop labels.
# Read-only diagnostic: no records are removed or changed.

duplicate_measurement_keys = [
    "State", "District", "Season", "Year",
    "Area", "Production", "Yield"
]

repeated_measurement_groups = (
    source_check.groupby(duplicate_measurement_keys, dropna=False)
    .agg(
        Crop_Label_Count=("Crop", "nunique"),
        Crop_Labels=(
            "Crop",
            lambda values: ", ".join(sorted(set(values.dropna().astype(str))))
        ),
    )
    .reset_index()
)

repeated_measurement_groups = (
    repeated_measurement_groups.loc[
        repeated_measurement_groups["Crop_Label_Count"] >= 2
    ]
    .sort_values(
        ["Crop_Label_Count", "Yield"],
        ascending=[False, False],
    )
    .copy()
)

print("Source rows:", len(source_check))
print(
    "Distinct measurement combinations shared by multiple crop labels:",
    len(repeated_measurement_groups),
)
print(
    "Crop-label rows represented in these groups:",
    int(repeated_measurement_groups["Crop_Label_Count"].sum()),
)

print("\n1. Distribution by number of crop labels sharing measurements")
display(
    repeated_measurement_groups["Crop_Label_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Crop_Label_Count")
    .to_frame("Measurement_Groups")
)

print("\n2. Highest-yield repeated measurement groups")
display(
    repeated_measurement_groups[
        duplicate_measurement_keys + ["Crop_Label_Count", "Crop_Labels"]
    ]
    .head(30)
    .round(4)
)

print("\n3. Repeated measurement groups involving broad crop labels")
broad_crop_labels = {
    "Cereals",
    "Total Food Grains",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Small Millets",
}

repeated_measurement_groups["Contains_Broad_Label"] = (
    repeated_measurement_groups["Crop_Labels"].apply(
        lambda labels: any(
            label in broad_crop_labels
            for label in labels.split(", ")
        )
    )
)

display(
    repeated_measurement_groups["Contains_Broad_Label"]
    .value_counts()
    .rename_axis("Contains_Broad_Label")
    .to_frame("Measurement_Groups")
)

Source rows: 376272
Distinct measurement combinations shared by multiple crop labels: 42791
Crop-label rows represented in these groups: 138158

1. Distribution by number of crop labels sharing measurements


,Measurement_Groups
Crop_Label_Count,
2,25709
3,7537
4,3108
5,1883
6,1230
7,882
8,491
9,271
10,257



2. Highest-yield repeated measurement groups


,State,District,Season,Year,Area,Production,Yield,Crop_Label_Count,Crop_Labels
51136,Chhattisgarh,Jashpur,Total,2013-14,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
52578,Chhattisgarh,Korba,Total,2013-14,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53034,Chhattisgarh,Korea,Total,2013-14,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53474,Chhattisgarh,Mahasamund,Total,2013-14,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53747,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2013-14,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53748,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2014-15,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53749,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2015-16,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53750,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2016-17,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53751,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2017-18,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."
53752,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B),Total,2018-19,NaN,NaN,NaN,19,"Bajra, Barley, Cereals, Gram, Jowar, Lentil, M..."



3. Repeated measurement groups involving broad crop labels


,Measurement_Groups
Contains_Broad_Label,
True,34746
False,8045


In [47]:
# Cell 45 — Quantify crop-label pair overlap in the historical source.
# Read-only analysis; no data is changed.

from itertools import combinations
from collections import Counter

pair_counts = Counter()
pair_yield_extreme_counts = Counter()
pair_area_small_counts = Counter()

for row in repeated_measurement_groups.itertuples(index=False):
    labels = sorted(set(str(row.Crop_Labels).split(", ")))
    pairs = combinations(labels, 2)

    for pair in pairs:
        pair_counts[pair] += 1

        if row.Yield >= 10000:
            pair_yield_extreme_counts[pair] += 1

        if row.Area <= 0.05:
            pair_area_small_counts[pair] += 1

pair_overlap_df = pd.DataFrame([
    {
        "Crop_A": pair[0],
        "Crop_B": pair[1],
        "Shared_Measurement_Groups": count,
        "Groups_Yield_GE_10000": pair_yield_extreme_counts[pair],
        "Groups_Area_LE_005": pair_area_small_counts[pair],
    }
    for pair, count in pair_counts.items()
]).sort_values(
    "Shared_Measurement_Groups",
    ascending=False,
)

print("Distinct crop-label pairs sharing identical measurements:",
      len(pair_overlap_df))

print("\n1. Most frequent overlapping crop-label pairs")
display(pair_overlap_df.head(30))

print("\n2. Overlap involving broad crop labels")
broad_crop_labels = {
    "Cereals",
    "Total Food Grains",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Small Millets",
}

pair_overlap_df["Pair_Involves_Broad_Label"] = pair_overlap_df.apply(
    lambda row: (
        row["Crop_A"] in broad_crop_labels
        or row["Crop_B"] in broad_crop_labels
    ),
    axis=1,
)

display(
    pair_overlap_df.groupby("Pair_Involves_Broad_Label")
    .agg(
        Distinct_Pairs=("Shared_Measurement_Groups", "size"),
        Total_Shared_Groups=("Shared_Measurement_Groups", "sum"),
        Extreme_Yield_Groups=("Groups_Yield_GE_10000", "sum"),
    )
    .round(2)
)

print("\n3. Most frequent pairs involving extreme yields")
display(
    pair_overlap_df.loc[
        pair_overlap_df["Groups_Yield_GE_10000"] > 0
    ]
    .sort_values(
        ["Groups_Yield_GE_10000", "Shared_Measurement_Groups"],
        ascending=False,
    )
    .head(20)
)

Distinct crop-label pairs sharing identical measurements: 171

1. Most frequent overlapping crop-label pairs


,Crop_A,Crop_B,Shared_Measurement_Groups,Groups_Yield_GE_10000,Groups_Area_LE_005
94,Maize,Nutri/Coarse Cereals,10459,21,4572
46,Cereals,Total Food Grains,6392,3,1712
40,Cereals,Nutri/Coarse Cereals,5803,0,1809
43,Cereals,Rice,5341,3,1234
150,Shree Anna /Nutri Cereals,Small Millets,5052,0,1853
73,Jowar,Shree Anna /Nutri Cereals,4506,1,1192
119,Nutri/Coarse Cereals,Shree Anna /Nutri Cereals,4415,2,978
136,Ragi,Shree Anna /Nutri Cereals,4302,1,1504
11,Bajra,Shree Anna /Nutri Cereals,4115,0,966
38,Cereals,Maize,4061,0,1134



2. Overlap involving broad crop labels


,Distinct_Pairs,Total_Shared_Groups,Extreme_Yield_Groups
Pair_Involves_Broad_Label,,,
False,91,119356,1
True,80,169398,36



3. Most frequent pairs involving extreme yields


,Crop_A,Crop_B,Shared_Measurement_Groups,Groups_Yield_GE_10000,Groups_Area_LE_005,Pair_Involves_Broad_Label
94,Maize,Nutri/Coarse Cereals,10459,21,4572,True
46,Cereals,Total Food Grains,6392,3,1712,True
43,Cereals,Rice,5341,3,1234,True
145,Rice,Total Food Grains,3705,3,817,True
119,Nutri/Coarse Cereals,Shree Anna /Nutri Cereals,4415,2,978,True
73,Jowar,Shree Anna /Nutri Cereals,4506,1,1192,True
136,Ragi,Shree Anna /Nutri Cereals,4302,1,1504,True
69,Jowar,Nutri/Coarse Cereals,1878,1,108,True
117,Nutri/Coarse Cereals,Ragi,1839,1,210,True
131,Other Pulses,Total Pulses,1122,1,208,False


In [48]:
# Cell 46 — Inspect crop taxonomy and representative shared measurements.
# Read-only diagnostic.

print("1. Crop label to Crop_Category mapping")

if "Crop_Category" in source_long_df.columns:
    crop_taxonomy = (
        source_long_df[["Crop", "Crop_Category"]]
        .drop_duplicates()
        .sort_values(["Crop", "Crop_Category"])
    )

    display(crop_taxonomy)

    print("\nCrop labels mapped to multiple categories:")
    category_counts = crop_taxonomy.groupby("Crop")["Crop_Category"].nunique()
    display(category_counts[category_counts > 1].to_frame("Category_Count"))
else:
    print("Crop_Category is not available in the saved long-format source.")

print("\n2. Representative shared measurements for the most frequent pairs")

pairs_to_inspect = [
    ("Maize", "Nutri/Coarse Cereals"),
    ("Cereals", "Total Food Grains"),
    ("Cereals", "Rice"),
    ("Shree Anna /Nutri Cereals", "Small Millets"),
    ("Moong", "Urad"),
]

for crop_a, crop_b in pairs_to_inspect:
    print(f"\nPair: {crop_a} | {crop_b}")

    pair_rows = repeated_measurement_groups.loc[
        repeated_measurement_groups["Crop_Labels"].apply(
            lambda labels: crop_a in labels.split(", ")
            and crop_b in labels.split(", ")
        )
    ].sort_values("Yield", ascending=False)

    if pair_rows.empty:
        print("No shared measurement groups found.")
        continue

    display(
        pair_rows[
            [
                "State", "District", "Season", "Year",
                "Area", "Production", "Yield",
                "Crop_Label_Count", "Crop_Labels",
            ]
        ].head(8).round(4)
    )

1. Crop label to Crop_Category mapping


,Crop,Crop_Category
66,Bajra,Individual Crop Candidate
3405,Barley,Individual Crop Candidate
0,Cereals,Grouping — Verify Source Definition
111,Gram,Individual Crop Candidate
71,Jowar,Individual Crop Candidate
1209,Lentil,Individual Crop Candidate
3,Maize,Individual Crop Candidate
5,Moong,Individual Crop Candidate
7,Nutri/Coarse Cereals,Grouping — Verify Source Definition
9,Other Pulses,Grouping — Verify Source Definition



Crop labels mapped to multiple categories:


,Category_Count
Crop,



2. Representative shared measurements for the most frequent pairs

Pair: Maize | Nutri/Coarse Cereals


,State,District,Season,Year,Area,Production,Yield,Crop_Label_Count,Crop_Labels
114939,Kerala,Malappuram,Total,2018-19,0.00,0.00,15000.0,2,"Maize, Nutri/Coarse Cereals"
114849,Kerala,Malappuram,Kharif,2018-19,0.00,0.00,15000.0,2,"Maize, Nutri/Coarse Cereals"
273953,West Bengal,Dakshin Dinajpur,Summer,2022-23,0.05,0.58,12525.0,2,"Maize, Nutri/Coarse Cereals"
274077,West Bengal,Dakshin Dinajpur,Total,2022-23,0.05,0.58,12485.0,2,"Maize, Nutri/Coarse Cereals"
273963,West Bengal,Dakshin Dinajpur,Summer,2024-25,0.02,0.29,12237.0,2,"Maize, Nutri/Coarse Cereals"
274100,West Bengal,Dakshin Dinajpur,Total,2024-25,0.02,0.29,12237.0,2,"Maize, Nutri/Coarse Cereals"
273958,West Bengal,Dakshin Dinajpur,Summer,2023-24,0.03,0.33,11620.0,2,"Maize, Nutri/Coarse Cereals"
274088,West Bengal,Dakshin Dinajpur,Total,2023-24,0.03,0.33,11558.0,2,"Maize, Nutri/Coarse Cereals"


... (4 outputs left out)


Pair: Shree Anna /Nutri Cereals | Small Millets


,State,District,Season,Year,Area,Production,Yield,Crop_Label_Count,Crop_Labels
190510,Rajasthan,Jhunjhunu,Rabi,2013-14,0.00,0.00,5000.0,2,"Shree Anna /Nutri Cereals, Small Millets"
65120,Gujarat,Kheda,Rabi,2022-23,0.00,0.01,3180.0,2,"Shree Anna /Nutri Cereals, Small Millets"
66009,Gujarat,Mahisagar,Rabi,2014-15,0.00,0.00,3086.0,2,"Shree Anna /Nutri Cereals, Small Millets"
60171,Gujarat,Banas Kantha,Rabi,2022-23,0.03,0.08,3057.0,2,"Shree Anna /Nutri Cereals, Small Millets"
65580,Gujarat,Mahesana,Rabi,2017-18,0.00,0.00,3000.0,2,"Shree Anna /Nutri Cereals, Small Millets"
65608,Gujarat,Mahesana,Rabi,2022-23,0.00,0.01,2502.0,2,"Shree Anna /Nutri Cereals, Small Millets"
58863,Gujarat,Amreli,Rabi,2020-21,0.01,0.02,2250.0,3,"Nutri/Coarse Cereals, Shree Anna /Nutri Cereal..."
19979,Assam,Goalpara,Kharif,2013-14,0.00,0.01,2151.0,2,"Shree Anna /Nutri Cereals, Small Millets"



Pair: Moong | Urad


,State,District,Season,Year,Area,Production,Yield,Crop_Label_Count,Crop_Labels
137020,Madhya Pradesh,Sidhi,Summer,2022-23,0.0,0.0,1000.0,3,"Moong, Total Food Grains, Urad"
176655,Puducherry,Yanam,Total,2020-21,0.0,0.0,1000.0,3,"Moong, Total Pulses, Urad"
190233,Rajasthan,Jhalawar,Summer,2023-24,0.0,0.0,1000.0,3,"Moong, Total Food Grains, Urad"
190234,Rajasthan,Jhalawar,Summer,2024-25,0.0,0.0,1000.0,3,"Moong, Total Food Grains, Urad"
28950,Bihar,Aurangabad,Total,2016-17,0.0,0.0,1000.0,2,"Moong, Urad"
176244,Puducherry,Puducherry,Rabi,2017-18,0.0,0.0,1000.0,2,"Moong, Urad"
42085,Bihar,Saran,Kharif,2020-21,0.0,0.0,1000.0,2,"Moong, Urad"
33219,Bihar,Jamui,Total,2013-14,0.0,0.0,1000.0,2,"Moong, Urad"


In [51]:
# Cell 47 — Audit provisional crop taxonomy in the primary cohort.
# Read-only; no data is modified.

taxonomy_audit = primary_df.copy()

print("Primary cohort rows:", len(taxonomy_audit))

print("\n1. Record counts by provisional taxonomy category")

category_summary = (
    taxonomy_audit.groupby("Crop_Category", dropna=False)
    .agg(
        Records=("Yield", "size"),
        Distinct_Crops=("Crop", "nunique"),
        Median_Yield=("Yield", "median"),
        P95_Yield=("Yield", lambda x: x.quantile(0.95)),
        Maximum_Yield=("Yield", "max"),
        Extreme_Yield_GE_10000=("Yield", lambda x: (x >= 10000).sum()),
    )
)

category_summary["Share_of_Cohort_Pct"] = (
    category_summary["Records"] / len(taxonomy_audit) * 100
)

display(category_summary.round(3))

print("\n2. Record counts by crop label and category")

display(
    taxonomy_audit.groupby(
        ["Crop_Category", "Crop"], dropna=False
    )
    .agg(
        Records=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        Maximum_Yield=("Yield", "max"),
        Extreme_Yield_GE_10000=("Yield", lambda x: (x >= 10000).sum()),
    )
    .sort_values(
        ["Crop_Category", "Records"],
        ascending=[True, False],
    )
    .round(3)
)

print("\n3. Missing taxonomy assignments")

print(
    "Rows without Crop_Category:",
    int(taxonomy_audit["Crop_Category"].isna().sum()),
)

print(
    "Crop labels without a taxonomy mapping:",
    sorted(
        set(taxonomy_audit["Crop"].dropna().unique())
        - set(source_long_df["Crop"].dropna().unique())
    ),
)

Primary cohort rows: 107267

1. Record counts by provisional taxonomy category


,Records,Distinct_Crops,Median_Yield,P95_Yield,Maximum_Yield,Extreme_Yield_GE_10000,Share_of_Cohort_Pct
Crop_Category,,,,,,,
Aggregate Total — Exclude from Individual-Crop Analysis,18931,1,2189.0,4381.5,44815.0,3,17.648
Grouping — Verify Source Definition,38691,4,2061.0,4772.0,44815.0,46,36.070
Individual Crop Candidate,49645,13,1606.0,4500.0,44815.0,54,46.282



2. Record counts by crop label and category


Records  \
Crop_Category                                      Crop                                 
Aggregate Total — Exclude from Individual-Crop ... Total Food Grains            18931   
Grouping — Verify Source Definition                Cereals                      17850   
                                                   Nutri/Coarse Cereals         11636   
                                                   Shree Anna /Nutri Cereals     5571   
                                                   Other Pulses                  3634   
Individual Crop Candidate                          Rice                         11143   
                                                   Maize                         8355   
                                                   Wheat                         4969   
                                                   Urad                          4162   
                                                   Gram                          3426   
                                                   Moong                         3321   
                                                   Tur                           3149   
                                                   Jowar                         2907   
                                                   Bajra                         2336   
                                                   Lentil                        2064   
                                                   Barley                        1536   
                                                   Small Millets                 1160   
                                                   Ragi                          1117   

                                                                              Median_Yield  \
Crop_Category                                      Crop                                      
Aggregate Total — Exclude from Individual-Crop ... Total Food Grains                2189.0   
Grouping — Verify Source Definition                Cereals                          2557.0   
                                                   Nutri/Coarse Cereals             2167.0   
                                                   Shree Anna /Nutri Cereals        1207.0   
                                                   Other Pulses                      924.0   
Individual Crop Candidate                          Rice                             2560.0   
                                                   Maize                            2613.0   
                                                   Wheat                            2856.0   
                                                   Urad                              641.5   
                                                   Gram                             1142.5   
                                                   Moong                             620.0   
                                                   Tur                               900.0   
                                                   Jowar                            1129.0   
                                                   Bajra                            1597.0   
                                                   Lentil                            886.0   
                                                   Barley                           2738.5   
                                                   Small Millets                     952.5   
                                                   Ragi                             1368.0   

                                                                              Maximum_Yield  \
Crop_Category                                      Crop                                       
Aggregate Total — Exclude from Individual-Crop ... Total Food Grains                44815.0   
Grouping — Verify Source Definition                Cereals                          44815.0   
                                    


3. Missing taxonomy assignments
Rows without Crop_Category: 0
Crop labels without a taxonomy mapping: []


In [52]:
# Cell 48 — Create a separate individual-crop candidate cohort.
# No existing dataset or fitted model is overwritten.

individual_crop_label = "Individual Crop Candidate"

required_columns = ["Crop", "Crop_Category", "Yield", "FE_YearStart"]

for name, df in [
    ("primary_train_weather", primary_train_weather),
    ("primary_holdout_weather", primary_holdout_weather),
]:
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise KeyError(f"{name} is missing required columns: {sorted(missing)}")

individual_train = primary_train_weather.loc[
    primary_train_weather["Crop_Category"].eq(individual_crop_label)
].copy()

individual_holdout = primary_holdout_weather.loc[
    primary_holdout_weather["Crop_Category"].eq(individual_crop_label)
].copy()

print("Individual-crop candidate cohort — weather-complete rows")
print(f"Training rows: {len(individual_train):,}")
print(f"Holdout rows:  {len(individual_holdout):,}")

print("\nTraining years:")
print(sorted(individual_train["FE_YearStart"].dropna().unique().tolist()))

print("\nHoldout years:")
print(sorted(individual_holdout["FE_YearStart"].dropna().unique().tolist()))

print("\nTraining records by crop:")
display(
    individual_train.groupby("Crop")
    .agg(
        Records=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        Maximum_Yield=("Yield", "max"),
    )
    .sort_values("Records", ascending=False)
    .round(2)
)

print("\nHoldout records by crop:")
display(
    individual_holdout.groupby("Crop")
    .agg(
        Records=("Yield", "size"),
        Median_Yield=("Yield", "median"),
        Maximum_Yield=("Yield", "max"),
    )
    .sort_values("Records", ascending=False)
    .round(2)
)

assert len(individual_train) > 0, "Individual-crop training cohort is empty."
assert len(individual_holdout) > 0, "Individual-crop holdout cohort is empty."
assert set(individual_holdout["FE_YearStart"].dropna().unique()).issubset({2023, 2024})

print("\nCohort checks passed. Original datasets remain unchanged.")

Individual-crop candidate cohort — weather-complete rows
Training rows: 35,784
Holdout rows:  7,369

Training years:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

Holdout years:
[2023, 2024]

Training records by crop:


,Records,Median_Yield,Maximum_Yield
Crop,,,
Rice,7611,2474.0,44815.0
Maize,5896,2446.0,12526.0
Wheat,3758,2876.5,6010.0
Urad,3143,640.0,3000.0
Gram,2534,1150.0,3156.0
Tur,2376,913.0,3698.0
Moong,2358,608.0,2280.0
Jowar,2056,1080.0,8293.0
Bajra,1732,1624.5,4883.0



Holdout records by crop:


,Records,Median_Yield,Maximum_Yield
Crop,,,
Rice,1529,2796.0,5684.0
Maize,1347,2923.0,12237.0
Wheat,756,3157.5,5544.0
Urad,649,662.0,1917.0
Moong,534,731.0,2199.0
Gram,529,1236.0,2557.0
Tur,418,943.5,3135.0
Jowar,412,1286.0,8057.0
Bajra,365,1756.0,5382.0



Cohort checks passed. Original datasets remain unchanged.


In [53]:
# Cell 49 — Individual-crop HistGradientBoosting model.
# Same model configuration; comparison uses identical holdout rows.

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor

individual_categorical_features = core_categorical_features
individual_numeric_features = core_numeric_features + weather_features
individual_feature_columns = (
    individual_categorical_features + individual_numeric_features
)

# Validate required inputs before fitting.
for name, df in [
    ("individual_train", individual_train),
    ("individual_holdout", individual_holdout),
]:
    missing = set(individual_feature_columns + ["Yield"]) - set(df.columns)
    if missing:
        raise KeyError(f"{name} is missing columns: {sorted(missing)}")

    if df[individual_feature_columns].isna().any().any():
        raise ValueError(f"{name} contains missing model features.")

X_individual_train = individual_train[individual_feature_columns]
y_individual_train = individual_train["Yield"].astype(float)

X_individual_holdout = individual_holdout[individual_feature_columns]
y_individual_holdout = individual_holdout["Yield"].astype(float)

individual_weather_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            clone(core_encoder),
            individual_categorical_features,
        ),
        ("numeric", "passthrough", individual_numeric_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

individual_weather_hgb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)

individual_X_train_transformed = (
    individual_weather_preprocessor.fit_transform(X_individual_train)
)
individual_X_holdout_transformed = (
    individual_weather_preprocessor.transform(X_individual_holdout)
)

individual_weather_hgb_model.fit(
    individual_X_train_transformed,
    y_individual_train,
)

individual_weather_predictions = individual_weather_hgb_model.predict(
    individual_X_holdout_transformed
)

if not np.isfinite(individual_weather_predictions).all():
    raise ValueError("Individual-crop model produced non-finite predictions.")

# Align the existing full-cohort model predictions by holdout row index.
full_weather_prediction_series = pd.Series(
    weather_hgb_predictions,
    index=primary_holdout_weather.index,
    name="Full_Cohort_Prediction",
)

matched_full_predictions = full_weather_prediction_series.loc[
    individual_holdout.index
].to_numpy()

individual_comparison = pd.DataFrame([
    {
        "Model": "Full-cohort HGB + Weather",
        **evaluate_regression(
            y_individual_holdout,
            matched_full_predictions,
        ),
    },
    {
        "Model": "Individual-crop HGB + Weather",
        **evaluate_regression(
            y_individual_holdout,
            individual_weather_predictions,
        ),
    },
])

print("Training rows:", len(X_individual_train))
print("Matched holdout rows:", len(X_individual_holdout))
print("\nMatched holdout comparison:")
display(individual_comparison.round(4))

individual_comparison["MAE_Change_vs_Full"] = (
    individual_comparison["MAE"]
    - individual_comparison.loc[
        individual_comparison["Model"] == "Full-cohort HGB + Weather",
        "MAE",
    ].iloc[0]
)

print("\nIndividual-crop model prediction range:")
print(
    "Min:", round(float(individual_weather_predictions.min()), 2),
    "| Max:", round(float(individual_weather_predictions.max()), 2),
)

print("\nActual individual-crop holdout yield range:")
print(
    "Min:", round(float(y_individual_holdout.min()), 2),
    "| Max:", round(float(y_individual_holdout.max()), 2),
)

Training rows: 35784
Matched holdout rows: 7369

Matched holdout comparison:


,Model,MAE,RMSE,R2
0,Full-cohort HGB + Weather,610.5406,963.7067,0.5744
1,Individual-crop HGB + Weather,549.4963,830.2216,0.6841



Individual-crop model prediction range:
Min: 267.15 | Max: 8741.35

Actual individual-crop holdout yield range:
Min: 74.0 | Max: 12237.0


In [54]:
# Cell 50 — Crop-level comparison on the same individual-crop holdout rows.
# Diagnostic only; no model or source data is modified.

crop_model_comparison_df = pd.DataFrame({
    "Crop": individual_holdout["Crop"].to_numpy(),
    "Actual": y_individual_holdout.to_numpy(),
    "Full_Cohort_Prediction": matched_full_predictions,
    "Individual_Crop_Prediction": individual_weather_predictions,
})

crop_comparison_rows = []

for crop, group in crop_model_comparison_df.groupby("Crop"):
    actual = group["Actual"].to_numpy()
    full_pred = group["Full_Cohort_Prediction"].to_numpy()
    individual_pred = group["Individual_Crop_Prediction"].to_numpy()

    full_metrics = evaluate_regression(actual, full_pred)
    individual_metrics = evaluate_regression(actual, individual_pred)

    crop_comparison_rows.append({
        "Crop": crop,
        "N": len(group),
        "Actual_Mean": actual.mean(),
        "Full_Cohort_MAE": full_metrics["MAE"],
        "Individual_Crop_MAE": individual_metrics["MAE"],
        "MAE_Improvement": full_metrics["MAE"] - individual_metrics["MAE"],
        "MAE_Improvement_Pct": (
            (full_metrics["MAE"] - individual_metrics["MAE"])
            / full_metrics["MAE"] * 100
            if full_metrics["MAE"] != 0 else np.nan
        ),
        "Full_Cohort_RMSE": full_metrics["RMSE"],
        "Individual_Crop_RMSE": individual_metrics["RMSE"],
        "Full_Cohort_Bias": np.mean(full_pred - actual),
        "Individual_Crop_Bias": np.mean(individual_pred - actual),
    })

crop_model_comparison = (
    pd.DataFrame(crop_comparison_rows)
    .sort_values("MAE_Improvement_Pct", ascending=False)
    .reset_index(drop=True)
)

print("1. Crop-level model comparison (minimum 100 holdout rows)")
display(
    crop_model_comparison.loc[crop_model_comparison["N"] >= 100]
    .round(2)
)

print("\n2. Summary across evaluated crops")
evaluated_crops = crop_model_comparison.loc[
    crop_model_comparison["N"] >= 100
]

print("Crops evaluated:", len(evaluated_crops))
print(
    "Individual-crop model wins:",
    int((evaluated_crops["MAE_Improvement"] > 0).sum()),
)
print(
    "Full-cohort model wins:",
    int((evaluated_crops["MAE_Improvement"] < 0).sum()),
)
print(
    "Ties:",
    int((evaluated_crops["MAE_Improvement"] == 0).sum()),
)

print("\n3. Largest remaining individual-crop model errors")
display(
    crop_comparison_rows
    and crop_model_comparison
    .sort_values("Individual_Crop_MAE", ascending=False)
    .head(5)
    .round(2)
)

1. Crop-level model comparison (minimum 100 holdout rows)


,Crop,N,Actual_Mean,Full_Cohort_MAE,Individual_Crop_MAE,MAE_Improvement,MAE_Improvement_Pct,Full_Cohort_RMSE,Individual_Crop_RMSE,Full_Cohort_Bias,Individual_Crop_Bias
0,Ragi,142,1324.55,663.39,473.24,190.15,28.66,879.96,624.28,385.73,109.40
1,Tur,418,977.80,390.93,300.43,90.49,23.15,627.86,399.16,228.81,104.20
2,Urad,649,739.86,317.10,250.10,67.00,21.13,498.90,344.33,148.39,34.45
3,Gram,529,1278.43,372.22,295.86,76.36,20.51,493.84,373.58,215.24,74.85
4,Small Millets,146,1029.08,340.90,285.50,55.39,16.25,711.71,457.22,98.18,-13.56
5,Jowar,412,1434.96,625.67,527.48,98.18,15.69,872.63,801.56,193.21,-23.54
6,Lentil,314,962.47,244.82,209.58,35.24,14.39,304.20,262.47,106.00,51.09
7,Rice,1529,2792.97,690.39,605.50,84.89,12.30,1116.94,823.22,219.67,171.35
8,Moong,534,758.69,281.30,247.16,34.14,12.14,357.72,319.87,94.41,43.48
9,Bajra,365,1774.62,636.16,567.32,68.84,10.82,943.79,717.65,237.93,142.16



2. Summary across evaluated crops
Crops evaluated: 13
Individual-crop model wins: 13
Full-cohort model wins: 0
Ties: 0

3. Largest remaining individual-crop model errors


,Crop,N,Actual_Mean,Full_Cohort_MAE,Individual_Crop_MAE,MAE_Improvement,MAE_Improvement_Pct,Full_Cohort_RMSE,Individual_Crop_RMSE,Full_Cohort_Bias,Individual_Crop_Bias
10,Maize,1347,3444.75,1069.38,1022.67,46.70,4.37,1493.40,1409.28,62.85,-27.59
12,Wheat,756,3086.68,609.45,607.11,2.34,0.38,784.11,770.88,12.57,22.91
7,Rice,1529,2792.97,690.39,605.50,84.89,12.30,1116.94,823.22,219.67,171.35
9,Bajra,365,1774.62,636.16,567.32,68.84,10.82,943.79,717.65,237.93,142.16
5,Jowar,412,1434.96,625.67,527.48,98.18,15.69,872.63,801.56,193.21,-23.54


In [55]:
# Cell 51 — Inspect the 20 largest individual-crop model errors.
# Read-only diagnostic; no records are removed.

individual_error_analysis = individual_holdout[
    [
        "State", "District", "Crop", "Season", "Year",
        "FE_YearStart", "Area", "Production", "Yield",
        "Crop_Category",
    ] + weather_features
].copy()

individual_error_analysis["Actual_Yield"] = (
    y_individual_holdout.to_numpy()
)
individual_error_analysis["Predicted_Yield"] = (
    individual_weather_predictions
)
individual_error_analysis["Signed_Error"] = (
    individual_error_analysis["Predicted_Yield"]
    - individual_error_analysis["Actual_Yield"]
)
individual_error_analysis["Absolute_Error"] = (
    individual_error_analysis["Signed_Error"].abs()
)

print("Individual-crop model error quantiles")
display(
    individual_error_analysis["Absolute_Error"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .rename("Absolute_Error")
    .to_frame()
    .round(2)
)

print("\n1. Twenty largest absolute errors")

display(
    individual_error_analysis
    .sort_values("Absolute_Error", ascending=False)
    [
        [
            "State", "District", "Crop", "Season", "Year",
            "Area", "Production", "Actual_Yield",
            "Predicted_Yield", "Signed_Error", "Absolute_Error",
        ]
    ]
    .head(20)
    .round(2)
)

print("\n2. Errors for actual yields >= 10,000")

extreme_holdout_errors = individual_error_analysis.loc[
    individual_error_analysis["Actual_Yield"] >= 10000
].copy()

print("Extreme-yield rows:", len(extreme_holdout_errors))

if not extreme_holdout_errors.empty:
    display(
        extreme_holdout_errors[
            [
                "State", "District", "Crop", "Season", "Year",
                "Area", "Production", "Actual_Yield",
                "Predicted_Yield", "Signed_Error", "Absolute_Error",
            ]
        ]
        .sort_values("Absolute_Error", ascending=False)
        .round(2)
    )

print("\n3. Mean signed error by crop")

display(
    individual_error_analysis.groupby("Crop")
    .agg(
        N=("Actual_Yield", "size"),
        MAE=("Absolute_Error", "mean"),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
    )
    .sort_values("MAE", ascending=False)
    .round(2)
)

Individual-crop model error quantiles


,Absolute_Error
0.50,360.18
0.75,705.31
0.90,1227.98
0.95,1699.56
0.99,3159.90



1. Twenty largest absolute errors


,State,District,Crop,Season,Year,Area,Production,Actual_Yield,Predicted_Yield,Signed_Error,Absolute_Error
344173,West Bengal,Dakshin Dinajpur,Maize,Summer,2023-24,0.03,0.33,11620.0,4851.92,-6768.08,6768.08
375529,West Bengal,Dakshin Dinajpur,Maize,Summer,2024-25,0.02,0.29,12237.0,5555.59,-6681.41,6681.41
336885,Tamil Nadu,Krishnagiri,Maize,Kharif,2023-24,0.01,0.10,9070.0,2671.98,-6398.02,6398.02
344556,West Bengal,Nadia,Maize,Kharif,2023-24,0.02,0.24,9872.0,4410.13,-5461.87,5461.87
345365,Andhra Pradesh,Guntur,Jowar,Rabi,2024-25,0.13,1.05,8057.0,2686.66,-5370.34,5370.34
339038,Telangana,Nirmal,Maize,Rabi,2023-24,0.44,3.72,8532.0,3169.65,-5362.35,5362.35
316890,Assam,Udalguri,Maize,Rabi,2023-24,0.02,0.01,784.0,5973.83,5189.83,5189.83
320101,Goa,South Goa,Rice,Kharif,2023-24,0.13,0.35,2714.0,7772.89,5058.89,5058.89
327188,Madhya Pradesh,Burhanpur,Maize,Summer,2023-24,0.03,0.21,7206.0,2157.39,-5048.61,5048.61
339086,Telangana,Nizamabad,Maize,Rabi,2023-24,0.09,0.71,8034.0,3053.54,-4980.46,4980.46



2. Errors for actual yields >= 10,000
Extreme-yield rows: 7


,State,District,Crop,Season,Year,Area,Production,Actual_Yield,Predicted_Yield,Signed_Error,Absolute_Error
344173,West Bengal,Dakshin Dinajpur,Maize,Summer,2023-24,0.03,0.33,11620.0,4851.92,-6768.08,6768.08
375529,West Bengal,Dakshin Dinajpur,Maize,Summer,2024-25,0.02,0.29,12237.0,5555.59,-6681.41,6681.41
314012,Andhra Pradesh,Guntur,Maize,Rabi,2023-24,0.15,1.58,10821.0,6936.43,-3884.57,3884.57
345286,Andhra Pradesh,East Godavari,Maize,Rabi,2024-25,0.07,0.76,10422.0,6738.16,-3683.84,3683.84
345368,Andhra Pradesh,Guntur,Maize,Rabi,2024-25,0.20,2.09,10512.0,7094.11,-3417.89,3417.89
313930,Andhra Pradesh,East Godavari,Maize,Rabi,2023-24,0.08,0.80,10162.0,7085.07,-3076.93,3076.93
345325,Andhra Pradesh,Eluru,Maize,Rabi,2024-25,0.20,2.08,10210.0,7188.44,-3021.56,3021.56



3. Mean signed error by crop


,N,MAE,Mean_Signed_Error,Max_Absolute_Error
Crop,,,,
Maize,1347,1022.67,-27.59,6768.08
Wheat,756,607.11,22.91,2455.31
Rice,1529,605.50,171.35,5058.89
Bajra,365,567.32,142.16,3005.59
Jowar,412,527.48,-23.54,5370.34
Barley,228,488.75,52.56,1513.98
Ragi,142,473.24,109.40,2108.17
Tur,418,300.43,104.20,2498.63
Gram,529,295.86,74.85,1368.76


In [56]:
# Cell 52 — Training support versus holdout yield errors by crop.
# Diagnostic only; no records are removed.

training_yield_support = (
    individual_train.groupby("Crop")["Yield"]
    .agg(
        Train_N="size",
        Train_Median="median",
        Train_P95=lambda x: x.quantile(0.95),
        Train_P99=lambda x: x.quantile(0.99),
        Train_Max="max",
    )
)

holdout_support = (
    individual_error_analysis.groupby("Crop")
    .agg(
        Holdout_N=("Actual_Yield", "size"),
        Holdout_Median=("Actual_Yield", "median"),
        Holdout_P95=("Actual_Yield", lambda x: x.quantile(0.95)),
        Holdout_Max=("Actual_Yield", "max"),
        Holdout_MAE=("Absolute_Error", "mean"),
    )
)

support_comparison = training_yield_support.join(holdout_support)

support_comparison["Holdout_Max_Above_Train_Max"] = (
    support_comparison["Holdout_Max"] > support_comparison["Train_Max"]
)

support_comparison["Holdout_Max_to_Train_P99"] = (
    support_comparison["Holdout_Max"] / support_comparison["Train_P99"].replace(0, np.nan)
)

print("1. Training support compared with holdout yield distribution")

display(
    support_comparison.sort_values("Holdout_MAE", ascending=False).round(2)
)

print("\n2. Holdout errors by actual-yield band")

error_df = individual_error_analysis.copy()

error_df["Yield_Band"] = pd.cut(
    error_df["Actual_Yield"],
    bins=[-np.inf, 1000, 2000, 3000, 5000, 10000, np.inf],
    labels=[
        "<=1000",
        "1000–2000",
        "2000–3000",
        "3000–5000",
        "5000–10000",
        ">10000",
    ],
)

yield_band_summary = (
    error_df.groupby("Yield_Band", observed=False)
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Maximum_Absolute_Error=("Absolute_Error", "max"),
    )
)

display(yield_band_summary.round(2))

print("\n3. High-yield holdout rows versus the training 99th percentile")

train_p99_by_crop = individual_train.groupby("Crop")["Yield"].quantile(0.99)

high_yield_holdout = error_df.loc[
    error_df.apply(
        lambda row: row["Actual_Yield"] > train_p99_by_crop.get(row["Crop"], np.inf),
        axis=1,
    )
].copy()

print(
    "Holdout rows above their crop's training P99:",
    len(high_yield_holdout),
)

display(
    high_yield_holdout[
        [
            "State", "District", "Crop", "Season", "Year",
            "Actual_Yield", "Predicted_Yield",
            "Signed_Error", "Absolute_Error",
        ]
    ]
    .sort_values("Absolute_Error", ascending=False)
    .head(20)
    .round(2)
)

1. Training support compared with holdout yield distribution


,Train_N,Train_Median,Train_P95,Train_P99,Train_Max,Holdout_N,Holdout_Median,Holdout_P95,Holdout_Max,Holdout_MAE,Holdout_Max_Above_Train_Max,Holdout_Max_to_Train_P99
Crop,,,,,,,,,,,,
Maize,5896,2446.0,7602.00,9642.00,12526.0,1347,2923.0,7619.50,12237.0,1022.67,False,1.27
Wheat,3758,2876.5,4824.75,5322.59,6010.0,756,3157.5,4926.75,5544.0,607.11,False,1.04
Rice,7611,2474.0,4271.00,5005.80,44815.0,1529,2796.0,4245.40,5684.0,605.50,False,1.14
Bajra,1732,1624.5,3134.00,3663.28,4883.0,365,1756.0,3148.40,5382.0,567.32,True,1.47
Jowar,2056,1080.0,2488.25,4250.00,8293.0,412,1286.0,2714.00,8057.0,527.48,False,1.90
Barley,1233,2717.0,4000.00,4518.36,5400.0,228,2768.0,3842.30,4808.0,488.75,False,1.06
Ragi,673,1408.0,4034.80,4732.88,5230.0,142,1099.5,3192.40,4008.0,473.24,False,0.85
Tur,2376,913.0,1816.25,2373.75,3698.0,418,943.5,1626.65,3135.0,300.43,False,1.32
Gram,2534,1150.0,1928.70,2347.00,3156.0,529,1236.0,1917.60,2557.0,295.86,False,1.09



2. Holdout errors by actual-yield band


,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,Maximum_Absolute_Error
Yield_Band,,,,,,
<=1000,1904,694.08,361.80,544.33,317.67,5189.83
1000–2000,2169,1437.29,458.87,658.92,224.33,4467.63
2000–3000,1479,2500.05,496.64,728.16,172.21,5058.89
3000–5000,1510,3681.16,728.87,957.13,-319.84,4712.96
5000–10000,300,6410.93,1664.74,2046.49,-1259.90,6398.02
>10000,7,10854.86,4362.04,4619.65,-4362.04,6768.08



3. High-yield holdout rows versus the training 99th percentile
Holdout rows above their crop's training P99: 85


,State,District,Crop,Season,Year,Actual_Yield,Predicted_Yield,Signed_Error,Absolute_Error
344173,West Bengal,Dakshin Dinajpur,Maize,Summer,2023-24,11620.0,4851.92,-6768.08,6768.08
375529,West Bengal,Dakshin Dinajpur,Maize,Summer,2024-25,12237.0,5555.59,-6681.41,6681.41
344556,West Bengal,Nadia,Maize,Kharif,2023-24,9872.0,4410.13,-5461.87,5461.87
345365,Andhra Pradesh,Guntur,Jowar,Rabi,2024-25,8057.0,2686.66,-5370.34,5370.34
348246,Assam,Udalguri,Maize,Rabi,2024-25,9974.0,5146.13,-4827.87,4827.87
345169,Andhra Pradesh,Bapatla,Jowar,Rabi,2024-25,6919.0,2148.63,-4770.37,4770.37
318619,Bihar,Supaul,Maize,Rabi,2023-24,9708.0,5147.24,-4560.76,4560.76
313813,Andhra Pradesh,Bapatla,Jowar,Rabi,2023-24,6036.0,1910.05,-4125.95,4125.95
314012,Andhra Pradesh,Guntur,Maize,Rabi,2023-24,10821.0,6936.43,-3884.57,3884.57
345286,Andhra Pradesh,East Godavari,Maize,Rabi,2024-25,10422.0,6738.16,-3683.84,3683.84


In [57]:
# Cell 53 — Individual-crop errors by area band and season.
# Read-only diagnostic; retains all holdout observations.

area_error_df = individual_error_analysis.copy()

area_error_df["Area_Band"] = pd.cut(
    area_error_df["Area"],
    bins=[0, 0.01, 0.05, 0.1, 0.5, 1, np.inf],
    labels=["<=0.01", "(0.01,0.05]", "(0.05,0.1]", "(0.1,0.5]", "(0.5,1]", ">1"],
    include_lowest=True,
)

print("1. Error metrics by Area band")

display(
    area_error_df.groupby("Area_Band", observed=False)
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
    )
    .round(2)
)

print("\n2. Error metrics by Season")

display(
    area_error_df.groupby("Season", dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
    )
    .sort_values("MAE", ascending=False)
    .round(2)
)

print("\n3. High-error rows by area band")

area_error_df["High_Error_Abs_3000"] = (
    area_error_df["Absolute_Error"] >= 3000
)

display(
    area_error_df.groupby("Area_Band", observed=False)
    .agg(
        N=("Absolute_Error", "size"),
        Rows_Abs_Error_At_Least_3000=("High_Error_Abs_3000", "sum"),
    )
    .assign(
        High_Error_Percent=lambda d: (
            100 * d["Rows_Abs_Error_At_Least_3000"] / d["N"]
        )
    )
    .round(2)
)

1. Error metrics by Area band


,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,Max_Absolute_Error
Area_Band,,,,,,
<=0.01,1427,1843.44,544.57,842.07,103.64,6398.02
"(0.01,0.05]",1945,1761.54,516.41,811.37,123.45,6768.08
"(0.05,0.1]",868,2080.14,552.37,842.29,62.76,4980.46
"(0.1,0.5]",1725,2284.86,579.23,875.52,34.36,5370.34
"(0.5,1]",679,2661.15,556.24,778.79,6.03,3654.92
>1,725,2874.11,567.46,776.37,-70.60,4722.30



2. Error metrics by Season


,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,Max_Absolute_Error
Season,,,,,,
Rabi,2708,2477.02,599.90,913.77,42.10,5370.34
Summer,928,2217.11,531.59,830.45,-39.05,6768.08
Kharif,3733,1856.18,517.38,763.85,100.96,6398.02



3. High-error rows by area band


,N,Rows_Abs_Error_At_Least_3000,High_Error_Percent
Area_Band,,,
<=0.01,1427,20,1.40
"(0.01,0.05]",1945,22,1.13
"(0.05,0.1]",868,13,1.50
"(0.1,0.5]",1725,29,1.68
"(0.5,1]",679,5,0.74
>1,725,5,0.69


In [58]:
# Cell 54 — Error analysis by crop and season.
# Diagnostic only; no records are removed.

crop_season_errors = (
    individual_error_analysis
    .groupby(["Crop", "Season"], dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        Max_Absolute_Error=("Absolute_Error", "max"),
        High_Error_Count=("Absolute_Error", lambda x: (x >= 3000).sum()),
    )
    .reset_index()
)

crop_season_errors["High_Error_Percent"] = (
    100 * crop_season_errors["High_Error_Count"]
    / crop_season_errors["N"]
)

print("1. Crop-season groups with at least 50 holdout records, ranked by MAE")

display(
    crop_season_errors.loc[crop_season_errors["N"] >= 50]
    .sort_values("MAE", ascending=False)
    .round(2)
)

print("\n2. Maize-specific season comparison")

display(
    crop_season_errors.loc[crop_season_errors["Crop"] == "Maize"]
    .sort_values("MAE", ascending=False)
    .round(2)
)

print("\n3. Number of crop-season groups with high-error rates above 2%")

eligible_groups = crop_season_errors.loc[crop_season_errors["N"] >= 50]

print(
    int((eligible_groups["High_Error_Percent"] > 2).sum()),
    "of",
    len(eligible_groups),
    "eligible groups",
)

1. Crop-season groups with at least 50 holdout records, ranked by MAE


,Crop,Season,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,Max_Absolute_Error,High_Error_Count,High_Error_Percent
9,Maize,Rabi,382,4542.55,1444.94,1822.55,5.57,5362.35,45,11.78
10,Maize,Summer,226,3630.31,1024.46,1424.67,-491.97,6768.08,9,3.98
8,Maize,Kharif,739,2820.54,803.85,1132.39,97.29,6398.02,21,2.84
5,Jowar,Rabi,136,1654.74,692.91,1123.91,-145.98,5370.34,6,4.41
1,Bajra,Summer,73,2163.08,686.81,836.80,297.97,1674.17,0,0.00
16,Rice,Rabi,200,3549.80,668.36,878.66,165.60,3159.57,1,0.50
15,Rice,Kharif,1096,2584.12,625.44,851.44,176.24,5058.89,11,1.00
23,Wheat,Rabi,756,3086.68,607.11,770.88,22.91,2455.31,0,0.00
0,Bajra,Kharif,292,1677.51,537.45,684.63,103.21,3005.59,1,0.34
2,Barley,Rabi,228,2633.78,488.75,615.38,52.56,1513.98,0,0.00



2. Maize-specific season comparison


,Crop,Season,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,Max_Absolute_Error,High_Error_Count,High_Error_Percent
9,Maize,Rabi,382,4542.55,1444.94,1822.55,5.57,5362.35,45,11.78
10,Maize,Summer,226,3630.31,1024.46,1424.67,-491.97,6768.08,9,3.98
8,Maize,Kharif,739,2820.54,803.85,1132.39,97.29,6398.02,21,2.84



3. Number of crop-season groups with high-error rates above 2%
4 of 22 eligible groups


In [59]:
# Cell 55 — Rabi Maize diagnostic.
# Read-only analysis; no records are removed.

train_rabi_maize = individual_train.loc[
    (individual_train["Crop"] == "Maize")
    & (individual_train["Season"] == "Rabi")
].copy()

holdout_rabi_maize = individual_error_analysis.loc[
    (individual_error_analysis["Crop"] == "Maize")
    & (individual_error_analysis["Season"] == "Rabi")
].copy()

print("1. Training versus holdout yield distribution")

distribution_summary = pd.DataFrame({
    "Training": train_rabi_maize["Yield"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
    "Holdout": holdout_rabi_maize["Actual_Yield"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
})

display(distribution_summary.round(2))

print("\n2. Rabi Maize errors by state (minimum 10 records)")

rabi_state_errors = (
    holdout_rabi_maize.groupby("State")
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        High_Error_Count=("Absolute_Error", lambda x: (x >= 3000).sum()),
    )
)

rabi_state_errors = rabi_state_errors.loc[
    rabi_state_errors["N"] >= 10
].sort_values("MAE", ascending=False)

display(rabi_state_errors.round(2))

print("\n3. Largest Rabi Maize errors")

display(
    holdout_rabi_maize.sort_values("Absolute_Error", ascending=False)[
        [
            "State", "District", "Year", "Area", "Production",
            "Actual_Yield", "Predicted_Yield",
            "Signed_Error", "Absolute_Error",
        ]
    ].head(15).round(2)
)

1. Training versus holdout yield distribution


,Training,Holdout
count,1335.00,382.00
mean,4534.66,4542.55
std,2467.56,2303.12
min,157.00,784.00
50%,4009.00,4329.50
90%,8085.40,8016.80
95%,9033.20,8723.50
99%,10375.14,10171.12
max,12526.00,10821.00



2. Rabi Maize errors by state (minimum 10 records)


,N,Mean_Actual,MAE,RMSE,Mean_Signed_Error,High_Error_Count
State,,,,,,
Jharkhand,14,2221.43,2218.40,2425.11,2211.04,3
Bihar,51,6112.22,1833.73,2205.21,-1665.74,11
Telangana,44,6155.59,1706.97,2056.88,-619.37,5
Andhra Pradesh,38,7504.42,1669.51,1947.01,-892.18,7
West Bengal,10,6453.10,1594.20,1828.58,-1301.11,1
Karnataka,16,3603.81,1566.79,1937.48,671.19,3
Arunachal Pradesh,16,1727.31,1432.43,1854.95,1432.43,4
Assam,30,4618.27,1377.63,1902.37,-99.60,3
Maharashtra,46,2168.54,1355.01,1759.14,981.18,5



3. Largest Rabi Maize errors


,State,District,Year,Area,Production,Actual_Yield,Predicted_Yield,Signed_Error,Absolute_Error
339038,Telangana,Nirmal,2023-24,0.44,3.72,8532.0,3169.65,-5362.35,5362.35
316890,Assam,Udalguri,2023-24,0.02,0.01,784.0,5973.83,5189.83,5189.83
339086,Telangana,Nizamabad,2023-24,0.09,0.71,8034.0,3053.54,-4980.46,4980.46
348246,Assam,Udalguri,2024-25,0.02,0.23,9974.0,5146.13,-4827.87,4827.87
316430,Assam,Kokrajhar,2023-24,0.01,0.02,3489.0,8201.96,4712.96,4712.96
318619,Bihar,Supaul,2023-24,0.05,0.52,9708.0,5147.24,-4560.76,4560.76
318494,Bihar,Sheohar,2023-24,0.01,0.12,8065.0,3543.26,-4521.74,4521.74
361577,Maharashtra,Nanded,2024-25,0.09,0.14,1481.0,5948.63,4467.63,4467.63
317807,Bihar,Madhepura,2023-24,0.20,1.88,9361.0,4969.69,-4391.31,4391.31
317715,Bihar,Kishanganj,2023-24,0.45,4.02,8934.0,4729.40,-4204.60,4204.60


In [60]:
# Cell 56 — Rabi Maize temporal stability audit.
# Read-only diagnostic; no records are removed.

print("1. Rabi Maize performance by holdout year")

display(
    holdout_rabi_maize.groupby("Year", dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        Mean_Actual=("Actual_Yield", "mean"),
        Median_Actual=("Actual_Yield", "median"),
        P95_Actual=("Actual_Yield", lambda x: x.quantile(0.95)),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Signed_Error", lambda x: np.sqrt(np.mean(np.square(x)))),
        Mean_Signed_Error=("Signed_Error", "mean"),
        High_Error_Count=("Absolute_Error", lambda x: (x >= 3000).sum()),
    )
    .round(2)
)

print("\n2. Historical Rabi Maize yield distribution by training year")

display(
    train_rabi_maize.groupby("Year", dropna=False)["Yield"]
    .agg(
        N="size",
        Mean="mean",
        Median="median",
        P95=lambda x: x.quantile(0.95),
        Maximum="max",
    )
    .round(2)
)

print("\n3. Holdout Rabi Maize errors by year and state")

display(
    holdout_rabi_maize.groupby(["Year", "State"], dropna=False)
    .agg(
        N=("Actual_Yield", "size"),
        MAE=("Absolute_Error", "mean"),
        Mean_Signed_Error=("Signed_Error", "mean"),
        High_Error_Count=("Absolute_Error", lambda x: (x >= 3000).sum()),
    )
    .query("N >= 5")
    .sort_values(["Year", "MAE"], ascending=[True, False])
    .round(2)
)

1. Rabi Maize performance by holdout year


,N,Mean_Actual,Median_Actual,P95_Actual,MAE,RMSE,Mean_Signed_Error,High_Error_Count
Year,,,,,,,,
2023-24,184,4492.18,4282.5,8731.55,1473.66,1871.06,-208.74,19
2024-25,198,4589.35,4345.5,8579.05,1418.24,1776.29,204.73,26



2. Historical Rabi Maize yield distribution by training year


,N,Mean,Median,P95,Maximum
Year,,,,,
2013-14,110,4278.22,3910.0,7681.70,8675.0
2014-15,114,4467.13,3538.0,8514.55,10075.0
2015-16,121,3982.55,3028.0,8213.00,10301.0
2016-17,127,4204.90,3291.0,9004.80,11986.0
2017-18,138,4756.06,4016.0,9495.75,10982.0
2018-19,129,4622.70,4566.0,9121.20,10090.0
2019-20,130,5008.88,4990.0,9673.30,11139.0
2020-21,141,4594.91,4407.0,9074.00,10500.0
2021-22,144,4679.99,4564.0,8514.80,10550.0



3. Holdout Rabi Maize errors by year and state


N      MAE  Mean_Signed_Error  High_Error_Count
Year    State                                                              
2023-24 Telangana          21  1986.45           -1797.27                 4
        Jharkhand           6  1890.96            1873.79                 1
        Bihar              25  1865.65           -1661.10                 5
        Andhra Pradesh     20  1861.95           -1308.26                 3
        Assam              14  1557.30             355.45                 2
        Tamil Nadu         19  1524.01            1064.54                 1
        West Bengal         5  1521.71            -935.52                 0
        Arunachal Pradesh   8  1369.63            1369.63                 2
        Maharashtra        23  1340.01             845.12                 1
        Gujarat            11   898.61             507.25                 0
        Karnataka           8   845.20            -773.15                 0
        Uttar Pradesh       8   827.10             197.21                 0
        Nagaland            5   637.74             637.74                 0
        Manipur             9   464.91             320.96                 0
2024-25 Jharkhand           8  2463.98            2463.98                 2
        Karnataka           8  2288.39            2115.54                 3
        Bihar              26  1803.05           -1670.19                 6
        West Bengal         5  1666.70           -1666.70                 1
        Arunachal Pradesh   8  1495.23            1495.23                 2
        Andhra Pradesh     18  1455.69            -429.88                 4
        Telangana          23  1451.79             456.10                 1
        Maharashtra        23  1370.02            1117.24                 4
        Gujarat            12  1316.69            1275.80                 1
        Assam              16  1220.41            -497.77                 1
        Tamil Nadu         20  1156.46             357.19                 1
        Nagaland            5  1010.17            1010.17                 0
        Madhya Pradesh     14   806.07            -606.92                 0
        Uttar Pradesh       9   581.19             581.19                 0

In [61]:
# Cell 57 — Time-based validation: test the value of location features.
# This uses training-period years, not the repeatedly inspected final holdout.

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

validation_train = individual_train.loc[
    individual_train["FE_YearStart"] <= 2020
].copy()

validation_set = individual_train.loc[
    individual_train["FE_YearStart"].isin([2021, 2022])
].copy()

categorical_without_location = core_categorical_features
categorical_with_location = core_categorical_features + ["State", "District"]
numeric_for_validation = core_numeric_features + weather_features

X_vtrain_base = validation_train[
    categorical_without_location + numeric_for_validation
]
X_vvalid_base = validation_set[
    categorical_without_location + numeric_for_validation
]

X_vtrain_location = validation_train[
    categorical_with_location + numeric_for_validation
]
X_vvalid_location = validation_set[
    categorical_with_location + numeric_for_validation
]

y_vtrain = validation_train["Yield"].to_numpy(dtype=float)
y_vvalid = validation_set["Yield"].to_numpy(dtype=float)

base_preprocessor = ColumnTransformer(
    [
        ("categorical", clone(core_encoder), categorical_without_location),
        ("numeric", "passthrough", numeric_for_validation),
    ],
    remainder="drop",
)

location_preprocessor = ColumnTransformer(
    [
        ("categorical", clone(core_encoder), categorical_with_location),
        ("numeric", "passthrough", numeric_for_validation),
    ],
    remainder="drop",
)

validation_models = {
    "Core + Weather": Pipeline([
        ("preprocessor", base_preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )),
    ]),
    "Core + Weather + State + District": Pipeline([
        ("preprocessor", location_preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )),
    ]),
}

validation_results = []

for name, pipeline in validation_models.items():
    X_fit = X_vtrain_base if name == "Core + Weather" else X_vtrain_location
    X_eval = X_vvalid_base if name == "Core + Weather" else X_vvalid_location

    pipeline.fit(X_fit, y_vtrain)
    predictions = pipeline.predict(X_eval)
    metrics = evaluate_regression(y_vvalid, predictions)

    validation_results.append({
        "Model": name,
        "Train_N": len(y_vtrain),
        "Validation_N": len(y_vvalid),
        **metrics,
    })

print("Training years: 2013–14 through 2020–21")
print("Validation years: 2021–22 and 2022–23")

display(pd.DataFrame(validation_results).round(3))

Training years: 2013–14 through 2020–21
Validation years: 2021–22 and 2022–23


,Model,Train_N,Validation_N,MAE,RMSE,R2
0,Core + Weather,28475,7309,578.419,1061.183,0.572
1,Core + Weather + State + District,28475,7309,430.785,902.226,0.691


In [62]:
# Cell 58 — Validation metrics split by year.
# Reuses the already-fitted models; no refitting or holdout access.

yearly_validation_results = []

for year, year_data in validation_set.groupby("FE_YearStart"):
    y_true_year = year_data["Yield"].to_numpy(dtype=float)

    X_base_year = year_data[
        categorical_without_location + numeric_for_validation
    ]
    X_location_year = year_data[
        categorical_with_location + numeric_for_validation
    ]

    for name, pipeline in validation_models.items():
        X_eval = (
            X_base_year
            if name == "Core + Weather"
            else X_location_year
        )

        predictions = pipeline.predict(X_eval)
        metrics = evaluate_regression(y_true_year, predictions)

        yearly_validation_results.append({
            "Year": year,
            "Model": name,
            "N": len(y_true_year),
            **metrics,
        })

yearly_validation_df = pd.DataFrame(yearly_validation_results)

display(
    yearly_validation_df
    .sort_values(["Year", "Model"])
    .round(3)
)

print("\nDistrict coverage in validation versus training")

train_districts = set(validation_train["District"].dropna().unique())
valid_districts = set(validation_set["District"].dropna().unique())

print("Distinct training districts:", len(train_districts))
print("Distinct validation districts:", len(valid_districts))
print("Validation districts seen in training:", len(valid_districts & train_districts))
print("Validation districts unseen in training:", len(valid_districts - train_districts))

,Year,Model,N,MAE,RMSE,R2
0,2021,Core + Weather,3597,587.064,844.168,0.659
1,2021,Core + Weather + State + District,3597,417.706,600.314,0.828
2,2022,Core + Weather,3712,570.042,1235.633,0.515
3,2022,Core + Weather + State + District,3712,443.458,1119.640,0.601



District coverage in validation versus training
Distinct training districts: 600
Distinct validation districts: 616
Validation districts seen in training: 594
Validation districts unseen in training: 22


In [63]:
# Cell 59 — Known-district vs unseen-district validation performance.

validation_with_predictions = validation_set[
    ["State", "District", "FE_YearStart", "Yield"]
].copy()

validation_with_predictions["District_Seen_In_Training"] = (
    validation_with_predictions["District"].isin(train_districts)
)

X_base = validation_set[
    categorical_without_location + numeric_for_validation
]
X_location = validation_set[
    categorical_with_location + numeric_for_validation
]

validation_with_predictions["Pred_Core_Weather"] = (
    validation_models["Core + Weather"].predict(X_base)
)
validation_with_predictions["Pred_With_Location"] = (
    validation_models["Core + Weather + State + District"].predict(X_location)
)

coverage_results = []

for (seen_status, year), group in validation_with_predictions.groupby(
    ["District_Seen_In_Training", "FE_YearStart"]
):
    y_true_group = group["Yield"].to_numpy(dtype=float)

    for model_name, prediction_column in [
        ("Core + Weather", "Pred_Core_Weather"),
        ("Core + Weather + State + District", "Pred_With_Location"),
    ]:
        metrics = evaluate_regression(
            y_true_group,
            group[prediction_column].to_numpy(dtype=float),
        )

        coverage_results.append({
            "District_Status": "Seen" if seen_status else "Unseen",
            "Year": year,
            "Model": model_name,
            "N": len(group),
            **metrics,
        })

coverage_comparison_df = pd.DataFrame(coverage_results)

display(
    coverage_comparison_df
    .sort_values(["Year", "District_Status", "Model"])
    .round(3)
)

,District_Status,Year,Model,N,MAE,RMSE,R2
4,Seen,2021,Core + Weather,3594,586.804,844.071,0.659
5,Seen,2021,Core + Weather + State + District,3594,417.641,600.317,0.828
0,Unseen,2021,Core + Weather,3,898.558,954.150,-15.473
1,Unseen,2021,Core + Weather + State + District,3,495.394,596.415,-5.436
6,Seen,2022,Core + Weather,3591,564.355,1235.987,0.505
7,Seen,2022,Core + Weather + State + District,3591,435.637,1119.236,0.594
2,Unseen,2022,Core + Weather,121,738.814,1225.064,0.674
3,Unseen,2022,Core + Weather + State + District,121,675.582,1131.552,0.722


In [64]:
# Cell 60 — Rolling time-based validation for location features.

from sklearn.base import clone

rolling_windows = [
    {"Train_Through": 2018, "Valid_Years": [2019, 2020]},
    {"Train_Through": 2020, "Valid_Years": [2021, 2022]},
]

rolling_results = []

for window in rolling_windows:
    cutoff = window["Train_Through"]
    valid_years = window["Valid_Years"]

    fold_train = individual_train.loc[
        individual_train["FE_YearStart"] <= cutoff
    ].copy()

    fold_valid = individual_train.loc[
        individual_train["FE_YearStart"].isin(valid_years)
    ].copy()

    y_fit = fold_train["Yield"].to_numpy(dtype=float)
    y_eval = fold_valid["Yield"].to_numpy(dtype=float)

    feature_sets = {
        "Core + Weather": (
            categorical_without_location + numeric_for_validation
        ),
        "Core + Weather + State + District": (
            categorical_with_location + numeric_for_validation
        ),
    }

    for model_name, feature_columns in feature_sets.items():
        X_fit = fold_train[feature_columns]
        X_eval = fold_valid[feature_columns]

        pipeline = clone(validation_models[model_name])
        pipeline.fit(X_fit, y_fit)
        predictions = pipeline.predict(X_eval)

        metrics = evaluate_regression(y_eval, predictions)

        rolling_results.append({
            "Train_Through": cutoff,
            "Validation_Years": ", ".join(map(str, valid_years)),
            "Model": model_name,
            "Train_N": len(fold_train),
            "Validation_N": len(fold_valid),
            **metrics,
        })

rolling_validation_df = pd.DataFrame(rolling_results)

display(
    rolling_validation_df
    .sort_values(["Train_Through", "Model"])
    .round(3)
)

,Train_Through,Validation_Years,Model,Train_N,Validation_N,MAE,RMSE,R2
0,2018,"2019, 2020",Core + Weather,21226,7249,608.446,882.634,0.652
1,2018,"2019, 2020",Core + Weather + State + District,21226,7249,448.105,674.568,0.797
2,2020,"2021, 2022",Core + Weather,28475,7309,578.419,1061.183,0.572
3,2020,"2021, 2022",Core + Weather + State + District,28475,7309,430.785,902.226,0.691


In [65]:
# Cell 61 — Audit location coverage using State + District keys.

def make_location_key(df):
    state = df["State"].astype("string").str.strip()
    district = df["District"].astype("string").str.strip()

    valid = (
        state.notna() & district.notna()
        & state.ne("") & district.ne("")
    )

    return (state + " | " + district).where(valid)

location_coverage_results = []

for window in rolling_windows:
    cutoff = window["Train_Through"]
    valid_years = window["Valid_Years"]

    fold_train = individual_train.loc[
        individual_train["FE_YearStart"] <= cutoff
    ].copy()

    fold_valid = individual_train.loc[
        individual_train["FE_YearStart"].isin(valid_years)
    ].copy()

    train_keys = set(make_location_key(fold_train).dropna())
    valid_keys = make_location_key(fold_valid)

    seen_mask = valid_keys.isin(train_keys) & valid_keys.notna()
    unseen_mask = valid_keys.notna() & ~valid_keys.isin(train_keys)

    location_coverage_results.append({
        "Train_Through": cutoff,
        "Validation_Years": ", ".join(map(str, valid_years)),
        "Train_Rows": len(fold_train),
        "Validation_Rows": len(fold_valid),
        "Distinct_Training_Locations": len(train_keys),
        "Distinct_Validation_Locations": valid_keys.nunique(),
        "Rows_Known_Location": int(seen_mask.sum()),
        "Rows_Unseen_Location": int(unseen_mask.sum()),
        "Rows_Missing_Location": int(valid_keys.isna().sum()),
        "Distinct_Unseen_Locations": valid_keys[unseen_mask].nunique(),
    })

location_coverage_df = pd.DataFrame(location_coverage_results)
display(location_coverage_df.round(3))

,Train_Through,Validation_Years,Train_Rows,Validation_Rows,Distinct_Training_Locations,Distinct_Validation_Locations,Rows_Known_Location,Rows_Unseen_Location,Rows_Missing_Location,Distinct_Unseen_Locations
0,2018,"2019, 2020",21226,7249,585,601,7172,77,0,17
1,2020,"2021, 2022",28475,7309,602,618,7185,124,0,22


In [66]:
# Cell 62 — Evaluate rolling validation by State + District coverage.

location_validation_results = []

for window in rolling_windows:
    cutoff = window["Train_Through"]
    valid_years = window["Valid_Years"]

    fold_train = individual_train.loc[
        individual_train["FE_YearStart"] <= cutoff
    ].copy()

    fold_valid = individual_train.loc[
        individual_train["FE_YearStart"].isin(valid_years)
    ].copy()

    train_keys = set(make_location_key(fold_train).dropna())
    valid_keys = make_location_key(fold_valid)

    fold_valid = fold_valid.copy()
    fold_valid["Location_Status"] = np.where(
        valid_keys.isin(train_keys),
        "Known",
        "Unseen",
    )

    y_fit = fold_train["Yield"].to_numpy(dtype=float)

    for model_name in [
        "Core + Weather",
        "Core + Weather + State + District",
    ]:
        feature_columns = (
            categorical_without_location + numeric_for_validation
            if model_name == "Core + Weather"
            else categorical_with_location + numeric_for_validation
        )

        pipeline = clone(validation_models[model_name])
        pipeline.fit(fold_train[feature_columns], y_fit)

        predictions = pipeline.predict(fold_valid[feature_columns])
        fold_valid[f"Pred_{model_name}"] = predictions

        for status, group in fold_valid.groupby("Location_Status"):
            y_true_group = group["Yield"].to_numpy(dtype=float)
            y_pred_group = group[f"Pred_{model_name}"].to_numpy(dtype=float)

            metrics = evaluate_regression(y_true_group, y_pred_group)

            location_validation_results.append({
                "Train_Through": cutoff,
                "Validation_Years": ", ".join(map(str, valid_years)),
                "Location_Status": status,
                "Model": model_name,
                "N": len(group),
                "Distinct_Locations": make_location_key(group).nunique(),
                **metrics,
            })

location_validation_df = pd.DataFrame(location_validation_results)

display(
    location_validation_df
    .sort_values(["Train_Through", "Location_Status", "Model"])
    .round(3)
)

,Train_Through,Validation_Years,Location_Status,Model,N,Distinct_Locations,MAE,RMSE,R2
0,2018,"2019, 2020",Known,Core + Weather,7172,584,608.161,882.553,0.651
2,2018,"2019, 2020",Known,Core + Weather + State + District,7172,584,447.712,674.556,0.796
1,2018,"2019, 2020",Unseen,Core + Weather,77,17,634.940,890.096,0.719
3,2018,"2019, 2020",Unseen,Core + Weather + State + District,77,17,484.747,675.712,0.838
4,2020,"2021, 2022",Known,Core + Weather,7185,596,575.584,1058.249,0.568
6,2020,"2021, 2022",Known,Core + Weather + State + District,7185,596,426.635,897.969,0.689
5,2020,"2021, 2022",Unseen,Core + Weather,124,22,742.678,1219.220,0.672
7,2020,"2021, 2022",Unseen,Core + Weather + State + District,124,22,671.223,1121.623,0.723


In [67]:
# Cell 63 — Consolidated location-feature validation comparison.

summary_rows = []

for (cutoff, years, status), group in location_validation_df.groupby(
    ["Train_Through", "Validation_Years", "Location_Status"]
):
    metrics_by_model = group.set_index("Model")

    base = metrics_by_model.loc["Core + Weather"]
    location = metrics_by_model.loc["Core + Weather + State + District"]

    summary_rows.append({
        "Train_Through": cutoff,
        "Validation_Years": years,
        "Location_Status": status,
        "N": int(base["N"]),
        "Distinct_Locations": int(base["Distinct_Locations"]),
        "Base_MAE": base["MAE"],
        "Location_MAE": location["MAE"],
        "MAE_Improvement_Pct": (
            (base["MAE"] - location["MAE"]) / base["MAE"] * 100
        ),
        "Base_RMSE": base["RMSE"],
        "Location_RMSE": location["RMSE"],
        "RMSE_Improvement_Pct": (
            (base["RMSE"] - location["RMSE"]) / base["RMSE"] * 100
        ),
        "Base_R2": base["R2"],
        "Location_R2": location["R2"],
        "R2_Change": location["R2"] - base["R2"],
    })

location_validation_summary = pd.DataFrame(summary_rows)

display(
    location_validation_summary
    .sort_values(["Train_Through", "Location_Status"])
    .round(3)
)

print("\nInterpretation guardrails:")
print("- Validation uses historical time windows, not the final holdout.")
print("- Unseen-location samples are small; treat their metrics as preliminary.")
print("- Unseen districts can still be in states seen during training.")
print("- These results support retaining location features as candidates,")
print("  not claiming production readiness or causal effects.")

,Train_Through,Validation_Years,Location_Status,N,Distinct_Locations,Base_MAE,Location_MAE,MAE_Improvement_Pct,Base_RMSE,Location_RMSE,RMSE_Improvement_Pct,Base_R2,Location_R2,R2_Change
0,2018,"2019, 2020",Known,7172,584,608.161,447.712,26.383,882.553,674.556,23.568,0.651,0.796,0.145
1,2018,"2019, 2020",Unseen,77,17,634.940,484.747,23.655,890.096,675.712,24.086,0.719,0.838,0.119
2,2020,"2021, 2022",Known,7185,596,575.584,426.635,25.878,1058.249,897.969,15.146,0.568,0.689,0.121
3,2020,"2021, 2022",Unseen,124,22,742.678,671.223,9.621,1219.220,1121.623,8.005,0.672,0.723,0.050



Interpretation guardrails:
- Validation uses historical time windows, not the final holdout.
- Unseen-location samples are small; treat their metrics as preliminary.
- Unseen districts can still be in states seen during training.
- These results support retaining location features as candidates,
  not claiming production readiness or causal effects.


In [68]:
# Cell 64 — Incremental value of weather after adding location features.

weather_ablation_results = []

feature_sets = {
    "Core + Location": (
        core_categorical_features + ["State", "District"],
        core_numeric_features,
    ),
    "Core + Location + Weather": (
        core_categorical_features + ["State", "District"],
        core_numeric_features + weather_features,
    ),
}

for window in rolling_windows:
    cutoff = window["Train_Through"]
    valid_years = window["Valid_Years"]

    fold_train = individual_train.loc[
        individual_train["FE_YearStart"] <= cutoff
    ].copy()

    fold_valid = individual_train.loc[
        individual_train["FE_YearStart"].isin(valid_years)
    ].copy()

    y_fit = fold_train["Yield"].to_numpy(dtype=float)
    y_eval = fold_valid["Yield"].to_numpy(dtype=float)

    for model_name, (categorical_features, numeric_features) in feature_sets.items():
        columns = categorical_features + numeric_features

        preprocessor = ColumnTransformer(
            [
                ("categorical", clone(core_encoder), categorical_features),
                ("numeric", "passthrough", numeric_features),
            ],
            remainder="drop",
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42,
            )),
        ])

        pipeline.fit(fold_train[columns], y_fit)
        predictions = pipeline.predict(fold_valid[columns])
        metrics = evaluate_regression(y_eval, predictions)

        weather_ablation_results.append({
            "Train_Through": cutoff,
            "Validation_Years": ", ".join(map(str, valid_years)),
            "Model": model_name,
            "Train_N": len(fold_train),
            "Validation_N": len(fold_valid),
            **metrics,
        })

weather_ablation_df = pd.DataFrame(weather_ablation_results)

display(
    weather_ablation_df
    .sort_values(["Train_Through", "Model"])
    .round(3)
)

,Train_Through,Validation_Years,Model,Train_N,Validation_N,MAE,RMSE,R2
0,2018,"2019, 2020",Core + Location,21226,7249,448.306,672.961,0.798
1,2018,"2019, 2020",Core + Location + Weather,21226,7249,448.105,674.568,0.797
2,2020,"2021, 2022",Core + Location,28475,7309,422.856,898.177,0.693
3,2020,"2021, 2022",Core + Location + Weather,28475,7309,430.785,902.226,0.691


In [69]:
# Cell 65 — Compare feature configurations across rolling windows.

base_results = rolling_validation_df[
    ["Train_Through", "Validation_Years", "Model",
     "Validation_N", "MAE", "RMSE", "R2"]
].copy()

base_results["Feature_Configuration"] = base_results["Model"]
base_results = base_results.drop(columns="Model")

ablation_results = weather_ablation_df[
    ["Train_Through", "Validation_Years", "Model",
     "Validation_N", "MAE", "RMSE", "R2"]
].copy()

ablation_results["Feature_Configuration"] = ablation_results["Model"]
ablation_results = ablation_results.drop(columns="Model")

feature_configuration_comparison = pd.concat(
    [base_results, ablation_results],
    ignore_index=True,
)

feature_configuration_comparison = (
    feature_configuration_comparison
    .drop_duplicates(
        subset=["Train_Through", "Validation_Years", "Feature_Configuration"]
    )
    .sort_values(["Train_Through", "Feature_Configuration"])
    .reset_index(drop=True)
)

display(feature_configuration_comparison.round(3))

,Train_Through,Validation_Years,Validation_N,MAE,RMSE,R2,Feature_Configuration
0,2018,"2019, 2020",7249,448.306,672.961,0.798,Core + Location
1,2018,"2019, 2020",7249,448.105,674.568,0.797,Core + Location + Weather
2,2018,"2019, 2020",7249,608.446,882.634,0.652,Core + Weather
3,2018,"2019, 2020",7249,448.105,674.568,0.797,Core + Weather + State + District
4,2020,"2021, 2022",7309,422.856,898.177,0.693,Core + Location
5,2020,"2021, 2022",7309,430.785,902.226,0.691,Core + Location + Weather
6,2020,"2021, 2022",7309,578.419,1061.183,0.572,Core + Weather
7,2020,"2021, 2022",7309,430.785,902.226,0.691,Core + Weather + State + District


In [70]:
# Cell 66 — Rolling validation: Core + Location vs Crop–Season median.

baseline_validation_results = []

for window in rolling_windows:
    cutoff = window["Train_Through"]
    valid_years = window["Valid_Years"]

    fold_train = individual_train.loc[
        individual_train["FE_YearStart"] <= cutoff
    ].copy()

    fold_valid = individual_train.loc[
        individual_train["FE_YearStart"].isin(valid_years)
    ].copy()

    # Fit baseline using training data only.
    global_median = float(fold_train["Yield"].median())

    crop_season_medians = (
        fold_train.groupby(["Crop", "Season"])["Yield"]
        .median()
    )

    baseline_predictions = pd.MultiIndex.from_frame(
        fold_valid[["Crop", "Season"]]
    ).map(crop_season_medians)

    baseline_predictions = np.asarray(
        pd.Series(baseline_predictions).fillna(global_median),
        dtype=float,
    )

    # Fit the Core + Location model on the same training rows.
    location_columns = (
        core_categorical_features
        + ["State", "District"]
        + core_numeric_features
    )

    location_preprocessor = ColumnTransformer(
        [
            (
                "categorical",
                clone(core_encoder),
                core_categorical_features + ["State", "District"],
            ),
            ("numeric", "passthrough", core_numeric_features),
        ],
        remainder="drop",
    )

    location_pipeline = Pipeline([
        ("preprocessor", location_preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )),
    ])

    location_pipeline.fit(
        fold_train[location_columns],
        fold_train["Yield"].to_numpy(dtype=float),
    )

    location_predictions = location_pipeline.predict(
        fold_valid[location_columns]
    )

    y_true = fold_valid["Yield"].to_numpy(dtype=float)

    for model_name, predictions in [
        ("Crop–Season Median", baseline_predictions),
        ("Core + Location", location_predictions),
    ]:
        baseline_validation_results.append({
            "Train_Through": cutoff,
            "Validation_Years": ", ".join(map(str, valid_years)),
            "Model": model_name,
            "Train_N": len(fold_train),
            "Validation_N": len(fold_valid),
            **evaluate_regression(y_true, predictions),
        })

baseline_validation_df = pd.DataFrame(baseline_validation_results)

display(
    baseline_validation_df
    .sort_values(["Train_Through", "Model"])
    .round(3)
)

,Train_Through,Validation_Years,Model,Train_N,Validation_N,MAE,RMSE,R2
1,2018,"2019, 2020",Core + Location,21226,7249,448.306,672.961,0.798
0,2018,"2019, 2020",Crop–Season Median,21226,7249,721.227,1133.961,0.425
3,2020,"2021, 2022",Core + Location,28475,7309,422.856,898.177,0.693
2,2020,"2021, 2022",Crop–Season Median,28475,7309,709.335,1277.207,0.380


In [71]:
# Cell 67 — Model-selection summary for Notebook 16.

selection_rows = []

for cutoff, years in [(2018, "2019, 2020"), (2020, "2021, 2022")]:
    fold_results = baseline_validation_df[
        baseline_validation_df["Train_Through"].eq(cutoff)
    ].set_index("Model")

    baseline = fold_results.loc["Crop–Season Median"]
    candidate = fold_results.loc["Core + Location"]

    selection_rows.append({
        "Train_Through": cutoff,
        "Validation_Years": years,
        "Validation_N": int(candidate["Validation_N"]),
        "Baseline_MAE": baseline["MAE"],
        "Candidate_MAE": candidate["MAE"],
        "MAE_Improvement_Pct": (
            (baseline["MAE"] - candidate["MAE"])
            / baseline["MAE"] * 100
        ),
        "Baseline_RMSE": baseline["RMSE"],
        "Candidate_RMSE": candidate["RMSE"],
        "RMSE_Improvement_Pct": (
            (baseline["RMSE"] - candidate["RMSE"])
            / baseline["RMSE"] * 100
        ),
        "Baseline_R2": baseline["R2"],
        "Candidate_R2": candidate["R2"],
    })

model_selection_summary = pd.DataFrame(selection_rows)
display(model_selection_summary.round(3))

print("\nMODEL-SELECTION CONCLUSION")
print("Leading candidate: HistGradientBoosting + core features + State + District")
print("Weather: optional; no consistent incremental benefit after adding location.")
print("Baseline: Crop–Season median, fitted on each fold's training data only.")
print("Validation: two rolling time windows within the individual-crop cohort.")
print("Limitations: small unseen-location samples; weather mapping uncertainty;")
print("repeated inspection of the final holdout; no production-readiness claim.")

,Train_Through,Validation_Years,Validation_N,Baseline_MAE,Candidate_MAE,MAE_Improvement_Pct,Baseline_RMSE,Candidate_RMSE,RMSE_Improvement_Pct,Baseline_R2,Candidate_R2
0,2018,"2019, 2020",7249,721.227,448.306,37.841,1133.961,672.961,40.654,0.425,0.798
1,2020,"2021, 2022",7309,709.335,422.856,40.387,1277.207,898.177,29.676,0.380,0.693



MODEL-SELECTION CONCLUSION
Leading candidate: HistGradientBoosting + core features + State + District
Weather: optional; no consistent incremental benefit after adding location.
Baseline: Crop–Season median, fitted on each fold's training data only.
Validation: two rolling time windows within the individual-crop cohort.
Limitations: small unseen-location samples; weather mapping uncertainty;
repeated inspection of the final holdout; no production-readiness claim.


In [72]:
# Cell 68 — Clean, correctly labeled model-selection table.

final_validation_table = model_selection_summary[
    [
        "Train_Through",
        "Validation_Years",
        "Validation_N",
        "Baseline_MAE",
        "Candidate_MAE",
        "MAE_Improvement_Pct",
        "Baseline_RMSE",
        "Candidate_RMSE",
        "RMSE_Improvement_Pct",
        "Baseline_R2",
        "Candidate_R2",
    ]
].copy()

numeric_columns = [
    "Baseline_MAE",
    "Candidate_MAE",
    "MAE_Improvement_Pct",
    "Baseline_RMSE",
    "Candidate_RMSE",
    "RMSE_Improvement_Pct",
    "Baseline_R2",
    "Candidate_R2",
]

final_validation_table[numeric_columns] = (
    final_validation_table[numeric_columns].round(3)
)

display(final_validation_table)

print("\nCandidate: HistGradientBoosting + core features + State + District")
print("Weather features: excluded from the leading candidate for now.")
print("Evaluation: two rolling time-based validation windows.")
print("Scope: individual-crop candidate cohort with complete weather data.")
print("Caution: final holdout was repeatedly inspected; its metrics are diagnostic,")
print("not an untouched final estimate of generalization performance.")

,Train_Through,Validation_Years,Validation_N,Baseline_MAE,Candidate_MAE,MAE_Improvement_Pct,Baseline_RMSE,Candidate_RMSE,RMSE_Improvement_Pct,Baseline_R2,Candidate_R2
0,2018,"2019, 2020",7249,721.227,448.306,37.841,1133.961,672.961,40.654,0.425,0.798
1,2020,"2021, 2022",7309,709.335,422.856,40.387,1277.207,898.177,29.676,0.380,0.693



Candidate: HistGradientBoosting + core features + State + District
Weather features: excluded from the leading candidate for now.
Evaluation: two rolling time-based validation windows.
Scope: individual-crop candidate cohort with complete weather data.
Caution: final holdout was repeatedly inspected; its metrics are diagnostic,
not an untouched final estimate of generalization performance.


In [75]:
# Cell 69 — Record model-selection findings in Markdown.

from IPython.display import display, Markdown

report = f"""
## Model Selection — Yield Prediction

### Selected candidate
**HistGradientBoosting Regressor + core features + State + District**

Weather features are excluded from the leading candidate because adding them
did not consistently improve rolling-validation performance.

### Validation evidence

{final_validation_table.to_markdown(index=False, floatfmt=".3f")}

### Baseline comparison

{baseline_validation_df[
    ["Train_Through", "Validation_Years", "Model",
     "Validation_N", "MAE", "RMSE", "R2"]
].sort_values(["Train_Through", "Model"]).to_markdown(
    index=False, floatfmt=".3f"
)}

### Scope and limitations
- Evaluation cohort: individual-crop candidates with complete weather data.
- Validation uses two historical time-based windows.
- State and District features improved results in both windows.
- Unseen-location samples were small; performance on entirely unseen states
  has not been established.
- Weather mapping uses a calendar-year proxy; season alignment remains unverified.
- The final holdout was repeatedly inspected during diagnostics and is not
  an untouched final estimate of generalization performance.
- These results do not establish causal effects or production readiness.
"""

display(Markdown(report))


## Model Selection — Yield Prediction

### Selected candidate
**HistGradientBoosting Regressor + core features + State + District**

Weather features are excluded from the leading candidate because adding them
did not consistently improve rolling-validation performance.

### Validation evidence

|   Train_Through | Validation_Years   |   Validation_N |   Baseline_MAE |   Candidate_MAE |   MAE_Improvement_Pct |   Baseline_RMSE |   Candidate_RMSE |   RMSE_Improvement_Pct |   Baseline_R2 |   Candidate_R2 |
|----------------:|:-------------------|---------------:|---------------:|----------------:|----------------------:|----------------:|-----------------:|-----------------------:|--------------:|---------------:|
|            2018 | 2019, 2020         |           7249 |        721.227 |         448.306 |                37.841 |        1133.961 |          672.961 |                 40.654 |         0.425 |          0.798 |
|            2020 | 2021, 2022         |           7309 |        709.335 |         422.856 |                40.387 |        1277.207 |          898.177 |                 29.676 |         0.380 |          0.693 |

### Baseline comparison

|   Train_Through | Validation_Years   | Model              |   Validation_N |     MAE |     RMSE |    R2 |
|----------------:|:-------------------|:-------------------|---------------:|--------:|---------:|------:|
|            2018 | 2019, 2020         | Core + Location    |           7249 | 448.306 |  672.961 | 0.798 |
|            2018 | 2019, 2020         | Crop–Season Median |           7249 | 721.227 | 1133.961 | 0.425 |
|            2020 | 2021, 2022         | Core + Location    |           7309 | 422.856 |  898.177 | 0.693 |
|            2020 | 2021, 2022         | Crop–Season Median |           7309 | 709.335 | 1277.207 | 0.380 |

### Scope and limitations
- Evaluation cohort: individual-crop candidates with complete weather data.
- Validation uses two historical time-based windows.
- State and District features improved results in both windows.
- Unseen-location samples were small; performance on entirely unseen states
  has not been established.
- Weather mapping uses a calendar-year proxy; season alignment remains unverified.
- The final holdout was repeatedly inspected during diagnostics and is not
  an untouched final estimate of generalization performance.
- These results do not establish causal effects or production readiness.


In [76]:
# Cell 70 — Final candidate data-integrity checks (no model fitting).

candidate_categorical_features = (
    core_categorical_features + ["State", "District"]
)
candidate_numeric_features = core_numeric_features
candidate_features = (
    candidate_categorical_features + candidate_numeric_features
)

candidate_train = individual_train.copy()
candidate_holdout = individual_holdout.copy()

print("Candidate: HistGradientBoosting + core features + State + District")
print("\nDataset sizes")
print("Training rows:", len(candidate_train))
print("Holdout rows:", len(candidate_holdout))
print("Training years:", sorted(candidate_train["FE_YearStart"].unique()))
print("Holdout years:", sorted(candidate_holdout["FE_YearStart"].unique()))

print("\nFeature and target integrity")
print("Missing candidate training features:",
      int(candidate_train[candidate_features].isna().sum().sum()))
print("Missing candidate holdout features:",
      int(candidate_holdout[candidate_features].isna().sum().sum()))
print("Non-finite training targets:",
      int((~np.isfinite(candidate_train["Yield"].to_numpy(dtype=float))).sum()))
print("Non-finite holdout targets:",
      int((~np.isfinite(candidate_holdout["Yield"].to_numpy(dtype=float))).sum()))
print("Non-positive training targets:",
      int((candidate_train["Yield"] <= 0).sum()))
print("Non-positive holdout targets:",
      int((candidate_holdout["Yield"] <= 0).sum()))

train_location_keys = set(make_location_key(candidate_train).dropna())
holdout_location_keys = set(make_location_key(candidate_holdout).dropna())

print("\nLocation coverage")
print("Training State–District combinations:", len(train_location_keys))
print("Holdout State–District combinations:", len(holdout_location_keys))
print("Holdout combinations unseen in training:",
      len(holdout_location_keys - train_location_keys))

assert candidate_train[candidate_features].notna().all().all()
assert candidate_holdout[candidate_features].notna().all().all()
assert np.isfinite(candidate_train["Yield"].to_numpy(dtype=float)).all()
assert np.isfinite(candidate_holdout["Yield"].to_numpy(dtype=float)).all()
assert (candidate_train["Yield"] > 0).all()
assert (candidate_holdout["Yield"] > 0).all()
assert set(candidate_train["FE_YearStart"]).isdisjoint(
    set(candidate_holdout["FE_YearStart"])
)

print("\nAll required data-integrity assertions passed.")

Candidate: HistGradientBoosting + core features + State + District

Dataset sizes
Training rows: 35784
Holdout rows: 7369
Training years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Holdout years: [np.int64(2023), np.int64(2024)]

Feature and target integrity
Missing candidate training features: 0
Missing candidate holdout features: 0
Non-finite training targets: 0
Non-finite holdout targets: 0
Non-positive training targets: 0
Non-positive holdout targets: 0

Location coverage
Training State–District combinations: 624
Holdout State–District combinations: 635
Holdout combinations unseen in training: 11

All required data-integrity assertions passed.


In [77]:
# Cell 71 — Fit the selected HistGradientBoosting candidate.

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

final_categorical_features = (
    core_categorical_features + ["State", "District"]
)
final_numeric_features = core_numeric_features
final_features = final_categorical_features + final_numeric_features

final_preprocessor = ColumnTransformer(
    [
        (
            "categorical",
            clone(core_encoder),
            final_categorical_features,
        ),
        ("numeric", "passthrough", final_numeric_features),
    ],
    remainder="drop",
)

final_y_train = candidate_train["Yield"].to_numpy(dtype=float)

final_yield_model = Pipeline([
    ("preprocessor", final_preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )),
])

final_yield_model.fit(
    candidate_train[final_features],
    final_y_train,
)

print("Final candidate fitted successfully.")
print("Training rows:", len(candidate_train))
print("Features:", final_features)
print("Training years:", int(candidate_train["FE_YearStart"].min()),
      "to", int(candidate_train["FE_YearStart"].max()))
print("Holdout evaluated: No")

Final candidate fitted successfully.
Training rows: 35784
Features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Training years: 2013 to 2022
Holdout evaluated: No


In [78]:
# Cell 72 — Save the fitted model and its metadata.

import json
import joblib
from pathlib import Path
from datetime import datetime, timezone

model_dir = PROJECT_ROOT / "models" / "yield_prediction"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "individual_crop_hgb_location.joblib"
metadata_path = model_dir / "individual_crop_hgb_location_metadata.json"

joblib.dump(final_yield_model, model_path)

metadata = {
    "model_name": "HistGradientBoostingRegressor",
    "project": "BharatCrop Intelligence",
    "candidate_scope": "Individual Crop Candidate cohort",
    "features": final_features,
    "categorical_features": final_categorical_features,
    "numeric_features": final_numeric_features,
    "target": "Yield",
    "training_rows": int(len(candidate_train)),
    "training_year_start": int(candidate_train["FE_YearStart"].min()),
    "training_year_end": int(candidate_train["FE_YearStart"].max()),
    "holdout_rows": int(len(candidate_holdout)),
    "holdout_evaluated_during_final_fit": False,
    "validation_windows": [
        {"train_through": 2018, "validation_years": [2019, 2020]},
        {"train_through": 2020, "validation_years": [2021, 2022]},
    ],
    "validation_metrics": {
        "2019-2020": {"MAE": 448.306, "RMSE": 672.961, "R2": 0.798},
        "2021-2022": {"MAE": 422.856, "RMSE": 898.177, "R2": 0.693},
    },
    "limitations": [
        "Final holdout was repeatedly inspected during diagnostics.",
        "Unseen-location validation samples were small.",
        "Weather features were excluded from the selected candidate.",
        "Model performance does not establish causal effects or production readiness.",
    ],
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
}

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Model saved:", model_path.resolve())
print("Metadata saved:", metadata_path.resolve())
print("Model file exists:", model_path.is_file())
print("Metadata file exists:", metadata_path.is_file())
print("Model size (MB):", round(model_path.stat().st_size / (1024 ** 2), 2))

Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location.joblib
Metadata saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location_metadata.json
Model file exists: True
Metadata file exists: True
Model size (MB): 0.75


In [79]:
# Cell 73 — Reload saved artifacts and run a training-row smoke test.

import joblib
import json
import numpy as np

reloaded_model = joblib.load(model_path)

with metadata_path.open("r", encoding="utf-8") as f:
    reloaded_metadata = json.load(f)

sample_X = candidate_train[final_features].iloc[[0]]
sample_y = float(candidate_train["Yield"].iloc[0])

sample_prediction = float(reloaded_model.predict(sample_X)[0])

assert np.isfinite(sample_prediction), "Prediction is not finite."
assert reloaded_metadata["features"] == final_features
assert reloaded_metadata["target"] == "Yield"

print("Reload successful:", reloaded_model is not None)
print("Metadata loaded:", reloaded_metadata["model_name"])
print("Sample crop:", sample_X["Crop"].iloc[0])
print("Sample state/district:",
      sample_X["State"].iloc[0], "/", sample_X["District"].iloc[0])
print("Actual training yield:", round(sample_y, 3))
print("Predicted yield:", round(sample_prediction, 3))
print("Absolute difference:", round(abs(sample_prediction - sample_y), 3))
print("Finite prediction:", np.isfinite(sample_prediction))
print("Artifact smoke test passed.")

Reload successful: True
Metadata loaded: HistGradientBoostingRegressor
Sample crop: Rice
Sample state/district: Andaman And Nicobar Islands / North And Middle Andaman
Actual training yield: 2010.0
Predicted yield: 2001.558
Absolute difference: 8.442
Finite prediction: True
Artifact smoke test passed.


In [80]:
# Cell 74 — Verify predictions are identical after saving and reloading.

sample_X = candidate_train[final_features].head(100).copy()

original_predictions = final_yield_model.predict(sample_X)
reloaded_predictions = reloaded_model.predict(sample_X)

max_abs_difference = float(
    np.max(np.abs(original_predictions - reloaded_predictions))
)

assert np.isfinite(reloaded_predictions).all()
assert np.allclose(
    original_predictions,
    reloaded_predictions,
    rtol=0,
    atol=1e-10,
), "Predictions changed after reloading the model."

print("Rows tested:", len(sample_X))
print("Maximum absolute prediction difference:", max_abs_difference)
print("All predictions finite:", bool(np.isfinite(reloaded_predictions).all()))
print("Serialization reproducibility test passed.")

Rows tested: 100
Maximum absolute prediction difference: 0.0
All predictions finite: True
Serialization reproducibility test passed.


In [81]:
# Cell 75 — Production-readiness audit (read-only).

from pathlib import Path
import json
import sys
import sklearn
import joblib

print("=== RUNTIME ===")
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

print("\n=== PROJECT STRUCTURE ===")
important_paths = [
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "tests",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "pyproject.toml",
    PROJECT_ROOT / "Dockerfile",
    PROJECT_ROOT / "docker-compose.yml",
    PROJECT_ROOT / ".gitignore",
]

for path in important_paths:
    print(f"{'EXISTS' if path.exists() else 'MISSING':7} {path.relative_to(PROJECT_ROOT)}")

print("\n=== SAVED MODEL METADATA ===")
with metadata_path.open("r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

for key in [
    "model_name",
    "candidate_scope",
    "features",
    "target",
    "training_rows",
    "training_year_start",
    "training_year_end",
    "holdout_rows",
    "holdout_evaluated_during_final_fit",
    "limitations",
]:
    print(f"{key}: {saved_metadata.get(key)}")

print("\n=== MODEL INTERFACE ===")
print("Expected features:", final_features)
print("Pipeline steps:", list(reloaded_model.named_steps.keys()))
print("Artifact exists:", model_path.is_file())
print("Metadata exists:", metadata_path.is_file())

print("\n=== TOP-LEVEL FILES (LIMIT 60) ===")
top_level_files = sorted(
    p.name for p in PROJECT_ROOT.iterdir()
    if p.is_file()
)
print(top_level_files[:60])

=== RUNTIME ===
Python: 3.13.5
scikit-learn: 1.8.0
joblib: 1.5.2

=== PROJECT STRUCTURE ===
EXISTS  models
EXISTS  data
EXISTS  notebooks
MISSING src
MISSING app
MISSING tests
MISSING requirements.txt
MISSING pyproject.toml
MISSING Dockerfile
MISSING docker-compose.yml
MISSING .gitignore

=== SAVED MODEL METADATA ===
model_name: HistGradientBoostingRegressor
candidate_scope: Individual Crop Candidate cohort
features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
target: Yield
training_rows: 35784
training_year_start: 2013
training_year_end: 2022
holdout_rows: 7369
holdout_evaluated_during_final_fit: False
limitations: ['Final holdout was repeatedly inspected during diagnostics.', 'Unseen-location validation samples were small.', 'Weather features were excluded from the selected candidate.', 'Model performance does not establish causal effects or production readiness.']

=== MODEL INTERFACE ===
Expected features: ['Crop', 'Season', 'Crop_Category', 'State', 'D

In [82]:
# Cell 76 — Define the model's input contract.

import pandas as pd
import numpy as np

MODEL_INPUT_CONTRACT = {
    "required_columns": list(final_features),
    "categorical_columns": list(final_categorical_features),
    "numeric_columns": list(final_numeric_features),
    "target": "Yield",
    "prediction_unit": "District × Crop × Season × Year",
    "model_scope": "Individual Crop Candidate cohort",
}

def validate_yield_model_inputs(X, expected_features):
    """Validate the structure and basic validity of model input data."""
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if X.empty:
        raise ValueError("Input contains no rows.")

    if X.columns.duplicated().any():
        duplicates = X.columns[X.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate input columns: {duplicates}")

    missing = sorted(set(expected_features) - set(X.columns))
    if missing:
        raise ValueError(f"Missing required features: {missing}")

    X_checked = X.loc[:, expected_features].copy()

    for col in MODEL_INPUT_CONTRACT["categorical_columns"]:
        if X_checked[col].isna().any():
            raise ValueError(f"Categorical feature '{col}' contains missing values.")

        X_checked[col] = X_checked[col].astype(str).str.strip()

        if X_checked[col].eq("").any():
            raise ValueError(f"Categorical feature '{col}' contains blank values.")

    for col in MODEL_INPUT_CONTRACT["numeric_columns"]:
        converted = pd.to_numeric(X_checked[col], errors="coerce")
        invalid = ~np.isfinite(converted.to_numpy(dtype=float))

        if invalid.any():
            raise ValueError(
                f"Numeric feature '{col}' contains "
                f"{int(invalid.sum())} missing or non-finite values."
            )

        X_checked[col] = converted.astype(float)

    return X_checked


# Test the contract using eligible training inputs.
contract_sample = validate_yield_model_inputs(
    candidate_train[final_features].head(10),
    final_features,
)

assert contract_sample.shape == (10, len(final_features))
assert contract_sample.columns.tolist() == final_features
assert np.isfinite(
    contract_sample[MODEL_INPUT_CONTRACT["numeric_columns"]].to_numpy(
        dtype=float
    )
).all()

print("Validated sample shape:", contract_sample.shape)
print("Column order:", contract_sample.columns.tolist())
print("Missing values:", int(contract_sample.isna().sum().sum()))
print("Input contract smoke test passed.")

Validated sample shape: (10, 6)
Column order: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Missing values: 0
Input contract smoke test passed.


In [83]:
# Cell 77 — Verify that invalid model inputs are rejected.

negative_tests = {}

valid_sample = candidate_train[final_features].head(3).copy()

# Test 1: Missing required feature
try:
    validate_yield_model_inputs(
        valid_sample.drop(columns=["District"]),
        final_features,
    )
    negative_tests["missing_column"] = "FAILED: invalid input accepted"
except ValueError:
    negative_tests["missing_column"] = "PASSED: rejected"

# Test 2: Missing categorical value
bad_category = valid_sample.copy()
bad_category.loc[bad_category.index[0], "Crop"] = np.nan

try:
    validate_yield_model_inputs(bad_category, final_features)
    negative_tests["missing_category"] = "FAILED: invalid input accepted"
except ValueError:
    negative_tests["missing_category"] = "PASSED: rejected"

# Test 3: Non-numeric year
bad_year = valid_sample.copy()
bad_year["FE_YearStart"] = "not-a-year"

try:
    validate_yield_model_inputs(bad_year, final_features)
    negative_tests["invalid_year"] = "FAILED: invalid input accepted"
except ValueError:
    negative_tests["invalid_year"] = "PASSED: rejected"

# Test 4: Infinite year
bad_infinity = valid_sample.copy()
bad_infinity["FE_YearStart"] = np.inf

try:
    validate_yield_model_inputs(bad_infinity, final_features)
    negative_tests["infinite_year"] = "FAILED: invalid input accepted"
except ValueError:
    negative_tests["infinite_year"] = "PASSED: rejected"

for test_name, result in negative_tests.items():
    print(f"{test_name}: {result}")

assert all(result.startswith("PASSED") for result in negative_tests.values()), (
    "At least one invalid-input test failed."
)

print("\nAll negative input-validation tests passed.")

missing_column: PASSED: rejected
missing_category: PASSED: rejected
invalid_year: PASSED: rejected
infinite_year: PASSED: rejected

All negative input-validation tests passed.


In [84]:
# Cell 78 — Verify inference with unseen categorical values.

unseen_sample = candidate_train[final_features].iloc[[0]].copy()

# Use synthetic category names that are not present in training.
unseen_sample["Crop"] = "__UNSEEN_TEST_CROP__"
unseen_sample["Crop_Category"] = "__UNSEEN_TEST_CATEGORY__"
unseen_sample["State"] = "__UNSEEN_TEST_STATE__"
unseen_sample["District"] = "__UNSEEN_TEST_DISTRICT__"

validated_unseen_sample = validate_yield_model_inputs(
    unseen_sample,
    final_features,
)

unseen_prediction = float(
    reloaded_model.predict(validated_unseen_sample)[0]
)

print("Synthetic unseen categories supplied:")
print(
    validated_unseen_sample[
        ["Crop", "Crop_Category", "State", "District"]
    ].iloc[0].to_dict()
)
print("Inference completed:", True)
print("Prediction is finite:", bool(np.isfinite(unseen_prediction)))
print("Predicted yield:", round(unseen_prediction, 3))
print(
    "Warning: this is a technical robustness test, "
    "not a valid agricultural prediction."
)

assert np.isfinite(unseen_prediction), "Unseen-category prediction is not finite."
print("Unseen-category smoke test passed.")

Synthetic unseen categories supplied:
{'Crop': '__UNSEEN_TEST_CROP__', 'Crop_Category': '__UNSEEN_TEST_CATEGORY__', 'State': '__UNSEEN_TEST_STATE__', 'District': '__UNSEEN_TEST_DISTRICT__'}
Inference completed: True
Prediction is finite: True
Predicted yield: 951.429
Unseen-category smoke test passed.


In [85]:
# Cell 79 — Check prediction validity and summarize prediction ranges.

sanity_X = validate_yield_model_inputs(
    candidate_train[final_features].head(1000),
    final_features,
)

sanity_predictions = reloaded_model.predict(sanity_X)
sanity_actuals = candidate_train["Yield"].iloc[:len(sanity_X)].to_numpy(dtype=float)

assert len(sanity_predictions) == len(sanity_actuals)
assert np.isfinite(sanity_predictions).all(), (
    "Model produced non-finite predictions."
)

prediction_summary = pd.Series(sanity_predictions, name="Predicted_Yield").describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
)

print("Rows checked:", len(sanity_predictions))
print("Non-finite predictions:", int((~np.isfinite(sanity_predictions)).sum()))
print("Negative predictions:", int((sanity_predictions < 0).sum()))
print("Zero predictions:", int((sanity_predictions == 0).sum()))
print("\nPrediction distribution:")
display(prediction_summary.to_frame())

print("\nActual yield distribution for the same rows:")
display(
    pd.Series(sanity_actuals, name="Actual_Yield").describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame()
)

print(
    "\nNote: this is a training-sample sanity check, "
    "not an independent performance evaluation."
)

Rows checked: 1000
Non-finite predictions: 0
Negative predictions: 0
Zero predictions: 0

Prediction distribution:


,Predicted_Yield
count,1000.000000
mean,1906.086265
std,1148.671291
min,347.153306
1%,481.039087
5%,600.639672
50%,1721.393366
95%,4139.654565
99%,6549.141368
max,7825.999702



Actual yield distribution for the same rows:


,Actual_Yield
count,1000.000000
mean,1961.843000
std,1399.541708
min,129.000000
1%,292.960000
5%,500.000000
50%,1535.500000
95%,4819.000000
99%,7454.000000
max,8620.000000



Note: this is a training-sample sanity check, not an independent performance evaluation.


In [86]:
# Cell 80 — Audit prediction range and training residuals.

diagnostic_X = validate_yield_model_inputs(
    candidate_train[final_features],
    final_features,
)

diagnostic_actual = candidate_train["Yield"].to_numpy(dtype=float)
diagnostic_predicted = reloaded_model.predict(diagnostic_X)

diagnostic_residual = diagnostic_actual - diagnostic_predicted
diagnostic_abs_error = np.abs(diagnostic_residual)

assert np.isfinite(diagnostic_predicted).all()
assert np.isfinite(diagnostic_residual).all()

print("Rows evaluated:", len(diagnostic_actual))
print("Negative predictions:", int((diagnostic_predicted < 0).sum()))
print("Zero predictions:", int((diagnostic_predicted == 0).sum()))

print("\nPrediction distribution:")
display(
    pd.Series(diagnostic_predicted, name="Predicted_Yield").describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame()
)

print("\nAbsolute training-error distribution:")
display(
    pd.Series(diagnostic_abs_error, name="Absolute_Error").describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame()
)

print("\nLargest absolute training errors:")
error_review = candidate_train[
    ["Crop", "Season", "State", "District", "FE_YearStart", "Yield"]
].copy()

error_review["Predicted_Yield"] = diagnostic_predicted
error_review["Residual_Actual_Minus_Predicted"] = diagnostic_residual
error_review["Absolute_Error"] = diagnostic_abs_error

display(
    error_review.nlargest(10, "Absolute_Error").reset_index(drop=True)
)

print(
    "\nInterpretation: these are in-sample diagnostics. "
    "Use historical validation windows—not training errors—to judge generalization."
)

Rows evaluated: 35784
Negative predictions: 5
Zero predictions: 0

Prediction distribution:


,Predicted_Yield
count,35784.000000
mean,1941.010560
std,1265.968598
min,-108.824537
1%,346.032703
5%,544.983292
50%,1703.978789
95%,4147.471795
99%,6833.883820
max,8567.444629



Absolute training-error distribution:


,Absolute_Error
count,35784.000000
mean,397.833037
std,508.912825
min,0.004694
50%,276.576030
75%,516.918307
90%,865.254883
95%,1176.892242
99%,2068.938789
max,40243.746220



Largest absolute training errors:


,Crop,Season,State,District,FE_YearStart,Yield,Predicted_Yield,Residual_Actual_Minus_Predicted,Absolute_Error
0,Rice,Kharif,Goa,North Goa,2022,44815.0,4571.253780,40243.746220,40243.746220
1,Rice,Kharif,Goa,South Goa,2022,39221.0,4571.253780,34649.746220,34649.746220
2,Rice,Kharif,Himachal Pradesh,Bilaspur,2020,12880.0,2247.761529,10632.238471,10632.238471
3,Maize,Rabi,Gujarat,Sabar Kantha,2017,9500.0,3136.241308,6363.758692,6363.758692
4,Maize,Summer,West Bengal,Dakshin Dinajpur,2022,12525.0,6383.588110,6141.411890,6141.411890
5,Jowar,Rabi,Andhra Pradesh,Bapatla,2022,8293.0,2377.773507,5915.226493,5915.226493
6,Maize,Kharif,Tamil Nadu,Viluppuram,2016,10763.0,5147.121384,5615.878616,5615.878616
7,Maize,Rabi,Maharashtra,Pune,2016,7888.0,2819.058679,5068.941321,5068.941321
8,Maize,Rabi,Telangana,Adilabad,2017,10982.0,6022.749606,4959.250394,4959.250394
9,Maize,Rabi,Bihar,Khagaria,2016,9915.0,5130.860124,4784.139876,4784.139876



Interpretation: these are in-sample diagnostics. Use historical validation windows—not training errors—to judge generalization.


In [87]:
# Cell 81 — Audit extreme yields and their prediction errors.

extreme_threshold = 10000

extreme_mask = diagnostic_actual >= extreme_threshold

extreme_summary = {
    "threshold": extreme_threshold,
    "extreme_rows": int(extreme_mask.sum()),
    "share_of_training_rows_pct": round(
        100 * extreme_mask.mean(), 3
    ),
    "mean_actual_yield_extreme": (
        round(float(diagnostic_actual[extreme_mask].mean()), 3)
        if extreme_mask.any() else None
    ),
    "mean_absolute_error_extreme": (
        round(float(diagnostic_abs_error[extreme_mask].mean()), 3)
        if extreme_mask.any() else None
    ),
    "mean_absolute_error_below_threshold": round(
        float(diagnostic_abs_error[~extreme_mask].mean()), 3
    ),
}

print("Extreme-yield summary:")
display(pd.Series(extreme_summary, name="Value").to_frame())

extreme_review = candidate_train[
    ["Crop", "Season", "State", "District", "FE_YearStart", "Area", "Production", "Yield"]
].copy()

extreme_review["Predicted_Yield"] = diagnostic_predicted
extreme_review["Absolute_Error"] = diagnostic_abs_error

print(f"\nTraining rows with Yield >= {extreme_threshold}:")
display(
    extreme_review.loc[extreme_mask]
    .sort_values("Yield", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("\nExtreme-yield counts by crop:")
display(
    candidate_train.loc[extreme_mask]
    .groupby("Crop")
    .agg(
        Rows=("Yield", "size"),
        Min_Yield=("Yield", "min"),
        Median_Yield=("Yield", "median"),
        Max_Yield=("Yield", "max"),
    )
    .sort_values("Rows", ascending=False)
)

Extreme-yield summary:


,Value
threshold,10000.000
extreme_rows,42.000
share_of_training_rows_pct,0.117
mean_actual_yield_extreme,12293.262
mean_absolute_error_extreme,5084.165
mean_absolute_error_below_threshold,392.326



Training rows with Yield >= 10000:


,Crop,Season,State,District,FE_YearStart,Area,Production,Yield,Predicted_Yield,Absolute_Error
0,Rice,Kharif,Goa,North Goa,2022,0.01,0.47,44815.0,4571.253780,40243.746220
1,Rice,Kharif,Goa,South Goa,2022,0.01,0.47,39221.0,4571.253780,34649.746220
2,Rice,Kharif,Himachal Pradesh,Bilaspur,2020,0.01,0.12,12880.0,2247.761529,10632.238471
3,Maize,Rabi,Andhra Pradesh,Guntur,2022,0.26,3.24,12526.0,8173.550514,4352.449486
4,Maize,Summer,West Bengal,Dakshin Dinajpur,2022,0.05,0.58,12525.0,6383.588110,6141.411890
5,Maize,Rabi,Bihar,Katihar,2016,0.45,5.34,11986.0,8422.266923,3563.733077
6,Maize,Kharif,Tamil Nadu,Tiruppur,2015,0.06,0.71,11978.0,8100.395088,3877.604912
7,Maize,Rabi,Andhra Pradesh,Eluru,2022,0.29,3.50,11933.0,7199.436163,4733.563837
8,Maize,Rabi,Andhra Pradesh,Bapatla,2022,0.40,4.68,11620.0,7199.436163,4420.563837
9,Maize,Kharif,Tamil Nadu,Perambalur,2017,0.37,4.26,11501.0,7888.391167,3612.608833



Extreme-yield counts by crop:


,Rows,Min_Yield,Median_Yield,Max_Yield
Crop,,,,
Maize,39,10047.0,10550.0,12526.0
Rice,3,12880.0,39221.0,44815.0


In [88]:
# Cell 82 — Investigate consistency between Yield, Production and Area.

ratio_check = candidate_train[
    ["Crop", "Season", "State", "District", "FE_YearStart",
     "Area", "Production", "Yield"]
].copy()

ratio_check["Calculated_Yield"] = (
    ratio_check["Production"] / ratio_check["Area"]
)

ratio_check["Absolute_Ratio_Difference"] = (
    ratio_check["Yield"] - ratio_check["Calculated_Yield"]
).abs()

ratio_check["Relative_Ratio_Difference_Pct"] = (
    100
    * ratio_check["Absolute_Ratio_Difference"]
    / ratio_check["Yield"].abs().replace(0, np.nan)
)

print("Overall ratio-consistency diagnostics:")
display(
    ratio_check[
        ["Absolute_Ratio_Difference", "Relative_Ratio_Difference_Pct"]
    ].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
)

print("\nRecords with Yield >= 10000:")
display(
    ratio_check.loc[ratio_check["Yield"] >= 10000]
    .sort_values("Yield", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("\nRecords with the largest relative ratio differences:")
display(
    ratio_check.nlargest(15, "Relative_Ratio_Difference_Pct")
    .reset_index(drop=True)
)

print(
    "\nImportant: this ratio check is only meaningful if Area, Production, "
    "and Yield use compatible units and definitions. Confirm the source "
    "metadata before treating a discrepancy as an error."
)

Overall ratio-consistency diagnostics:


,count,mean,std,min,50%,90%,95%,99%,max
Absolute_Ratio_Difference,35784.0,1940.683783,1470.724680,25.969231,1549.585714,3769.127273,4495.000000,7349.235455,44768.000000
Relative_Ratio_Difference_Pct,35784.0,99.898398,0.019531,99.715909,99.900000,99.915201,99.926308,99.946357,99.965422



Records with Yield >= 10000:


,Crop,Season,State,District,FE_YearStart,Area,Production,Yield,Calculated_Yield,Absolute_Ratio_Difference,Relative_Ratio_Difference_Pct
0,Rice,Kharif,Goa,North Goa,2022,0.01,0.47,44815.0,47.000000,44768.000000,99.895124
1,Rice,Kharif,Goa,South Goa,2022,0.01,0.47,39221.0,47.000000,39174.000000,99.880166
2,Rice,Kharif,Himachal Pradesh,Bilaspur,2020,0.01,0.12,12880.0,12.000000,12868.000000,99.906832
3,Maize,Rabi,Andhra Pradesh,Guntur,2022,0.26,3.24,12526.0,12.461538,12513.538462,99.900515
4,Maize,Summer,West Bengal,Dakshin Dinajpur,2022,0.05,0.58,12525.0,11.600000,12513.400000,99.907385
5,Maize,Rabi,Bihar,Katihar,2016,0.45,5.34,11986.0,11.866667,11974.133333,99.900996
6,Maize,Kharif,Tamil Nadu,Tiruppur,2015,0.06,0.71,11978.0,11.833333,11966.166667,99.901208
7,Maize,Rabi,Andhra Pradesh,Eluru,2022,0.29,3.50,11933.0,12.068966,11920.931034,99.898861
8,Maize,Rabi,Andhra Pradesh,Bapatla,2022,0.40,4.68,11620.0,11.700000,11608.300000,99.899312
9,Maize,Kharif,Tamil Nadu,Perambalur,2017,0.37,4.26,11501.0,11.513514,11489.486486,99.899891



Records with the largest relative ratio differences:


,Crop,Season,State,District,FE_YearStart,Area,Production,Yield,Calculated_Yield,Absolute_Ratio_Difference,Relative_Ratio_Difference_Pct
0,Maize,Summer,Uttar Pradesh,Firozabad,2020,0.01,0.01,2892.0,1.0,2891.0,99.965422
1,Maize,Rabi,Bihar,Munger,2014,0.01,0.01,2792.0,1.0,2791.0,99.964183
2,Maize,Rabi,Bihar,Jamui,2022,0.01,0.01,2773.0,1.0,2772.0,99.963938
3,Rice,Summer,Uttar Pradesh,Balrampur,2019,0.01,0.01,2772.0,1.0,2771.0,99.963925
4,Maize,Summer,Gujarat,Arvalli,2022,0.01,0.01,2751.0,1.0,2750.0,99.963650
5,Maize,Summer,Uttar Pradesh,Firozabad,2021,0.01,0.01,2728.0,1.0,2727.0,99.963343
6,Maize,Summer,Uttar Pradesh,Etawah,2021,0.01,0.01,2726.0,1.0,2725.0,99.963316
7,Rice,Rabi,Nagaland,Peren,2015,0.01,0.01,2698.0,1.0,2697.0,99.962936
8,Bajra,Kharif,Tamil Nadu,Krishnagiri,2015,0.01,0.01,2693.0,1.0,2692.0,99.962867
9,Maize,Kharif,Uttarakhand,Champawat,2021,0.01,0.01,2689.0,1.0,2688.0,99.962811



Important: this ratio check is only meaningful if Area, Production, and Yield use compatible units and definitions. Confirm the source metadata before treating a discrepancy as an error.


In [89]:
# Cell 83 — Locate APY source files and inspect their schemas.

from pathlib import Path

print("Project root:", PROJECT_ROOT)

search_roots = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
]

for root in search_roots:
    print(f"\n--- {root} ---")

    if not root.exists():
        print("Directory does not exist.")
        continue

    candidates = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".csv", ".xlsx", ".xls", ".parquet"}
        and any(
            token in p.name.lower()
            for token in ["apy", "agricultural", "crop", "production", "historical"]
        )
    )

    for path in candidates[:60]:
        print(path.relative_to(PROJECT_ROOT))

print("\n--- Integrated dataset column names ---")
print(integrated_df.columns.tolist())

print("\n--- Original APY-related columns and sample records ---")
apy_columns = [
    c for c in integrated_df.columns
    if any(token in c.lower() for token in [
        "crop", "season", "state", "district",
        "area", "production", "yield", "unit", "year"
    ])
]

display(integrated_df[apy_columns].head(10))

Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence

--- C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\raw ---
data\raw\weather_pilot_2013_2024\APY-GEO-0005_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0006_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0055_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0056_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0320_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0321_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0491_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0492_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0561_weather_2013_2024.csv
data\raw\weather_pilot_2013_2024\APY-GEO-0562_weather_2013_2024.csv

--- C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed ---
data\processed\.ipynb_checkpoints\historical_food_grains_apy_2013-14_to_2024-25-checkpoint.csv
d

,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction,start_year,weather_proxy_calendar_year,QC_Long_District_Grid_Distance,weather_full_season_alignment_verified
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,True,2013,2013.0,False,False
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,True,2013,2013.0,False,False
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,True,2013,2013.0,False,False
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,True,2013,2013.0,False,False
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,True,2013,2013.0,False,False
5,Andaman And Nicobar Islands,Nicobars,Moong,Rabi,Individual Crop Candidate,2013-14,NaN,NaN,NaN,False,False,False,2013,2013.0,False,False
6,Andaman And Nicobar Islands,Nicobars,Moong,Total,Individual Crop Candidate,2013-14,NaN,NaN,NaN,False,False,False,2013,2013.0,False,False
7,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,True,2013,2013.0,False,False
8,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,True,2013,2013.0,False,False
9,Andaman And Nicobar Islands,Nicobars,Other Pulses,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,3865.0,False,False,True,2013,2013.0,False,False


In [90]:
# Cell 84 — Inspect APY source schemas and measurement values.

source_paths = {
    "cleaned_flagged": (
        PROJECT_ROOT
        / "data" / "processed"
        / "historical_food_grains_apy_cleaned_flagged.csv"
    ),
    "long_format": (
        PROJECT_ROOT
        / "data" / "processed"
        / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
    ),
    "original_processed": (
        PROJECT_ROOT
        / "data" / "processed"
        / "historical_food_grains_apy_2013-14_to_2024-25.csv"
    ),
}

for label, path in source_paths.items():
    print(f"\n{'=' * 80}")
    print(f"{label}: {path.name}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    source = pd.read_csv(path, low_memory=False)

    print("Shape:", source.shape)
    print("Columns:", source.columns.tolist())

    relevant_columns = [
        col for col in source.columns
        if any(
            term in col.lower()
            for term in [
                "state", "district", "crop", "season", "year",
                "area", "production", "yield", "unit", "source"
            ]
        )
    ]

    display(source[relevant_columns].head(5))

    measurement_columns = [
        col for col in source.columns
        if any(term in col.lower() for term in ["area", "production", "yield"])
    ]

    if measurement_columns:
        print("Measurement-column summaries:")
        display(source[measurement_columns].describe().T)


cleaned_flagged: historical_food_grains_apy_cleaned_flagged.csv
Exists: True
Shape: (31356, 89)
Columns: ['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Production-2013-14', 'Yield-2013-14', 'Area-2014-15', 'Production-2014-15', 'Yield-2014-15', 'Area-2015-16', 'Production-2015-16', 'Yield-2015-16', 'Area-2016-17', 'Production-2016-17', 'Yield-2016-17', 'Area-2017-18', 'Production-2017-18', 'Yield-2017-18', 'Area-2018-19', 'Production-2018-19', 'Yield-2018-19', 'Area-2019-20', 'Production-2019-20', 'Yield-2019-20', 'Area-2020-21', 'Production-2020-21', 'Yield-2020-21', 'Area-2021-22', 'Production-2021-22', 'Yield-2021-22', 'Area-2022-23', 'Production-2022-23', 'Yield-2022-23', 'Area-2023-24', 'Production-2023-24', 'Yield-2023-24', 'Area-2024-25', 'Production-2024-25', 'Yield-2024-25', 'Crop_Category', 'QC-2013-14-AreaZero-ProductionPositive', 'QC-2013-14-AreaPositive-ProductionZero', 'QC-2013-14-NegativeMeasurements', 'QC-2013-14-PositiveYieldWithoutPositiveAreaProduction', '

,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,QC-2021-22-PositiveYieldWithoutPositiveAreaProduction,QC-2022-23-AreaZero-ProductionPositive,QC-2022-23-AreaPositive-ProductionZero,QC-2022-23-PositiveYieldWithoutPositiveAreaProduction,QC-2023-24-AreaZero-ProductionPositive,QC-2023-24-AreaPositive-ProductionZero,QC-2023-24-PositiveYieldWithoutPositiveAreaProduction,QC-2024-25-AreaZero-ProductionPositive,QC-2024-25-AreaPositive-ProductionZero,QC-2024-25-PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,False,False,False,True,False,False,True,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,True,False,False,False,False,False,False,False,False,False
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,True,False,False,True,False,False,True,False,False,True
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,True,False,False,False,False,False,False,False,False,False
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,0.0,0.0,526.0,0.0,0.0,9167.0,...,True,False,False,False,False,False,False,False,False,False


Measurement-column summaries:


,count,mean,std,min,25%,50%,75%,max
Area-2013-14,20928.0,0.377688,0.807904,0.0,0.00,0.03,0.34,11.63
Production-2013-14,20808.0,0.846260,2.023841,0.0,0.00,0.04,0.56,28.70
Yield-2013-14,20808.0,1665.355873,1225.886281,2.0,777.00,1285.50,2300.00,11375.00
Area-2014-15,21355.0,0.368406,0.776899,0.0,0.00,0.04,0.33,9.94
Production-2014-15,21204.0,0.782656,1.858980,0.0,0.00,0.04,0.55,26.63
Yield-2014-15,21204.0,1634.272684,1222.958686,0.0,760.00,1330.00,2188.00,11292.00
Area-2015-16,21623.0,0.360860,0.771538,0.0,0.00,0.03,0.31,10.81
Production-2015-16,21578.0,0.734970,1.839879,0.0,0.00,0.04,0.45,27.20
Yield-2015-16,21578.0,1570.087357,1221.702343,0.0,701.00,1192.00,2178.00,11978.00
Area-2016-17,22489.0,0.367157,0.803707,0.0,0.00,0.03,0.32,11.51



long_format: historical_food_grains_apy_long_2013-14_to_2024-25.csv
Exists: True
Shape: (376272, 13)
Columns: ['State', 'District', 'Crop', 'Season', 'Crop_Category', 'Year', 'Area', 'Production', 'Yield', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_NegativeMeasurements', 'QC_PositiveYieldWithoutPositiveAreaProduction']


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,True
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,True
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,True


... (2 outputs left out)


original_processed: historical_food_grains_apy_2013-14_to_2024-25.csv
Exists: True
Shape: (31356, 40)
Columns: ['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Production-2013-14', 'Yield-2013-14', 'Area-2014-15', 'Production-2014-15', 'Yield-2014-15', 'Area-2015-16', 'Production-2015-16', 'Yield-2015-16', 'Area-2016-17', 'Production-2016-17', 'Yield-2016-17', 'Area-2017-18', 'Production-2017-18', 'Yield-2017-18', 'Area-2018-19', 'Production-2018-19', 'Yield-2018-19', 'Area-2019-20', 'Production-2019-20', 'Yield-2019-20', 'Area-2020-21', 'Production-2020-21', 'Yield-2020-21', 'Area-2021-22', 'Production-2021-22', 'Yield-2021-22', 'Area-2022-23', 'Production-2022-23', 'Yield-2022-23', 'Area-2023-24', 'Production-2023-24', 'Yield-2023-24', 'Area-2024-25', 'Production-2024-25', 'Yield-2024-25']


,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,NaN,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,1133.0,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Measurement-column summaries:


,count,mean,std,min,25%,50%,75%,max
Area-2013-14,20928.0,0.377688,0.807904,0.0,0.00,0.03,0.34,11.63
Production-2013-14,20808.0,0.846260,2.023841,0.0,0.00,0.04,0.56,28.70
Yield-2013-14,20808.0,1665.355873,1225.886281,2.0,777.00,1285.50,2300.00,11375.00
Area-2014-15,21355.0,0.368406,0.776899,0.0,0.00,0.04,0.33,9.94
Production-2014-15,21204.0,0.782656,1.858980,0.0,0.00,0.04,0.55,26.63
Yield-2014-15,21204.0,1634.272684,1222.958686,0.0,760.00,1330.00,2188.00,11292.00
Area-2015-16,21623.0,0.360860,0.771538,0.0,0.00,0.03,0.31,10.81
Production-2015-16,21578.0,0.734970,1.839879,0.0,0.00,0.04,0.45,27.20
Yield-2015-16,21578.0,1570.087357,1221.702343,0.0,701.00,1192.00,2178.00,11978.00
Area-2016-17,22489.0,0.367157,0.803707,0.0,0.00,0.03,0.32,11.51


In [91]:
# Cell 85 — Test whether a factor of 1,000 explains the APY scale difference.

unit_check = candidate_train[
    ["Crop", "Season", "State", "District", "FE_YearStart",
     "Area", "Production", "Yield"]
].copy()

unit_check = unit_check.loc[
    (unit_check["Area"] > 0)
    & (unit_check["Production"] > 0)
    & (unit_check["Yield"] > 0)
].copy()

unit_check["Calculated_Yield_x1000"] = (
    unit_check["Production"] / unit_check["Area"] * 1000
)

unit_check["Absolute_Difference"] = (
    unit_check["Yield"] - unit_check["Calculated_Yield_x1000"]
).abs()

unit_check["Relative_Difference_Pct"] = (
    100 * unit_check["Absolute_Difference"]
    / unit_check["Yield"].abs()
)

print("Rows evaluated:", len(unit_check))

print("\nDifference after applying the proposed factor of 1,000:")
display(
    unit_check[
        ["Absolute_Difference", "Relative_Difference_Pct"]
    ].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
)

print("\nExample high-yield records after conversion:")
display(
    unit_check.loc[unit_check["Yield"] >= 10000, [
        "Crop", "Season", "State", "District", "FE_YearStart",
        "Area", "Production", "Yield",
        "Calculated_Yield_x1000", "Relative_Difference_Pct"
    ]]
    .sort_values("Yield", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print(
    "\nThis is a diagnostic test, not confirmation of official units. "
    "We must verify the source's measurement definitions before changing "
    "data or the model target."
)

Rows evaluated: 35784

Difference after applying the proposed factor of 1,000:


,count,mean,std,min,50%,90%,95%,99%,max
Absolute_Difference,35784.0,147.267003,285.967679,0.0,39.000000,400.000000,657.000000,1414.170000,7779.000000
Relative_Difference_Pct,35784.0,10.015457,16.843567,0.0,2.777778,29.873773,43.472023,76.843522,184.090909



Example high-yield records after conversion:


,Crop,Season,State,District,FE_YearStart,Area,Production,Yield,Calculated_Yield_x1000,Relative_Difference_Pct
0,Rice,Kharif,Goa,North Goa,2022,0.01,0.47,44815.0,47000.000000,4.875600
1,Rice,Kharif,Goa,South Goa,2022,0.01,0.47,39221.0,47000.000000,19.833763
2,Rice,Kharif,Himachal Pradesh,Bilaspur,2020,0.01,0.12,12880.0,12000.000000,6.832298
3,Maize,Rabi,Andhra Pradesh,Guntur,2022,0.26,3.24,12526.0,12461.538462,0.514622
4,Maize,Summer,West Bengal,Dakshin Dinajpur,2022,0.05,0.58,12525.0,11600.000000,7.385230
5,Maize,Rabi,Bihar,Katihar,2016,0.45,5.34,11986.0,11866.666667,0.995606
6,Maize,Kharif,Tamil Nadu,Tiruppur,2015,0.06,0.71,11978.0,11833.333333,1.207770
7,Maize,Rabi,Andhra Pradesh,Eluru,2022,0.29,3.50,11933.0,12068.965517,1.139408
8,Maize,Rabi,Andhra Pradesh,Bapatla,2022,0.40,4.68,11620.0,11700.000000,0.688468
9,Maize,Kharif,Tamil Nadu,Perambalur,2017,0.37,4.26,11501.0,11513.513514,0.108804



This is a diagnostic test, not confirmation of official units. We must verify the source's measurement definitions before changing data or the model target.


In [92]:
# Cell 86 — Assess ratio consistency by reported area.

area_check = unit_check.copy()

area_check["Area_Size_Band"] = pd.cut(
    area_check["Area"],
    bins=[-np.inf, 0.05, 0.10, 0.25, 0.50, 1.00, np.inf],
    labels=[
        "<=0.05",
        "0.05–0.10",
        "0.10–0.25",
        "0.25–0.50",
        "0.50–1.00",
        ">1.00",
    ],
)

area_consistency_summary = (
    area_check.groupby("Area_Size_Band", observed=False)
    .agg(
        Rows=("Yield", "size"),
        Median_Area=("Area", "median"),
        Median_Absolute_Difference=("Absolute_Difference", "median"),
        Median_Relative_Difference_Pct=("Relative_Difference_Pct", "median"),
        P90_Relative_Difference_Pct=(
            "Relative_Difference_Pct",
            lambda s: s.quantile(0.90),
        ),
    )
    .reset_index()
)

display(area_consistency_summary)

print("\nRecords with the largest reported areas:")
display(
    area_check.nlargest(10, "Area")[
        [
            "Crop", "Season", "State", "District", "FE_YearStart",
            "Area", "Production", "Yield",
            "Calculated_Yield_x1000", "Relative_Difference_Pct",
        ]
    ].reset_index(drop=True)
)

,Area_Size_Band,Rows,Median_Area,Median_Absolute_Difference,Median_Relative_Difference_Pct,P90_Relative_Difference_Pct
0,<=0.05,16042,0.02,170.000000,13.569576,45.239502
1,0.05–0.10,4329,0.08,45.333333,3.549383,9.825835
2,0.10–0.25,5319,0.16,24.428571,1.643951,4.609577
3,0.25–0.50,3438,0.36,11.333333,0.713472,1.882979
4,0.50–1.00,3269,0.73,6.029851,0.335557,0.896837
5,>1.00,3387,1.53,2.880342,0.136993,0.372243



Records with the largest reported areas:


,Crop,Season,State,District,FE_YearStart,Area,Production,Yield,Calculated_Yield_x1000,Relative_Difference_Pct
0,Bajra,Kharif,Rajasthan,Barmer,2022,9.97,12.15,1218.0,1218.655968,0.053856
1,Bajra,Kharif,Rajasthan,Barmer,2021,9.90,1.60,162.0,161.616162,0.236937
2,Bajra,Kharif,Rajasthan,Barmer,2019,9.52,5.94,623.0,623.949580,0.152421
3,Bajra,Kharif,Rajasthan,Barmer,2020,9.23,6.81,737.0,737.811484,0.110106
4,Bajra,Kharif,Rajasthan,Barmer,2017,9.01,2.09,232.0,231.964484,0.015309
5,Bajra,Kharif,Rajasthan,Barmer,2018,9.00,1.27,141.0,141.111111,0.078802
6,Bajra,Kharif,Rajasthan,Barmer,2013,8.63,3.74,434.0,433.371958,0.144710
7,Bajra,Kharif,Rajasthan,Barmer,2016,7.95,1.19,150.0,149.685535,0.209644
8,Bajra,Kharif,Rajasthan,Barmer,2015,7.79,1.13,145.0,145.057766,0.039839
9,Bajra,Kharif,Rajasthan,Barmer,2014,7.52,2.07,275.0,275.265957,0.096712


In [93]:
# Cell 87 — Recompute consistency statistics with explicit column names.

area_consistency_verified = (
    area_check.groupby("Area_Size_Band", observed=False)
    .agg(
        row_count=("Yield", "size"),
        median_area=("Area", "median"),
        median_abs_difference=("Absolute_Difference", "median"),
        median_relative_difference_pct=("Relative_Difference_Pct", "median"),
        p90_relative_difference_pct=(
            "Relative_Difference_Pct",
            lambda values: values.quantile(0.90),
        ),
        p95_relative_difference_pct=(
            "Relative_Difference_Pct",
            lambda values: values.quantile(0.95),
        ),
    )
    .reset_index()
)

display(area_consistency_verified.round(3))

print("\nOverall relative-difference quantiles:")
display(
    area_check["Relative_Difference_Pct"]
    .quantile([0.50, 0.90, 0.95, 0.99])
    .rename("relative_difference_pct")
    .to_frame()
    .round(3)
)

,Area_Size_Band,row_count,median_area,median_abs_difference,median_relative_difference_pct,p90_relative_difference_pct,p95_relative_difference_pct
0,<=0.05,16042,0.02,170.000,13.570,45.240,57.729
1,0.05–0.10,4329,0.08,45.333,3.549,9.826,13.194
2,0.10–0.25,5319,0.16,24.429,1.644,4.610,6.137
3,0.25–0.50,3438,0.36,11.333,0.713,1.883,2.612
4,0.50–1.00,3269,0.73,6.030,0.336,0.897,1.177
5,>1.00,3387,1.53,2.880,0.137,0.372,0.467



Overall relative-difference quantiles:


,relative_difference_pct
0.50,2.778
0.90,29.874
0.95,43.472
0.99,76.844


In [94]:
# Cell 88 — Print the exact statistics without a formatted table.

for _, row in area_consistency_verified.iterrows():
    print(
        f"Area band: {row['Area_Size_Band']!s:>12} | "
        f"n={int(row['row_count']):5d} | "
        f"median area={row['median_area']:.4f} | "
        f"median absolute difference={row['median_abs_difference']:.3f} | "
        f"median relative difference={row['median_relative_difference_pct']:.3f}% | "
        f"P90={row['p90_relative_difference_pct']:.3f}% | "
        f"P95={row['p95_relative_difference_pct']:.3f}%"
    )

print("\nCheck percentile ordering for every area band:")
print(
    "P90 >= median:",
    bool(
        (
            area_consistency_verified["p90_relative_difference_pct"]
            >= area_consistency_verified["median_relative_difference_pct"]
        ).all()
    ),
)
print(
    "P95 >= P90:",
    bool(
        (
            area_consistency_verified["p95_relative_difference_pct"]
            >= area_consistency_verified["p90_relative_difference_pct"]
        ).all()
    ),
)

Area band:       <=0.05 | n=16042 | median area=0.0200 | median absolute difference=170.000 | median relative difference=13.570% | P90=45.240% | P95=57.729%
Area band:    0.05–0.10 | n= 4329 | median area=0.0800 | median absolute difference=45.333 | median relative difference=3.549% | P90=9.826% | P95=13.194%
Area band:    0.10–0.25 | n= 5319 | median area=0.1600 | median absolute difference=24.429 | median relative difference=1.644% | P90=4.610% | P95=6.137%
Area band:    0.25–0.50 | n= 3438 | median area=0.3600 | median absolute difference=11.333 | median relative difference=0.713% | P90=1.883% | P95=2.612%
Area band:    0.50–1.00 | n= 3269 | median area=0.7300 | median absolute difference=6.030 | median relative difference=0.336% | P90=0.897% | P95=1.177%
Area band:        >1.00 | n= 3387 | median area=1.5300 | median absolute difference=2.880 | median relative difference=0.137% | P90=0.372% | P95=0.467%

Check percentile ordering for every area band:
P90 >= median: True
P95 >= P90:

In [95]:
# Cell 89 — Record the APY consistency diagnostic and modeling decision.

apy_consistency_note = {
    "diagnostic": "Compare reported Yield with Production / Area * 1000",
    "observations_checked": int(len(area_check)),
    "overall_median_relative_difference_pct": float(
        area_check["Relative_Difference_Pct"].median()
    ),
    "overall_p90_relative_difference_pct": float(
        area_check["Relative_Difference_Pct"].quantile(0.90)
    ),
    "overall_p95_relative_difference_pct": float(
        area_check["Relative_Difference_Pct"].quantile(0.95)
    ),
    "area_pattern": (
        "Relative discrepancy decreases as reported area increases."
    ),
    "interpretation": (
        "Consistent with rounding or measurement-precision effects, "
        "but the cause and official measurement units are not verified."
    ),
    "modeling_decision": (
        "Retain the original reported Yield target; do not rescale or "
        "remove records based solely on this diagnostic."
    ),
}

from pprint import pprint
pprint(apy_consistency_note, sort_dicts=False)

{'diagnostic': 'Compare reported Yield with Production / Area * 1000',
 'observations_checked': 35784,
 'overall_median_relative_difference_pct': 2.7777777777777746,
 'overall_p90_relative_difference_pct': 29.873772791023843,
 'overall_p95_relative_difference_pct': 43.47202295552367,
 'area_pattern': 'Relative discrepancy decreases as reported area increases.',
 'interpretation': 'Consistent with rounding or measurement-precision effects, '
                   'but the cause and official measurement units are not '
                   'verified.',
 'modeling_decision': 'Retain the original reported Yield target; do not '
                      'rescale or remove records based solely on this '
                      'diagnostic.'}


In [97]:
# Cell 90 — Inspect saved yield-model metadata.

import json
from pathlib import Path
from pprint import pprint

project_root = Path.cwd()
metadata_path = (
    project_root
    / "models"
    / "yield_prediction"
    / "individual_crop_hgb_location_metadata.json"
)

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Expected model metadata file was not found: {metadata_path}"
    )

with metadata_path.open("r", encoding="utf-8") as file:
    saved_model_metadata = json.load(file)

print("Metadata file:", metadata_path)
print("\nSaved metadata:")
print(saved_model_metadata, sort_dicts=False)

FileNotFoundError: Expected model metadata file was not found: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\notebooks\models\yield_prediction\individual_crop_hgb_location_metadata.json

In [98]:
# Cell 91 — Find the existing metadata file from the notebook directory.

from pathlib import Path

current_dir = Path.cwd().resolve()

project_root = next(
    (
        parent
        for parent in [current_dir, *current_dir.parents]
        if (parent / "models" / "yield_prediction").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'models/yield_prediction'. Check the notebook's directory."
    )

metadata_path = (
    project_root
    / "models"
    / "yield_prediction"
    / "individual_crop_hgb_location_metadata.json"
)

print("Notebook directory:", current_dir)
print("Detected project root:", project_root)
print("Metadata file exists:", metadata_path.is_file())
print("Expected metadata path:", metadata_path)

if metadata_path.is_file():
    print("\nOther files in the model directory:")
    for path in sorted(metadata_path.parent.iterdir()):
        print(f"  {path.name}")

Notebook directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\notebooks
Detected project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Metadata file exists: True
Expected metadata path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location_metadata.json

Other files in the model directory:
  individual_crop_hgb_location.joblib
  individual_crop_hgb_location_metadata.json


In [99]:
# Cell 92 — Load metadata using the verified project root.

import json
from pprint import pprint

metadata_path = (
    project_root
    / "models"
    / "yield_prediction"
    / "individual_crop_hgb_location_metadata.json"
)

with metadata_path.open("r", encoding="utf-8") as file:
    saved_model_metadata = json.load(file)

print("Metadata loaded successfully.")
print("\nSaved model metadata:")
pprint(saved_model_metadata, sort_dicts=False)

Metadata loaded successfully.

Saved model metadata:
{'model_name': 'HistGradientBoostingRegressor',
 'project': 'BharatCrop Intelligence',
 'candidate_scope': 'Individual Crop Candidate cohort',
 'features': ['Crop',
              'Season',
              'Crop_Category',
              'State',
              'District',
              'FE_YearStart'],
 'categorical_features': ['Crop',
                          'Season',
                          'Crop_Category',
                          'State',
                          'District'],
 'numeric_features': ['FE_YearStart'],
 'target': 'Yield',
 'training_rows': 35784,
 'training_year_start': 2013,
 'training_year_end': 2022,
 'holdout_rows': 7369,
 'holdout_evaluated_during_final_fit': False,
 'validation_windows': [{'train_through': 2018,
                         'validation_years': [2019, 2020]},
                        {'train_through': 2020,
                         'validation_years': [2021, 2022]}],
 'validation_metrics': {'2019-20

In [101]:
# Cell 94 — Verify saved-model predictions without relying on final_model.

import numpy as np

saved_model_features = saved_model_metadata["features"]

X_check = validate_yield_model_inputs(
    candidate_train.loc[:, saved_model_features].head(100),
    saved_model_features,
)

saved_predictions = np.asarray(saved_model.predict(X_check), dtype=float)

assert len(saved_predictions) == len(X_check), (
    "Prediction count does not match input row count."
)
assert np.isfinite(saved_predictions).all(), (
    "Saved model produced non-finite predictions."
)

print("Saved model loaded:", True)
print("Model type:", type(saved_model).__name__)
print("Rows checked:", len(X_check))
print("Predictions finite:", True)
print("Prediction range:", (
    round(float(saved_predictions.min()), 3),
    round(float(saved_predictions.max()), 3),
))
print("Features in metadata:", saved_model_features)
print("Feature count matches metadata:", (
    len(saved_model_features) == len(
        saved_model_metadata["categorical_features"]
    ) + len(saved_model_metadata["numeric_features"])
))

Saved model loaded: True
Model type: Pipeline
Rows checked: 100
Predictions finite: True
Prediction range: (503.623, 7826.0)
Features in metadata: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Feature count matches metadata: True


In [102]:
# Cell 95 — Verify the saved pipeline's expected feature names.

expected_features = saved_model_metadata["features"]

if not hasattr(saved_model, "feature_names_in_"):
    raise AttributeError(
        "The saved pipeline does not expose feature_names_in_."
    )

artifact_features = list(saved_model.feature_names_in_)

print("Metadata features:", expected_features)
print("Artifact input features:", artifact_features)
print("Feature order matches:", artifact_features == expected_features)

if artifact_features != expected_features:
    raise AssertionError(
        "Saved pipeline feature order differs from the metadata."
    )

print("\nSaved model input contract verified.")

Metadata features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Artifact input features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Feature order matches: True

Saved model input contract verified.


In [103]:
# Cell 96 — Record Notebook 16 audit findings.

notebook16_audit = {
    "notebook": "16_yield_prediction_baselines.ipynb",
    "status": "AUDIT_CHECKS_PASSED_WITH_LIMITATIONS",
    "saved_model_loads": True,
    "saved_model_predictions_finite": True,
    "metadata_feature_order_matches_artifact": True,
    "in_memory_prediction_equivalence_verified": False,
    "holdout_repeatedly_inspected_during_diagnostics": True,
    "official_apy_units_verified": False,
    "production_readiness_established": False,
    "remaining_limitations": [
        "Final holdout was repeatedly inspected during diagnostics.",
        "Unseen-location validation samples were small.",
        "Official APY measurement units remain unverified.",
        "Saved-model predictions were not compared with the original in-memory model.",
        "Technical handling of unseen categories does not establish agronomic validity.",
    ],
}

from pprint import pprint
pprint(notebook16_audit, sort_dicts=False)

{'notebook': '16_yield_prediction_baselines.ipynb',
 'status': 'AUDIT_CHECKS_PASSED_WITH_LIMITATIONS',
 'saved_model_loads': True,
 'saved_model_predictions_finite': True,
 'metadata_feature_order_matches_artifact': True,
 'in_memory_prediction_equivalence_verified': False,
 'holdout_repeatedly_inspected_during_diagnostics': True,
 'official_apy_units_verified': False,
 'production_readiness_established': False,
 'remaining_limitations': ['Final holdout was repeatedly inspected during '
                           'diagnostics.',
                           'Unseen-location validation samples were small.',
                           'Official APY measurement units remain unverified.',
                           'Saved-model predictions were not compared with the '
                           'original in-memory model.',
                           'Technical handling of unseen categories does not '
                           'establish agronomic validity.']}



# Notebook 16 — Yield Prediction Baselines and Model Selection

## 1. Objective

The objective of this notebook was to establish and compare yield-prediction baselines, evaluate candidate regression models using time-aware validation, select a model for individual-crop yield prediction, and verify its saved artifact and input contract.

## 2. Dataset and Evaluation Strategy

- **Source:** Existing APY crop statistics integrated with district-level weather data.
- **Target variable:** `Yield`, retained in its original reported form.
- **Candidate cohort:** Individual Crop Candidate records with the required crop, season, location, year, and target information.
- **Eligible training rows:** 35,784.
- **Holdout rows:** 7,369.
- **Training period:** 2013–14 to 2022–23.
- **Holdout period:** 2023–24 and 2024–25.
- **Validation strategy:** Historical time-based validation windows, supplemented by a final holdout evaluation.

Aggregate records were excluded according to the notebook's filtering rules. The primary modeling cohort required positive, finite Yield, Area, and Production values.

## 3. Baseline and Candidate Model Comparison

The initial experiments compared a global median baseline, a Crop–Season median baseline, linear regression, Ridge regression, Decision Tree, Random Forest, and HistGradientBoostingRegressor.

The HistGradientBoostingRegressor achieved the strongest reported baseline performance among these initial candidates:

| Metric | Initial HGB baseline |
|---|---:|
| MAE | 780.275 |
| RMSE | 1,111.999 |
| R² | 0.4001 |

These figures belong to the earlier primary-cohort experiment and should not be confused with the later individual-crop candidate model.

## 4. Selected Individual-Crop Model

The selected candidate was a HistGradientBoostingRegressor pipeline with one-hot categorical preprocessing.

**Input features:**
- `Crop`
- `Season`
- `Crop_Category`
- `State`
- `District`
- `FE_YearStart`

**Target:** `Yield`

**Training rows:** 35,784

**Saved artifacts:**
- `models/yield_prediction/individual_crop_hgb_location.joblib`
- `models/yield_prediction/individual_crop_hgb_location_metadata.json`

Weather features were excluded from the selected model because their improvement was not consistent once location features were included.

## 5. Historical Validation Results

| Training cutoff | Validation years | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| 2018 | 2019–2020 | 448.306 | 672.961 | 0.798 |
| 2020 | 2021–2022 | 422.856 | 898.177 | 0.693 |

The Core + Location candidate outperformed the Crop–Season median baseline in both reported historical validation windows. The variation in RMSE and R² across windows indicates that performance differs across time periods.

These historical validation metrics must not be presented as independent final-holdout results.

## 6. Data-Quality Investigation

The notebook investigated the consistency between reported Yield and the ratio `Production / Area × 1000`.

For 35,784 candidate training observations:
- Median relative difference: approximately 2.778%.
- 90th-percentile relative difference: approximately 29.874%.
- 95th-percentile relative difference: approximately 43.472%.

The relative discrepancy decreased as reported Area increased. This pattern is consistent with possible rounding or measurement-precision effects, but the cause and official APY measurement units remain unverified.

**Decision:** Retain the original reported Yield target. Do not rescale it or remove observations solely because of this diagnostic. Extreme-yield records were investigated separately and were not automatically discarded based on model error.

## 7. Artifact and Input-Contract Verification

The saved pipeline was loaded successfully from disk and generated finite predictions for a 100-row sample.

The following checks passed:
- Saved model artifact exists and loads.
- Sample predictions are finite.
- Metadata feature count is consistent with the five categorical and one numeric features.
- Saved pipeline input feature names and order match the metadata.

The saved model's predictions were not compared with the original in-memory model because the latter was unavailable under the variable name used for that check. Therefore, prediction equivalence with the original in-memory model remains unverified.

## 8. Limitations and Risks

1. The final holdout was repeatedly inspected during diagnostics, so it should not be described as a pristine, untouched test set.
2. Validation coverage for unseen locations was limited.
3. Technical handling of unseen categorical values does not establish agronomic validity for those locations or crops.
4. Official APY measurement units and the cause of the yield-ratio discrepancies remain unverified.
5. Weather data coverage and season alignment are incomplete or unverified. Weather results are exploratory and do not establish causal relationships.
6. Model performance alone does not establish production readiness or reliability for every crop, district, and season.
7. Negative predictions occurred in full-training diagnostics, so downstream prediction policies should validate outputs rather than assume all predictions are agronomically valid.

## 9. Final Status

**Status: Audit checks passed with limitations.**

The selected model and metadata are saved, and the artifact's basic loading, prediction, and feature-contract checks passed. The documented limitations remain open and must be considered in subsequent training, evaluation, and deployment-policy work.

Notebook 16 establishes the current yield-model baseline and selected candidate. Notebook 17 will continue with yield-model training and evaluation, building on these artifacts without assuming that the current candidate is production-ready.
